# Overview: Aging mouse brain SpiderNet workflow

This notebook provides the end-to-end SpiderNet analysis of the ten ageing mouse brain coronal MERFISH sections used for model training. It starts from the processed bundle generated by `spidernet_dataloading_MIdimselection_AgingBrain`, trains and exports the SpiderNet model, characterizes learned meta-interactions (MIs), evaluates cell-age prediction, examines the T-cell-associated MI-29 program, and runs the associated in silico perturbation analyses.

### Main workflow

1. Configure the coronal MERFISH input, processed-data, and output paths.
2. Validate and load the preprocessed SpiderNet bundle.
3. Train the model and export MI activities and LR, sender, and receiver loadings.
4. Characterize MI correlations, pathway loadings, and sender--receiver cell-type enrichment.
5. Evaluate cell-type-specific age prediction with SpiderNet and baseline representations.
6. Analyze MI-29 activity, ageing-module scores, and spatial sender--receiver patterns.
7. Compare T-cell-sender ageing associations across SpiderNet, NMF-LR, COMMOT, and ScCChain using standardized mean differences (SMDs).
8. Run MI-29-guided in silico cell-replacement and gene-perturbation analyses and associated GO enrichment.

### Major inputs and outputs

The notebook expects raw coronal AnnData files under `DATA_ROOT`, the processed bundle under `PROCESSED_DATA_DIR`, and precomputed baseline outputs where required by optional comparisons. Core SpiderNet outputs are written to the versioned run directory. Optional downstream analyses add age-prediction tables and figures, MI-29 ageing-module results, CCC-method SMD comparisons, perturbation results, and GO-enrichment outputs.

The primary ageing-brain workflow supports the main ageing-brain figure and associated supplementary analyses in the SpiderNet manuscript. The multi-method T-cell-sender SMD comparison is an additional diagnostic analysis and is not explicitly represented in the current manuscript.


## 0. Dataset setup (edit this cell first)

Set the raw-data path, the processed-data directory produced by `spidernet_dataloading_MIdimselection_AgingBrain`, and the training/output settings here.


In [ ]:
from workflow_paths import DATA_DIR, RESULTS_ROOT, input_path, output_path
from pathlib import Path

import json
import numpy as np
import shutil

# Paths
DATA_ROOT = DATA_DIR  # path to raw input data
PROCESSED_DATA_DIR = (RESULTS_ROOT / 'ProcessedData')  # output directory generated by spidernet_dataloading_MIdimselection_AgingBrain
OUTPUT_ROOT = RESULTS_ROOT  # root directory for training outputs and downstream analysis results

# Dataset-specific fields
SPECIES = "mouse"          # "human" or "mouse"
SAMPLE_ID_COL = "age"      # column in adata.obs
CELL_TYPE_COL = "celltype" # column in adata.obs
SPATIAL_KEY = "spatial"    # key in adata.obsm

# Preprocessing parameters recorded for reproducibility.
# These should match the settings used in spidernet_dataloading_MIdimselection_AgingBrain.
N_HVG = 1000        # number of highly variable genes used for SpiderNet model training
N_HVG_LR = 2000     # number of highly variable genes used for ligand-receptor candidate selection
NUM_NEIGHBORS = 5   # number of spatial neighbors used to build directed cell pairs

# Training parameters
DIM_ENVIR = 30      # number of latent MI dimensions
N_JOBS = 5          # number of parallel CPU cores for initialization and training
MAX_EPOCH = 50000   # maximum number of training epochs

VERSION = "V1"

# Create config dict
config = {
    "DATA_ROOT": str(DATA_ROOT),
    "PROCESSED_DATA_DIR": str(PROCESSED_DATA_DIR),
    "OUTPUT_ROOT": str(OUTPUT_ROOT),
    "SPECIES": SPECIES,
    "SAMPLE_ID_COL": SAMPLE_ID_COL,
    "CELL_TYPE_COL": CELL_TYPE_COL,
    "SPATIAL_KEY": SPATIAL_KEY,
    "N_HVG": N_HVG,
    "N_HVG_LR": N_HVG_LR,
    "NUM_NEIGHBORS": NUM_NEIGHBORS,
    "DIM_ENVIR": DIM_ENVIR,
    "N_JOBS": N_JOBS,
    "MAX_EPOCH": MAX_EPOCH,
    "VERSION": VERSION,
}

# Make sure output folder exists
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Save to OUTPUT_ROOT
config_path = OUTPUT_ROOT / "config.json"
with open(output_path(config_path), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4)

print(f"Config saved to: {config_path}")


## 1. Imports and device setup

Load SpiderNet utilities and choose CPU or CUDA.

In [ ]:
import json

from SpiderNet.utils import *
from SpiderNet.config import *

cuda_available = torch.cuda.is_available()
if cuda_available:
    device = "cuda"
else:
    device = "cpu"
print(f"Using device: {device}")


## 2. Build paths from the dataset setup

This cell uses the values from the dataset setup cell above to build the model-output directories.


In [ ]:
paths = PathConfig(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT,
    species=SPECIES,
    version=VERSION,
)

paths


In [ ]:
##Copy sample IDs and cell type labels to the expected obs fields if needed, and check that the expected fields are present.
import scanpy as sc

adata_dir = Path(DATA_ROOT) / "adata"
sample_files = sorted(adata_dir.glob("*.h5ad"))

updated_rows = []

for sample_path in sample_files:
    adata_i = sc.read_h5ad(input_path(sample_path))

    ok_sample_id_col = SAMPLE_ID_COL in adata_i.obs.columns
    ok_cell_type_col = CELL_TYPE_COL in adata_i.obs.columns

    if ok_sample_id_col:
        adata_i.obs["SAMPLE_ID"] = adata_i.obs[SAMPLE_ID_COL].copy()
    if ok_cell_type_col:
        adata_i.obs["CELL_TYPE"] = adata_i.obs[CELL_TYPE_COL].copy()

    adata_i.write_h5ad(sample_path)

    updated_rows.append({
        "file": sample_path.name,
        f"has obs['{SAMPLE_ID_COL}']": ok_sample_id_col,
        f"has obs['{CELL_TYPE_COL}']": ok_cell_type_col,
        f"created obs['SAMPLE_ID']": ok_sample_id_col,
        f"created obs['CELL_TYPE']": ok_cell_type_col,
    })

display(pd.DataFrame(updated_rows))

### Check paths, AnnData fields, and the processed bundle

Use the next cell to confirm that the raw-data paths exist and that the processed bundle produced by `spidernet_dataloading_MIdimselection_AgingBrain` is available.


In [ ]:
adata_dir = Path(DATA_ROOT) / "adata"
sample_files = sorted(adata_dir.glob("*.h5ad"))
sample_path = sample_files[0] if sample_files else None

processed_required_files = [
    "adata_all.h5ad",
    "SpiderNet_data_pyg_list.pkl",
    "LR_list.pkl",
    "LR_list_all.pkl",
    "LR_list_cellchatdb.pkl",
    "LR_meta_cellchatdb.pkl",
    "batch_cell_unique.pkl",
    "batch_cell.pkl",
    "genenames.pkl",
    "genenames_train.pkl",
    "adata_list.pkl",
    "cellclass_unique.pkl",
]
processed_missing = [f for f in processed_required_files if not (input_path(Path(PROCESSED_DATA_DIR) / f)).exists()]

rows = [
    {"item": "DATA_ROOT", "value": str(DATA_ROOT), "ok": input_path(Path(DATA_ROOT)).exists()},
    {"item": "PROCESSED_DATA_DIR", "value": str(PROCESSED_DATA_DIR), "ok": input_path(Path(PROCESSED_DATA_DIR)).exists()},
    {"item": "OUTPUT_ROOT", "value": str(OUTPUT_ROOT), "ok": input_path(Path(OUTPUT_ROOT)).exists()},
    {"item": "CellChat DB", "value": str(paths.cellchat_db), "ok": input_path(Path(paths.cellchat_db)).exists()},
    {"item": "scSeqComm DB", "value": str(paths.scseqcomm_db), "ok": input_path(Path(paths.scseqcomm_db)).exists()},
    {"item": "adata folder", "value": str(adata_dir), "ok": input_path(adata_dir).exists()},
    {"item": "number of .h5ad files", "value": len(sample_files), "ok": len(sample_files) > 0},
    {"item": "processed bundle files", "value": f"{len(processed_required_files) - len(processed_missing)} / {len(processed_required_files)} present", "ok": len(processed_missing) == 0},
]

if sample_path is not None:
    adata_example = sc.read_h5ad(input_path(sample_path), backed="r")
    rows.extend([
        {"item": "example file", "value": sample_path.name, "ok": True},
        {"item": f"obs['{SAMPLE_ID_COL}']", "value": SAMPLE_ID_COL, "ok": SAMPLE_ID_COL in adata_example.obs.columns},
        {"item": f"obs['{CELL_TYPE_COL}']", "value": CELL_TYPE_COL, "ok": CELL_TYPE_COL in adata_example.obs.columns},
        {"item": f"obsm['{SPATIAL_KEY}']", "value": SPATIAL_KEY, "ok": SPATIAL_KEY in adata_example.obsm_keys()},
    ])
    adata_example.file.close()
else:
    rows.append({"item": "example file", "value": "No .h5ad file found", "ok": False})

display(pd.DataFrame(rows))

if processed_missing:
    print("Missing processed bundle files:")
    for name in processed_missing:
        print("-", name)


In [ ]:
# ============================================================
# Raw cell count summary: total and per original slice
# This is BEFORE SpiderNet preprocessing / filtering.
#
# For the AgingBrain workflow, each raw .h5ad file under
# DATA_ROOT / "adata" is treated as one original slice.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc

raw_adata_dir = Path(DATA_ROOT) / "adata"
raw_h5ad_files = sorted(raw_adata_dir.glob("*.h5ad"))

if len(raw_h5ad_files) == 0:
    raise FileNotFoundError(f"No raw .h5ad files found under: {raw_adata_dir}")

raw_cell_count_rows = []
raw_cell_count_by_sample_rows = []

for raw_slice_index, h5ad_path in enumerate(raw_h5ad_files):
    print(f"Reading raw slice {raw_slice_index + 1}/{len(raw_h5ad_files)}: {h5ad_path.name}")

    adata_raw = sc.read_h5ad(input_path(h5ad_path), backed="r")

    try:
        obs_raw = adata_raw.obs.copy()

        # Basic per-file / per-slice metadata
        if SAMPLE_ID_COL in obs_raw.columns:
            sample_values = np.sort(obs_raw[SAMPLE_ID_COL].astype(str).unique())
            sample_id_values = ";".join(sample_values)
            n_sample_id_values = len(sample_values)
        else:
            sample_id_values = np.nan
            n_sample_id_values = 0

        if CELL_TYPE_COL in obs_raw.columns:
            n_cell_types = int(obs_raw[CELL_TYPE_COL].astype(str).nunique())
        else:
            n_cell_types = np.nan

        raw_cell_count_rows.append(
            {
                "raw_slice_index": raw_slice_index,
                "raw_slice_id": h5ad_path.stem,
                "source_file": h5ad_path.name,
                SAMPLE_ID_COL: sample_id_values,
                f"n_unique_{SAMPLE_ID_COL}": n_sample_id_values,
                "raw_n_cells": int(adata_raw.n_obs),
                "raw_n_genes": int(adata_raw.n_vars),
                "raw_n_cell_types": n_cell_types,
                f"has_obsm_{SPATIAL_KEY}": SPATIAL_KEY in adata_raw.obsm_keys(),
            }
        )

        # Optional: if the sample/age column exists, also summarize raw cells by that column.
        # This is useful for AgingBrain because SAMPLE_ID_COL is usually "age".
        if SAMPLE_ID_COL in obs_raw.columns:
            tmp_by_sample = (
                obs_raw
                .groupby(SAMPLE_ID_COL, dropna=False)
                .size()
                .reset_index(name="raw_n_cells")
            )
            tmp_by_sample["raw_slice_index"] = raw_slice_index
            tmp_by_sample["raw_slice_id"] = h5ad_path.stem
            tmp_by_sample["source_file"] = h5ad_path.name
            raw_cell_count_by_sample_rows.append(tmp_by_sample)

    finally:
        adata_raw.file.close()

raw_cell_count_df = pd.DataFrame(raw_cell_count_rows)

raw_cell_count_df = raw_cell_count_df.sort_values(
    ["raw_slice_index", "raw_slice_id"]
).reset_index(drop=True)

print("\nRaw data cell-count summary")
print("=" * 60)
print(f"Raw total number of cells: {raw_cell_count_df['raw_n_cells'].sum():,}")
print(f"Number of raw .h5ad files / original slices: {raw_cell_count_df.shape[0]:,}")
print(f"Mean raw cells per slice: {raw_cell_count_df['raw_n_cells'].mean():,.1f}")
print(f"Median raw cells per slice: {raw_cell_count_df['raw_n_cells'].median():,.1f}")
print(f"Min raw cells per slice: {raw_cell_count_df['raw_n_cells'].min():,}")
print(f"Max raw cells per slice: {raw_cell_count_df['raw_n_cells'].max():,}")

display(raw_cell_count_df)

# Optional age/sample-level summary
if len(raw_cell_count_by_sample_rows) > 0:
    raw_cell_count_by_sample_df = pd.concat(
        raw_cell_count_by_sample_rows,
        axis=0,
        ignore_index=True,
    )

    raw_cell_count_by_sample_df = raw_cell_count_by_sample_df.sort_values(
        ["raw_slice_index", SAMPLE_ID_COL]
    ).reset_index(drop=True)

    print(f"\nRaw cell counts by raw slice and {SAMPLE_ID_COL}")
    display(raw_cell_count_by_sample_df)

    raw_cell_count_by_sample_summary = (
        raw_cell_count_by_sample_df
        .groupby(SAMPLE_ID_COL, as_index=False)
        .agg(
            raw_n_slices=("raw_slice_id", "nunique"),
            raw_total_cells=("raw_n_cells", "sum"),
            raw_mean_cells_per_slice=("raw_n_cells", "mean"),
            raw_median_cells_per_slice=("raw_n_cells", "median"),
            raw_min_cells_per_slice=("raw_n_cells", "min"),
            raw_max_cells_per_slice=("raw_n_cells", "max"),
        )
    )

    print(f"\nRaw cell-count summary by {SAMPLE_ID_COL}")
    display(raw_cell_count_by_sample_summary)

# Save summaries
raw_cell_count_path = Path(OUTPUT_ROOT) / "raw_cell_count_per_original_slice.csv"
raw_cell_count_df.to_csv(output_path(raw_cell_count_path), index=False)
print(f"Saved raw per-slice cell counts to: {raw_cell_count_path}")

if len(raw_cell_count_by_sample_rows) > 0:
    raw_cell_count_by_sample_path = Path(OUTPUT_ROOT) / f"raw_cell_count_by_slice_and_{SAMPLE_ID_COL}.csv"
    raw_cell_count_by_sample_summary_path = Path(OUTPUT_ROOT) / f"raw_cell_count_by_{SAMPLE_ID_COL}.csv"

    raw_cell_count_by_sample_df.to_csv(output_path(raw_cell_count_by_sample_path), index=False)
    raw_cell_count_by_sample_summary.to_csv(output_path(raw_cell_count_by_sample_summary_path), index=False)

    print(f"Saved raw slice-by-{SAMPLE_ID_COL} cell counts to: {raw_cell_count_by_sample_path}")
    print(f"Saved raw {SAMPLE_ID_COL}-level cell-count summary to: {raw_cell_count_by_sample_summary_path}")

## 3. Build preprocessing and training configs

These values are taken from the dataset setup cell. The preprocessing config is recorded for reproducibility and should match the settings used to generate `PROCESSED_DATA_DIR`.


In [ ]:
preprocess_cfg = PreprocessConfig(
    n_hvg=N_HVG,
    n_hvg_lr=N_HVG_LR,
    num_neighbors=NUM_NEIGHBORS
)

train_cfg = TrainingConfig(
    dim_envir=DIM_ENVIR,
    n_jobs=N_JOBS,
    max_epoch=MAX_EPOCH,
    version=VERSION
)

run_dirs = paths.ensure_dirs(
    dim_envir=train_cfg.dim_envir
)
print(run_dirs)

with open(output_path(paths.output_root / "run_dirs.json"), "w", encoding="utf-8") as handle:
    json.dump({k: str(v) for k, v in run_dirs.items()}, handle, indent=2)


## 4. Reuse the preprocessed bundle and load processed data

Copy the processed objects generated by `spidernet_dataloading_MIdimselection_AgingBrain` into the current run directory so that the rest of this notebook can use the standard `run_dir` layout.


In [ ]:
processed_dir = Path(PROCESSED_DATA_DIR)
run_dir = Path(run_dirs["run_dir"])
run_dir.mkdir(parents=True, exist_ok=True)

required_files = [
    "adata_all.h5ad",
    "SpiderNet_data_pyg_list.pkl",
    "LR_list.pkl",
    "LR_list_all.pkl",
    "LR_list_cellchatdb.pkl",
    "LR_meta_cellchatdb.pkl",
    "batch_cell_unique.pkl",
    "batch_cell.pkl",
    "genenames.pkl",
    "genenames_train.pkl",
    "adata_list.pkl",
    "cellclass_unique.pkl",
]
optional_files = [
    "LR_list_val.pkl",
    "LR_pairs_correlation_heatmap_clustered.png",
    "bundle_summary.json",
]

missing_files = [name for name in required_files if not (input_path(processed_dir / name)).exists()]
if missing_files:
    raise FileNotFoundError(
        "The processed AgingBrain bundle is incomplete. "
        f"Missing files in {processed_dir}: {missing_files}"
    )

copied_files = []
skipped_files = []
for name in required_files + optional_files:
    src = processed_dir / name
    dst = run_dir / name
    if not input_path(src).exists():
        continue

    if input_path(dst).exists() and dst.stat().st_size == src.stat().st_size and dst.stat().st_mtime >= src.stat().st_mtime:
        skipped_files.append(name)
    else:
        shutil.copy2(src, dst)
        copied_files.append(name)

source_record = {
    "processed_data_dir": str(processed_dir),
    "run_dir": str(run_dir),
    "copied_files": copied_files,
    "skipped_files": skipped_files,
}
with open(output_path(run_dir / "processed_bundle_source.json"), "w", encoding="utf-8") as handle:
    json.dump(source_record, handle, indent=2)

print(f"Using processed bundle from: {processed_dir}")
print(f"Run directory: {run_dir}")
print(f"Copied {len(copied_files)} files and skipped {len(skipped_files)} files.")
if copied_files:
    print("Copied files:")
    for name in copied_files:
        print("-", name)


Reload the processed objects for training and downstream analysis.

Check that the number of batches, LR pairs, and training genes is reasonable.


In [ ]:
from SpiderNet.io import load_processed_data
processed = load_processed_data(run_dirs["run_dir"])

print("Processed bundle source:", PROCESSED_DATA_DIR)
print("Number of batches:", len(processed.spidernet_data))
print("Number of LR pairs:", len(processed.lr_list))
print("Number of training genes:", processed.genenames_train.shape[0])
print("Number of cells:", np.sum([adata.n_obs for adata in processed.adata_list]))


In [ ]:
# ============================================================
# Training cell count summary: actual cells used by SpiderNet
# This counts processed.spidernet_data[i].x.shape[0],
# i.e. the cell size actually passed to model training.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch

train_cell_count_rows = []

# Optional slice/sample labels saved by preprocessing
batch_cell_unique_path = Path(run_dirs["run_dir"]) / "batch_cell_unique.pkl"
if input_path(batch_cell_unique_path).exists():
    batch_cell_unique = pd.read_pickle(input_path(batch_cell_unique_path))
else:
    batch_cell_unique = None

for batch_index in range(len(processed.spidernet_data)):
    data_i = processed.spidernet_data[batch_index]
    adata_i = processed.adata_list[batch_index]

    # Actual number of cells used by SpiderNet model
    n_train_cells = int(data_i.x.shape[0])

    # AnnData cell number after preprocessing/filtering
    n_adata_cells = int(adata_i.n_obs)

    # Number of genes/features used by model
    n_train_genes = int(data_i.x.shape[1])

    # Number of directed cell-cell edges used by model
    if "edge_index" in data_i:
        edge_index_i = data_i["edge_index"]
        n_edges = int(edge_index_i.shape[0])
        edge_index_max = int(torch.max(edge_index_i).detach().cpu().item())
        edge_index_matches_n_cells = (edge_index_max + 1 == n_train_cells)
    else:
        n_edges = np.nan
        edge_index_max = np.nan
        edge_index_matches_n_cells = np.nan

    # Slice label from preprocessing if available
    if batch_cell_unique is not None and batch_index < len(batch_cell_unique):
        slice_id = str(batch_cell_unique[batch_index])
    else:
        slice_id = f"batch_{batch_index}"

    # Metadata from adata.obs
    if SAMPLE_ID_COL in adata_i.obs.columns:
        sample_values = np.sort(adata_i.obs[SAMPLE_ID_COL].astype(str).unique())
        sample_id = ";".join(sample_values)
        n_unique_sample_id = len(sample_values)
    else:
        sample_id = np.nan
        n_unique_sample_id = np.nan

    if CELL_TYPE_COL in adata_i.obs.columns:
        n_cell_types = int(adata_i.obs[CELL_TYPE_COL].astype(str).nunique())
    else:
        n_cell_types = np.nan

    train_cell_count_rows.append({
        "batch_index": batch_index,
        "slice_id": slice_id,
        SAMPLE_ID_COL: sample_id,
        f"n_unique_{SAMPLE_ID_COL}": n_unique_sample_id,
        "train_n_cells": n_train_cells,
        "adata_n_cells": n_adata_cells,
        "train_n_genes": n_train_genes,
        "n_edges": n_edges,
        "raw_edge_index_max": edge_index_max,
        "edge_index_matches_train_n_cells": edge_index_matches_n_cells,
        "adata_matches_train_n_cells": n_adata_cells == n_train_cells,
        "n_cell_types": n_cell_types,
    })

train_cell_count_df = pd.DataFrame(train_cell_count_rows)

print("\nActual SpiderNet training cell-count summary")
print("=" * 70)
print(f"Total cells used for training: {train_cell_count_df['train_n_cells'].sum():,}")
print(f"Number of training slices / batches: {train_cell_count_df.shape[0]:,}")
print(f"Mean cells per training slice: {train_cell_count_df['train_n_cells'].mean():,.1f}")
print(f"Median cells per training slice: {train_cell_count_df['train_n_cells'].median():,.1f}")
print(f"Min cells per training slice: {train_cell_count_df['train_n_cells'].min():,}")
print(f"Max cells per training slice: {train_cell_count_df['train_n_cells'].max():,}")

print("\nPer-slice training cell counts:")
display(train_cell_count_df)

# Optional summary by SAMPLE_ID_COL, e.g. age in AgingBrain
if SAMPLE_ID_COL in train_cell_count_df.columns:
    train_cell_count_by_sample = (
        train_cell_count_df
        .groupby(SAMPLE_ID_COL, as_index=False)
        .agg(
            n_training_slices=("slice_id", "nunique"),
            total_train_cells=("train_n_cells", "sum"),
            mean_train_cells_per_slice=("train_n_cells", "mean"),
            median_train_cells_per_slice=("train_n_cells", "median"),
            min_train_cells_per_slice=("train_n_cells", "min"),
            max_train_cells_per_slice=("train_n_cells", "max"),
        )
    )

    print(f"\nTraining cell-count summary by {SAMPLE_ID_COL}:")
    display(train_cell_count_by_sample)

# Warnings if there is any mismatch
if not train_cell_count_df["adata_matches_train_n_cells"].all():
    print("\nWARNING: Some slices have adata.n_obs != data.x.shape[0].")
    display(train_cell_count_df.loc[~train_cell_count_df["adata_matches_train_n_cells"]])

if not train_cell_count_df["edge_index_matches_train_n_cells"].all():
    print("\nWARNING: Some slices have edge_index max inconsistent with train_n_cells.")
    display(train_cell_count_df.loc[~train_cell_count_df["edge_index_matches_train_n_cells"]])

# Optional: save
train_cell_count_path = Path(run_dirs["run_dir"]) / "training_cell_count_per_slice.csv"
train_cell_count_df.to_csv(output_path(train_cell_count_path), index=False)
print(f"\nSaved training cell-count summary to: {train_cell_count_path}")

Check whether the processed data are non-empty and internally consistent before training.


In [ ]:
summary_rows = []
for i in range(len(processed.adata_list)):
    adata_i = processed.adata_list[i]
    edge_index_max = torch.max(processed.spidernet_data[i]["edge_index"]).cpu().item()
    summary_rows.append(
        {
            "batch_index": i,
            "n_cells": adata_i.n_obs,
            "n_genes": adata_i.n_vars,
            "edge_index_max": edge_index_max,
            "edge_index_matches_n_cells": adata_i.n_obs == (edge_index_max + 1),
        }
    )

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

issues = []
if len(processed.spidernet_data) == 0:
    issues.append("No processed batches were loaded.")
if len(processed.lr_list) == 0:
    issues.append("No ligand-receptor pairs were retained.")
if processed.genenames_train.shape[0] == 0:
    issues.append("No training genes were retained.")
if not summary_df["edge_index_matches_n_cells"].all():
    issues.append("At least one batch has an edge index / cell-count mismatch.")

if issues:
    print("Sanity check flagged the following issues:")
    for issue in issues:
        print("-", issue)
else:
    print("Sanity checks passed. The processed data appear internally consistent.")


## 5. Build and train the SpiderNet model

Initialize the model using the processed data and training settings.


In [ ]:
from SpiderNet.api import build_model
model = build_model(
    processed=processed,
    train_cfg=train_cfg,
    device=device,
    hidden_channels=256
)

print(model)


Runs model training and saves checkpoints.

In [ ]:
from SpiderNet.api import run_training
model = run_training(
    model=model,
    processed=processed,
    train_cfg=train_cfg,
    model_dir=run_dirs["model_dir"],
    device=device,
)


In [ ]:
model.eval() # set to eval mode for inference

### Extract the inferred meta-interactions and loadings

This step generates the main outputs:
- MI activities
- LR loadings
- sender loadings
- receiver loadings


In [ ]:
from SpiderNet.api import infer_meta_interactions, normalize_outputs
results = infer_meta_interactions(model=model, processed=processed)
results = normalize_outputs(results)

print("Factor_envir shape:", results["factor_envir"].shape)
print("LR loading shape:", results["loading_lr"].shape)
print("Receiver loading shape:", results["loading_receiver"].shape)
print("Sender loading shape:", results["loading_sender"].shape)


Saves the main outputs and config files.

In [ ]:
from SpiderNet.api import export_results
export_results(
    results=results,
    processed=processed,
    precessed_data_dir=PROCESSED_DATA_DIR,
    output_dir=output_path(run_dirs["run_dir"]),
)

# Save the model and preprocessing configurations for reproducibility.
with open(output_path(run_dirs["model_dir"] / "SpiderNet_model_config.json"), "w", encoding="utf-8") as handle:
    json.dump(train_cfg.to_dict(), handle, indent=2)

with open(output_path(run_dirs["model_dir"] / "SpiderNet_preprocess_config.json"), "w", encoding="utf-8") as handle:
    json.dump(preprocess_cfg.to_dict(), handle, indent=2)

print(f"Results saved to: {run_dirs['run_dir']}")


## Key output files

Most important outputs:
- `Factor_envir_use.npy`
- `loading_LR_use.npy`
- `loading_sender_use.npy`
- `loading_receiver_use.npy`

Check these files before running the optional downstream analyses below.


## Optional downstream analyses

The remaining sections are not required for model training. They interpret the learned MIs and reproduce the ageing-related analyses used in this workflow. Some sections require separately generated outputs from baseline CCC methods; those dependencies are stated where they are used.

The analyses remain in a mostly cell-by-cell form to preserve the published computational workflow and its intermediate outputs.


## Optional A1. MI correlation analysis

In [ ]:
from SpiderNet.analysis import MI_correlation

mi_results = MI_correlation(output_path(run_dirs['run_dir']), input_path(run_dirs['run_dir'] / "Factor_envir_use.npy"), show=True)


## Optional A2. LR-loading-based pathway enrichment

In [ ]:
from SpiderNet.analysis import LRLoading_enrichment
LRLoading_enrichment(
    loading_LR_use_path=input_path(run_dirs['run_dir'] / "loading_LR_use.npy"),
    lr_list_path=input_path(run_dirs['run_dir'] / "LR_list.pkl"),
    lr_list_cellchatdb_path=input_path(run_dirs['run_dir'] / "LR_list_cellchatdb.pkl"),
    lr_meta_cellchatdb_path=input_path(run_dirs['run_dir'] / "LR_meta_cellchatdb.pkl"),
    Factor_envir_use_path=input_path(run_dirs['run_dir'] / "Factor_envir_use.npy"),
    file_savepath_main=output_path(run_dirs['run_dir']),
    show=True
)


## Optional A3. Sender-cell-type--receiver-cell-type MI enrichment

In [ ]:
from SpiderNet.analysis import MI_Celltypepair_enrichment
MI_Celltypepair_enrichment(
    SpiderNet_data_pyg_list_path=input_path(run_dirs['run_dir'] / "SpiderNet_data_pyg_list.pkl"),
    Factor_envir_use_path=input_path(run_dirs['run_dir'] / "Factor_envir_use.npy"),
    file_savepath_main=output_path(run_dirs['run_dir']),
    metadata_sample_path="None",
    LR_loading_pathway_path=input_path(run_dirs['run_dir'] / "LR_loading_pathway.csv"),
    dim_envir=train_cfg.dim_envir,
    MIlevel_agg_threshold=0.6,
    batch_cell_unique_path=input_path(run_dirs['run_dir'] / "batch_cell_unique.pkl"),
    adata_list_path=input_path(run_dirs['run_dir'] / "adata_list.pkl"),
    adata_copy_path=input_path(run_dirs['run_dir'] / "adata_all.h5ad"),
    show=True
)


In [ ]:
# ============================================================
# MI-specific sender→receiver cell-type bubble plot
# ------------------------------------------------------------
# Plot definition:
#   1. x-axis = MI, ordered by the configured MI order
#   2. y-axis = sender→receiver cell-type pair
#   3. Dense plot: all selected pairs are shown across all MIs
#   4. Cell-type pair labels are shown as "Sender  →  Receiver"
#   5. TOP_K_PER_MI = 3
#   6. Font sizes are further enlarged by 2x
#   7. Dot edge color matches fill color
#   8. Reduced top/bottom y-axis padding
#
# Dot size  = raw mean MI activity for sender→receiver pair
# Dot color = max-normalized MI score within each MI, range 0–1
#
# Max_normalized_score[m, pair] =
#     mean_activity[m, pair] / max_pair mean_activity[m, pair]
# ============================================================

import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# Analysis parameters
# ============================================================

TOP_K_PER_MI = 3
MIN_EDGES_PER_PAIR = 30
EXCLUDE_SAME_CELL_TYPE = False

USE_MAXNORM_THRESHOLD = False
MI_LEVEL_AGG_THRESHOLD = 0.6

# Ranking options:
#   "maxnorm"      : rank by max-normalized score
#   "activity"     : rank by raw mean MI activity
#   "contribution" : rank by within-MI contribution
RANK_BY = "maxnorm"

# MI display order
CUSTOM_MI_ORDER = [
    "MI-25",
    "MI-15",
    "MI-20",
    "MI-1",
    "MI-9",
    "MI-13",
    "MI-8",
    "MI-23",
    "MI-28",
    "MI-16",
    "MI-17",
    "MI-22",
    "MI-5",
    "MI-27",
    "MI-18",
    "MI-14",
    "MI-4",
    "MI-10",
    "MI-2",
    "MI-6",
    "MI-11",
    "MI-12",
    "MI-24",
    "MI-21",
    "MI-26",
    "MI-7",
    "MI-30",
    "MI-19",
    "MI-3",
    "MI-29",
][::-1]

CELL_TYPE_COL_CANDIDATES = [
    globals().get("CELL_TYPE_COL", None),
    "celltype",
    "cell_type",
    "cell.types",
    "cell_class",
    "cellclass",
    "CellType",
    "celltype_major",
    "major_celltype",
    "annotation",
]

CELL_TYPE_COL_CANDIDATES = [x for x in CELL_TYPE_COL_CANDIDATES if x is not None]

EPS = 1e-8

out_dir = Path(run_dirs["run_dir"])
out_dir.mkdir(parents=True, exist_ok=True)


# ============================================================
# Font-size settings
# ------------------------------------------------------------
# Large fonts are used because the complete figure contains many MI and cell-type-pair labels.
# ============================================================

FONT_TITLE = 120
FONT_AXIS_LABEL = 104
FONT_XTICK = 88
FONT_YTICK = 80
FONT_CBAR_LABEL = 88
FONT_CBAR_TICK = 72
FONT_LEGEND = 72
FONT_LEGEND_TITLE = 80


# ============================================================
# Helper functions
# ============================================================

def _to_numpy(x):
    """Convert torch tensor / numpy-like object to numpy array."""
    if hasattr(x, "detach"):
        x = x.detach()
    if hasattr(x, "cpu"):
        x = x.cpu()
    return np.asarray(x)


def _get_field(obj, key):
    """Get field from dict-like or PyG Data-like object."""
    if isinstance(obj, dict):
        return obj[key]
    if hasattr(obj, key):
        return getattr(obj, key)
    try:
        return obj[key]
    except Exception as e:
        raise KeyError(f"Cannot find field `{key}` in object of type {type(obj)}") from e


def _has_field(obj, key):
    if isinstance(obj, dict):
        return key in obj
    if hasattr(obj, key):
        return True
    try:
        obj[key]
        return True
    except Exception:
        return False


def _find_celltype_col(adata):
    for col in CELL_TYPE_COL_CANDIDATES:
        if col in adata.obs.columns:
            return col
    return None


def _natural_mi_order(mi_name):
    try:
        return int(str(mi_name).replace("MI-", "").replace("MI_", ""))
    except Exception:
        return 10**9


def _format_pair_label(label):
    """
    Format sender→receiver label clearly.
    The arrow is kept on the same line.
    """
    label = str(label)
    label = label.replace("-->", "→")
    label = label.replace("->", "→")
    label = label.replace("=>", "→")
    label = label.replace("→", " → ")
    label = " ".join(label.split())
    label = label.replace(" → ", "  →  ")
    return label


def _scale_size(values, global_values=None, min_size=35, max_size=560, clip=True):
    """
    Scale raw activity values to dot sizes.
    If clip=True, legend values outside the observed range are clipped.
    """
    values = np.asarray(values, dtype=float)

    if global_values is None:
        global_values = values

    global_values = np.asarray(global_values, dtype=float)

    vmin = np.nanmin(global_values)
    vmax = np.nanmax(global_values)

    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmax <= vmin:
        return np.full_like(values, 160.0, dtype=float)

    values_use = values.copy()
    if clip:
        values_use = np.clip(values_use, vmin, vmax)

    return min_size + (values_use - vmin) / (vmax - vmin) * (max_size - min_size)


# ============================================================
# Load SpiderNet outputs
# ============================================================

factor_path = out_dir / "Factor_envir_use.npy"
spidernet_data_path = out_dir / "SpiderNet_data_pyg_list.pkl"
adata_list_path = out_dir / "adata_list.pkl"

if not input_path(factor_path).exists():
    raise FileNotFoundError(f"Cannot find Factor_envir_use.npy: {factor_path}")

if not input_path(spidernet_data_path).exists():
    raise FileNotFoundError(f"Cannot find SpiderNet_data_pyg_list.pkl: {spidernet_data_path}")

if not input_path(adata_list_path).exists():
    raise FileNotFoundError(f"Cannot find adata_list.pkl: {adata_list_path}")

Factor_envir_use = np.load(input_path(factor_path))

with open(input_path(spidernet_data_path), "rb") as f:
    spidernet_data_list = pickle.load(f)

with open(input_path(adata_list_path), "rb") as f:
    adata_list = pickle.load(f)

num_edges_total = Factor_envir_use.shape[0]
num_mi = Factor_envir_use.shape[1]
mi_names = [f"MI-{i + 1}" for i in range(num_mi)]

print("Factor_envir_use shape:", Factor_envir_use.shape)
print("Number of batches:", len(spidernet_data_list))
print("Number of adata objects:", len(adata_list))


# ============================================================
# Reconstruct edge-level sender and receiver cell types
# ============================================================

sender_all = []
receiver_all = []
batch_all = []

for batch_idx, data_cur in enumerate(spidernet_data_list):
    edge_index = _to_numpy(_get_field(data_cur, "edge_index"))

    if edge_index.ndim != 2:
        raise ValueError(
            f"edge_index for batch {batch_idx} should be 2D, got {edge_index.shape}"
        )

    if edge_index.shape[0] == 2 and edge_index.shape[1] != 2:
        edge_index = edge_index.T

    if edge_index.shape[1] != 2:
        raise ValueError(
            f"edge_index for batch {batch_idx} should have shape [E, 2] or [2, E], "
            f"got {edge_index.shape}"
        )

    adata_cur = adata_list[batch_idx]
    celltype_col = _find_celltype_col(adata_cur)

    if celltype_col is not None:
        celltypes_cur = np.asarray(adata_cur.obs[celltype_col].astype(str))
        if batch_idx == 0:
            print(f"Using cell-type column from adata.obs: {celltype_col}")

    elif _has_field(data_cur, "cell_class"):
        celltypes_cur = _to_numpy(_get_field(data_cur, "cell_class")).astype(str)
        if batch_idx == 0:
            print("Using cell-type field from data_cur['cell_class']")

    else:
        raise ValueError(
            f"Cannot find cell-type annotation for batch {batch_idx}. "
            f"Tried adata.obs columns: {CELL_TYPE_COL_CANDIDATES}, "
            f"and data_cur['cell_class']."
        )

    edge_index = edge_index.astype(int)

    if edge_index.max() >= len(celltypes_cur):
        raise ValueError(
            f"edge_index contains node index {edge_index.max()}, "
            f"but batch {batch_idx} only has {len(celltypes_cur)} cells."
        )

    sender_cur = celltypes_cur[edge_index[:, 0]]
    receiver_cur = celltypes_cur[edge_index[:, 1]]

    sender_all.append(sender_cur)
    receiver_all.append(receiver_cur)
    batch_all.append(np.repeat(batch_idx, edge_index.shape[0]))

sender_all = np.concatenate(sender_all)
receiver_all = np.concatenate(receiver_all)
batch_all = np.concatenate(batch_all)

if len(sender_all) != num_edges_total:
    if "results" in globals() and isinstance(results, dict) and "factor_envir_list" in results:
        print(
            "Warning: reconstructed edge count does not match Factor_envir_use.npy. "
            "Using np.vstack(results['factor_envir_list']) instead."
        )
        Factor_envir_use = np.vstack(results["factor_envir_list"])
        num_edges_total = Factor_envir_use.shape[0]
        num_mi = Factor_envir_use.shape[1]
        mi_names = [f"MI-{i + 1}" for i in range(num_mi)]

    if len(sender_all) != num_edges_total:
        raise ValueError(
            "The number of reconstructed edges does not match Factor_envir_use.\n"
            f"Reconstructed edges: {len(sender_all)}\n"
            f"Factor_envir_use rows: {num_edges_total}\n"
            "Please check whether SpiderNet_data_pyg_list.pkl and Factor_envir_use.npy "
            "come from the same run."
        )

edge_meta = pd.DataFrame({
    "Batch": batch_all,
    "Sender": sender_all,
    "Receiver": receiver_all,
})

if EXCLUDE_SAME_CELL_TYPE:
    valid_edge_mask = edge_meta["Sender"].values != edge_meta["Receiver"].values
else:
    valid_edge_mask = np.ones(edge_meta.shape[0], dtype=bool)

edge_meta = edge_meta.loc[valid_edge_mask].copy()
Factor_use = Factor_envir_use[valid_edge_mask, :]

edge_meta["Pair"] = (
    edge_meta["Sender"].astype(str) + " → " + edge_meta["Receiver"].astype(str)
)

print("Edges used for summary:", edge_meta.shape[0])
print("Unique sender→receiver pairs:", edge_meta["Pair"].nunique())


# ============================================================
# Compute mean MI activity for each sender→receiver pair
# ============================================================

pair_cat = pd.Categorical(edge_meta["Pair"])
pair_codes = pair_cat.codes
pair_names = np.asarray(pair_cat.categories)
num_pairs = len(pair_names)

pair_counts = np.bincount(pair_codes, minlength=num_pairs)

sum_mat = np.zeros((num_pairs, num_mi), dtype=float)

for mi_idx in range(num_mi):
    sum_mat[:, mi_idx] = np.bincount(
        pair_codes,
        weights=Factor_use[:, mi_idx],
        minlength=num_pairs,
    )

mean_mat = sum_mat / np.maximum(pair_counts[:, None], 1)

activity_df = pd.DataFrame(
    mean_mat.T,
    index=mi_names,
    columns=pair_names,
)

pair_info = (
    edge_meta[["Pair", "Sender", "Receiver"]]
    .drop_duplicates("Pair")
    .set_index("Pair")
    .loc[pair_names]
)

pair_count_s = pd.Series(pair_counts, index=pair_names, name="Pair_edge_count")

keep_pairs = pair_count_s.index[pair_count_s >= MIN_EDGES_PER_PAIR].tolist()

if len(keep_pairs) == 0:
    raise ValueError(
        f"No sender→receiver pair has at least MIN_EDGES_PER_PAIR={MIN_EDGES_PER_PAIR} edges. "
        "Please lower MIN_EDGES_PER_PAIR."
    )

activity_keep = activity_df.loc[:, keep_pairs].copy()

print("Pairs kept after edge-count filtering:", len(keep_pairs))


# ============================================================
# Heatmap-style max-normalized MI score
# ============================================================

mi_max_activity = activity_keep.max(axis=1)

maxnorm_df = activity_keep.div(mi_max_activity + EPS, axis=0)
maxnorm_df = maxnorm_df.replace([np.inf, -np.inf], np.nan).fillna(0)

contribution_df = activity_keep.div(activity_keep.sum(axis=1) + EPS, axis=0)


# ============================================================
# Build full metric table
# ============================================================

full_metric_rows = []

for mi in mi_names:
    tmp = pd.DataFrame({
        "MI": mi,
        "MI_order": _natural_mi_order(mi),
        "Pair": keep_pairs,
        "Sender": pair_info.loc[keep_pairs, "Sender"].values,
        "Receiver": pair_info.loc[keep_pairs, "Receiver"].values,
        "Pair_edge_count": pair_count_s.loc[keep_pairs].values,
        "Activity": activity_keep.loc[mi, keep_pairs].values,
        "Max_normalized_score": maxnorm_df.loc[mi, keep_pairs].values,
        "MI_contribution": contribution_df.loc[mi, keep_pairs].values,
    })
    full_metric_rows.append(tmp)

full_metric_df = pd.concat(full_metric_rows, axis=0, ignore_index=True)


# ============================================================
# Select top-k sender→receiver pairs for each MI
# ============================================================

if RANK_BY == "maxnorm":
    sort_cols = ["Max_normalized_score", "Activity", "MI_contribution"]
elif RANK_BY == "activity":
    sort_cols = ["Activity", "Max_normalized_score", "MI_contribution"]
elif RANK_BY == "contribution":
    sort_cols = ["MI_contribution", "Activity", "Max_normalized_score"]
else:
    raise ValueError("RANK_BY must be one of: 'maxnorm', 'activity', 'contribution'.")

top_rows = []

for mi in mi_names:
    tmp = full_metric_df.loc[full_metric_df["MI"] == mi].copy()

    if USE_MAXNORM_THRESHOLD:
        tmp = tmp.loc[tmp["Max_normalized_score"] >= MI_LEVEL_AGG_THRESHOLD].copy()

    tmp = tmp.sort_values(sort_cols, ascending=False)
    tmp = tmp.head(TOP_K_PER_MI)
    tmp["Top_rank_within_MI"] = np.arange(1, tmp.shape[0] + 1)

    if tmp.shape[0] > 0:
        top_rows.append(tmp)

if len(top_rows) == 0:
    raise ValueError(
        "No MI-pair combinations were selected. "
        "Try setting USE_MAXNORM_THRESHOLD=False or lowering MI_LEVEL_AGG_THRESHOLD."
    )

top_metric_df = pd.concat(top_rows, axis=0, ignore_index=True)


# ============================================================
# Define custom MI order
# ============================================================

custom_set = set(CUSTOM_MI_ORDER)

missing_custom_mis = [mi for mi in CUSTOM_MI_ORDER if mi not in mi_names]
extra_mis = [mi for mi in mi_names if mi not in custom_set]

if len(missing_custom_mis) > 0:
    print("Warning: these MIs are in CUSTOM_MI_ORDER but not in the data:")
    print(missing_custom_mis)

if len(extra_mis) > 0:
    print("Warning: these MIs are in the data but not in CUSTOM_MI_ORDER; appended at the end:")
    print(extra_mis)

mi_order = [mi for mi in CUSTOM_MI_ORDER if mi in mi_names] + sorted(extra_mis, key=_natural_mi_order)


# ============================================================
# Define sender→receiver pair order
# ------------------------------------------------------------
# y-axis: union of top-k sender→receiver pairs across MIs.
# Pair order follows the custom MI order first, then top rank.
# ============================================================

mi_order_map = {mi: i for i, mi in enumerate(mi_order)}

top_metric_df["MI_custom_order"] = top_metric_df["MI"].map(mi_order_map)

pair_order_df = (
    top_metric_df
    .sort_values(["MI_custom_order", "Top_rank_within_MI", "Pair"])
    .drop_duplicates("Pair")
)

pair_order = pair_order_df["Pair"].tolist()


# Dense plot:
# Use full_metric_df for all selected pairs across all MIs.
plot_df = full_metric_df.loc[
    full_metric_df["Pair"].isin(pair_order)
].copy()

plot_df["MI"] = pd.Categorical(plot_df["MI"], categories=mi_order, ordered=True)
plot_df["Pair"] = pd.Categorical(plot_df["Pair"], categories=pair_order, ordered=True)

plot_df["x"] = plot_df["MI"].cat.codes
plot_df["y"] = plot_df["Pair"].cat.codes

top_key = set(zip(top_metric_df["MI"].astype(str), top_metric_df["Pair"].astype(str)))

plot_df["Is_top_pair"] = [
    (str(mi), str(pair)) in top_key
    for mi, pair in zip(plot_df["MI"], plot_df["Pair"])
]


# ============================================================
# Save metric tables
# ============================================================

threshold_tag = (
    f"threshold{MI_LEVEL_AGG_THRESHOLD}"
    if USE_MAXNORM_THRESHOLD
    else "noThreshold"
)

full_metric_path = (
    out_dir
    / f"MI_sender_receiver_pair_full_metrics_maxnorm_minEdges{MIN_EDGES_PER_PAIR}.csv"
)

top_metric_path = (
    out_dir
    / f"MI_sender_receiver_pair_top{TOP_K_PER_MI}_{RANK_BY}_{threshold_tag}_metrics_customMIorder.csv"
)

full_metric_df.to_csv(output_path(full_metric_path), index=False)
top_metric_df.to_csv(output_path(top_metric_path), index=False)

print(f"Saved full metric table: {full_metric_path}")
print(f"Saved top-k metric table: {top_metric_path}")

display(
    top_metric_df[
        [
            "MI",
            "Top_rank_within_MI",
            "Sender",
            "Receiver",
            "Pair_edge_count",
            "Activity",
            "Max_normalized_score",
            "MI_contribution",
        ]
    ].head(50)
)



### Export and display the MI sender-receiver dense bubble figure

The following plot uses the A3 metric tables and retains the configured MI order, top-three pair selection, minimum edge count, max-normalization, and dot sizes. With the default settings, the final vector PDF is `MI_sender_receiver_top3_maxnorm_noThreshold_maxnorm_dense_bubble_plot_customMIorder_hugefont_dotsize2x.pdf` in the current run directory. A compact preview is exported from the same figure and displayed immediately after the plotting cell, avoiding an oversized inline rendering of the full-resolution canvas.


In [ ]:
# ============================================================
# Plot swapped-axis dense bubble dot plot
# ============================================================

norm = plt.Normalize(vmin=0, vmax=1)

# Enlarge all dot sizes by 2x.
# Note: matplotlib scatter `s` is area-based.
DOT_SIZE_MULTIPLIER = 15.0

# Larger canvas for extremely large fonts.
# TOP_K_PER_MI is now 3, so pair_order is smaller than before.
fig_width = max(38, 2.65 * len(mi_order) + 12)
fig_height = max(26, 1.9 * len(pair_order) + 8)

plt.close()
fig, ax = plt.subplots(figsize=(fig_width, fig_height), facecolor="white")
ax.set_facecolor("white")

sizes_all = _scale_size(
    plot_df["Activity"].values,
    global_values=full_metric_df["Activity"].values,
    min_size=55,
    max_size=760,
)

# Non-top dots are still shown, but more subtle.
sizes_all = np.where(
    plot_df["Is_top_pair"].values,
    sizes_all,
    np.maximum(sizes_all * 0.65, 35),
)

# Make all dot sizes 2x larger than before.
sizes_all = sizes_all * DOT_SIZE_MULTIPLIER

non_top_df = plot_df.loc[~plot_df["Is_top_pair"]].copy()
top_df = plot_df.loc[plot_df["Is_top_pair"]].copy()

non_top_sizes = sizes_all[~plot_df["Is_top_pair"].values]
top_sizes = sizes_all[plot_df["Is_top_pair"].values]

# Background dots: all non-top MI-pair values.
# edgecolors="face" makes boundary color identical to fill color.
sc_bg = ax.scatter(
    non_top_df["x"],
    non_top_df["y"],
    s=non_top_sizes,
    c=non_top_df["Max_normalized_score"].values,
    cmap="Reds",
    norm=norm,
    edgecolors="face",
    linewidths=0.8,
    alpha=0.45,
)

# Highlighted dots: top-k pairs for each MI.
# edgecolors="face" makes boundary color identical to fill color.
sc = ax.scatter(
    top_df["x"],
    top_df["y"],
    s=top_sizes,
    c=top_df["Max_normalized_score"].values,
    cmap="Reds",
    norm=norm,
    edgecolors="face",
    linewidths=1.5,
    alpha=0.98,
)

# x-axis: MIs in custom order.
ax.set_xticks(np.arange(len(mi_order)))
ax.set_xticklabels(
    mi_order,
    rotation=90,
    ha="center",
    va="top",
    fontsize=FONT_XTICK,
)

# y-axis: sender→receiver pairs.
ax.set_yticks(np.arange(len(pair_order)))
ax.set_yticklabels(
    [_format_pair_label(p) for p in pair_order],
    fontsize=FONT_YTICK,
)

# Reduce the distance from the first row to the top border
# and from the last row to the bottom border.
# Since y-axis is inverted, top is -0.25 and bottom is len(pair_order)-0.75.
ax.set_ylim(len(pair_order) - 0.75, -0.25)

ax.set_xlabel("Meta-interaction IDs", fontsize=FONT_AXIS_LABEL, labelpad=45)
ax.set_ylabel("Sender  →  receiver cell-type pairs", fontsize=FONT_AXIS_LABEL, labelpad=45)

ax.set_title(
    f"MI-specific sender→receiver cell-type interactions\n"
    f"Rows are union of top {TOP_K_PER_MI} pairs per MI; all MIs are shown for each selected pair",
    fontsize=FONT_TITLE,
    pad=70,
)

ax.grid(axis="both", color="#E6E6E6", linewidth=1.5)
ax.set_axisbelow(True)

for spine in ax.spines.values():
    spine.set_linewidth(2.2)
    spine.set_color("#333333")

# Colorbar.
cbar = plt.colorbar(sc, ax=ax, fraction=0.025, pad=0.02)
cbar.set_label(
    "Max-normalized MI score\nwithin each MI",
    fontsize=FONT_CBAR_LABEL,
    labelpad=42,
)
cbar.ax.tick_params(labelsize=FONT_CBAR_TICK)

# ============================================================
# Fixed dot-size legend: 0.1, 0.2, 0.3
# ------------------------------------------------------------
# Dot edge color also matches fill color in legend.
# Dot sizes are also enlarged by 2x.
# ============================================================

legend_values = np.array([0.1, 0.2, 0.3])

handles = []
labels = []

for v in legend_values:
    legend_size = float(
        _scale_size(
            np.array([v]),
            global_values=full_metric_df["Activity"].values,
            min_size=55,
            max_size=760,
            clip=True,
        )[0]
    ) * DOT_SIZE_MULTIPLIER

    handles.append(
        ax.scatter(
            [],
            [],
            s=legend_size,
            color="#BDBDBD",
            edgecolors="face",
            linewidths=1.2,
        )
    )
    labels.append(f"{v:.1f}")

ax.legend(
    handles,
    labels,
    title="Mean MI\nactivity",
    loc="upper left",
    bbox_to_anchor=(1.03, 1.0),
    frameon=False,
    fontsize=FONT_LEGEND,
    title_fontsize=FONT_LEGEND_TITLE,
    labelspacing=1.7,
    handletextpad=1.4,
)

plt.tight_layout()

pdf_path = (
    out_dir
    / f"MI_sender_receiver_top{TOP_K_PER_MI}_{RANK_BY}_{threshold_tag}_maxnorm_dense_bubble_plot_customMIorder_hugefont_dotsize2x.pdf"
)

png_path = (
    out_dir
    / f"MI_sender_receiver_top{TOP_K_PER_MI}_{RANK_BY}_{threshold_tag}_maxnorm_dense_bubble_plot_customMIorder_hugefont_dotsize2x.png"
)

plt.savefig(output_path(pdf_path), bbox_inches="tight", dpi=300)
plt.savefig(output_path(png_path), bbox_inches="tight", dpi=300)

# Use a bounded-size preview for notebook display of this large figure.
mi_pair_dense_pdf_path = pdf_path
mi_pair_dense_preview_path = pdf_path.with_name(pdf_path.stem + "_preview.png")
fig.savefig(
    output_path(mi_pair_dense_preview_path),
    bbox_inches="tight",
    dpi=1800 / fig.get_figwidth(),
)
plt.close(fig)

print(f"Saved custom-MI-order dense bubble plot PDF: {pdf_path}")
print(f"Saved custom-MI-order dense bubble plot PNG: {png_path}")


# ============================================================
# Optional: concise interpretation table
# ============================================================

summary_table = (
    top_metric_df
    .sort_values(["MI_custom_order", "Top_rank_within_MI"])
    .groupby("MI", sort=False)
    .apply(
        lambda x: "; ".join(
            [
                f"{row['Sender']}→{row['Receiver']} "
                f"(norm={row['Max_normalized_score']:.2f}, activity={row['Activity']:.3f})"
                for _, row in x.iterrows()
            ]
        )
    )
    .reset_index(name="Top_sender_receiver_pairs")
)

summary_path = (
    out_dir
    / f"MI_sender_receiver_pair_top{TOP_K_PER_MI}_{RANK_BY}_{threshold_tag}_summary_customMIorder.csv"
)

summary_table.to_csv(output_path(summary_path), index=False)

print(f"Saved concise summary table: {summary_path}")
display(summary_table.head(30))

In [ ]:
# Display the compact preview of the figure exported immediately above.
from IPython.display import Image, display

print(f"MI sender-receiver dense bubble plot: {mi_pair_dense_pdf_path}")
display(Image(filename=str(input_path(mi_pair_dense_preview_path)), width=1200))


In [ ]:
## Load `Factor_envir_use.npy` and processed objects for MI enrichment analysis
Factor_envir_use = np.load(input_path(run_dirs['run_dir'] / "Factor_envir_use.npy"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

Factor_envir_use_list = [Factor_envir_use]

for res_index in range(1):
    Factor_envir_use_show = Factor_envir_use_list[res_index]
    if res_index == 0:
        type_show = "SpiderNet"
    else:
        type_show = "Initial"

    ## Calculate the mean MI of each time point
    timepoint_unique = np.sort(np.unique(processed.batch_cell.astype(str)))

    timepoint_edge = np.hstack([
        np.array(
            processed.adata_list[batch_index_cur].obs['age'].astype(str)
        )[processed.spidernet_data[batch_index_cur]['edge_index'][:, 0].cpu().detach().numpy()]
        for batch_index_cur in range(len(processed.spidernet_data))
    ])

    MI_mean_list = []
    for i in range(len(timepoint_unique)):
        cur_mask = (timepoint_edge == timepoint_unique[i])
        MI_mean_list.append(
            np.mean(
                Factor_envir_use_show[cur_mask, :],
                axis=0
            )
        )

    MI_mean_pd = pd.DataFrame(
        np.vstack(MI_mean_list),
        index=timepoint_unique,
        columns=["MI_" + str(i + 1) for i in range(Factor_envir_use_show.shape[1])]
    )

    ## Convert age index to numeric and sort by age
    MI_mean_pd.index = pd.to_numeric(MI_mean_pd.index, errors="coerce")
    MI_mean_pd = MI_mean_pd[~MI_mean_pd.index.isna()]
    MI_mean_pd = MI_mean_pd.sort_index()

    ## Max-normalize each MI
    MI_mean_pd_maxnorm = MI_mean_pd.div(MI_mean_pd.max(axis=0), axis=1)

    ## Numeric age values for downstream weighted mean calculations
    age_values = MI_mean_pd_maxnorm.index.to_numpy(dtype=float)

    ## Order MIs by weighted aging score
    weighted_aged_MI = []
    for i in range(MI_mean_pd_maxnorm.shape[1]):
        values = MI_mean_pd_maxnorm.iloc[:, i].to_numpy(dtype=float)
        total = np.sum(values)

        if total == 0 or np.isnan(total):
            weighted_aged_MI.append(np.inf)
        else:
            weighted_aged_MI.append(np.sum(values / total * age_values))

    MI_mean_pd_maxnorm = MI_mean_pd_maxnorm.iloc[:, np.argsort(weighted_aged_MI)]

    MI_mean_pd_maxnorm.to_csv(output_path(run_dirs["run_dir"] / "MI_mean_by_age_maxnorm.csv"))

    ## ===========================
    ## Show the heatmap
    plt.rcParams['pdf.fonttype'] = 42
    plt.rcParams['mathtext.fontset'] = 'dejavuserif'
    plt.rcParams['font.family'] = 'arial'
    ## ===========================

    plt.figure(figsize=(9, 10))
    cmap = LinearSegmentedColormap.from_list(
        "custom_coolwarm_graycenter",
        ["#4575b4", "#f0f0f0", "#d73027"],
        N=256
    )
    sns.heatmap(MI_mean_pd_maxnorm.T, cmap=cmap)

    # ==== Rename y-ticks from MI_XX to MI-XX ====
    yticks_old = MI_mean_pd_maxnorm.T.index
    yticks_new = [t.replace("MI_", "MI-") for t in yticks_old]
    plt.gca().set_yticklabels(yticks_new)
    # ==================================

    plt.xticks(fontsize=18)
    plt.yticks(fontsize=18)
    plt.xlabel('Age', fontsize=22)
    plt.ylabel("Meta-interaction IDs", fontsize=22)
    plt.savefig(
        output_path(str(run_dirs['run_dir']) + '/MI_heatmap_' + str(type_show) + '.pdf'),
        format='pdf', bbox_inches='tight', dpi=300
    )
    plt.show()
    plt.close()

    ## ===========================
    ## mean age per MI (stem plot)
    ## ===========================
    mean_age_MI = []
    for i in range(MI_mean_pd_maxnorm.shape[1]):
        values = MI_mean_pd_maxnorm.iloc[:, i].to_numpy(dtype=float)
        total = np.sum(values)

        if total == 0 or np.isnan(total):
            mean_age_MI.append(np.nan)
        else:
            mean_age_MI.append(np.sum(values / total * age_values))

    x = np.arange(len(mean_age_MI))

    plt.close()
    fig, ax = plt.subplots(figsize=(1.5, 8))

    ax.stem(
        x,
        mean_age_MI,
        orientation="horizontal",
        linefmt="C1-",
        markerfmt="C1o",
        basefmt="C3-"
    )

    plt.xlim(16, 21)
    plt.xticks(rotation=270)

    ## invert y-axis so that MI sorted top-to-bottom
    plt.gca().invert_yaxis()
    plt.savefig(
        output_path(str(run_dirs['run_dir']) + '/mean_age_MI_' + str(type_show) + '.png'),
        format='png', bbox_inches='tight', dpi=300
    )
    plt.show()
    plt.close()

In [ ]:
##Load LR_loading_pathway
# LR_loading_pathway.to_csv(os.path.join(file_savepath_main, "LR_loading_pathway.csv"))
LR_loading_pathway = pd.read_csv(input_path(run_dirs['run_dir'] / "LR_loading_pathway.csv"),index_col=0)
MI_orderbyage = MI_mean_pd_maxnorm.columns.tolist()
##Change the "MI_" to "MI-" in the index of LR_loading_pathway
MI_orderbyage = [x.replace("MI_", "MI-") for x in MI_orderbyage]
LR_loading_pathway = LR_loading_pathway.loc[MI_orderbyage, :]
LR_loading_pathway = LR_loading_pathway.iloc[:,
                     np.argsort(np.array(np.max(LR_loading_pathway, axis=0)))[::-1]
                     ]
# Refined pathway ordering
argmax_1 = np.argmax(np.array(LR_loading_pathway), axis=0)
max_1 = np.max(np.array(LR_loading_pathway), axis=0)
order_index_LRpathway = []
for argmax_1_cur in np.sort(np.unique(argmax_1)):
    index_cur = np.where(argmax_1 == argmax_1_cur)[0]
    index_cur = index_cur[np.argsort(max_1[index_cur])[::-1]]
    index_cur = index_cur.tolist()
    order_index_LRpathway.extend(index_cur)

LR_loading_pathway = LR_loading_pathway.iloc[:, order_index_LRpathway]


In [ ]:
# Visualization
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
import os
plt.close()
fig, ax = plt.subplots(figsize=(9, 10))
# cmap = LinearSegmentedColormap.from_list(
#     "white_red", ["white", "#FFDFEF", "#EABDE6", "#D69ADE", "#AA60C8"], N=256
# )
cmap = LinearSegmentedColormap.from_list(
    "white_red", ["#FCF5F0", "#F9B2BC", "#F6689F", "#C31988", "#510269"], N=256
)

data = LR_loading_pathway.values
vmin = data.min()
vmax = min(0.4, np.max(data) * 0.7)
norm = TwoSlopeNorm(vmin=vmin, vcenter=(vmin + vmax) / 2, vmax=vmax)

mesh = ax.pcolormesh(
    np.arange(data.shape[1] + 1),
    np.arange(data.shape[0] + 1),
    data,
    cmap=cmap,
    norm=norm,
    edgecolors="#B6B9BA",
    linewidth=1.0
)

ax.set_xticks(np.arange(data.shape[1]) + 0.5)
# ax.set_xticklabels(LR_loading_pathway.columns, rotation=60, fontsize=22)
ax.set_xticklabels(
    LR_loading_pathway.columns,
    rotation=60,
    fontsize=22,
    ha="left",
    rotation_mode="anchor"
)
ax.set_yticks(np.arange(data.shape[0]) + 0.5)
ax.set_yticklabels(LR_loading_pathway.index, fontsize=22)
ax.xaxis.set_ticks_position("top")
ax.xaxis.set_label_position("top")
ax.invert_yaxis()

plt.colorbar(mesh, ax=ax)
plt.tight_layout()

save_png = os.path.join(run_dirs['run_dir'], "LR_loading_pathway.png")
plt.savefig(output_path(save_png), format="png", bbox_inches="tight", dpi=300)

save_pdf = os.path.join(run_dirs['run_dir'], "LR_loading_pathway.pdf")
plt.savefig(output_path(save_pdf), format="pdf", bbox_inches="tight", dpi=300)

plt.show()
plt.close()


## Save the cell types and MI_edge_index_all_choose for downstream visualization

In [ ]:
cell_meta_all = []
for batch_index in range(len(processed.adata_list)):
    cell_meta_all.append(processed.adata_list[batch_index].obs[['celltype', 'age']])
cell_meta_all = pd.concat(cell_meta_all, axis=0)
cell_meta_all['cellname'] = cell_meta_all.index
print(cell_meta_all)
##save as csv
cell_meta_all.to_csv(output_path(str(run_dirs['run_dir']) + "/cell_meta_all.csv"), index=False)

MI_show_threshold = 0.7
MI_edge_index_all_choose = []
for batch_index_cur in range(len(processed.spidernet_data)):
    Factor_envir_curbatch = results['factor_envir_list'][batch_index_cur]
    edge_index_curbatch = processed.spidernet_data[batch_index_cur]['edge_index'].cpu().detach().numpy()
    cellname_curbatch = np.array(processed.adata_list[batch_index_cur].obs.index)
    cellname_edge_index_curbatch = cellname_curbatch[edge_index_curbatch]
    Factor_envir_curbatch_largenum = np.sum(Factor_envir_curbatch >= MI_show_threshold, axis=1)
    Factor_envir_curbatch_largenum_nonzeroidnex = np.where(Factor_envir_curbatch_largenum > 0)[0]
    MI_edge_index_all_choose_df = pd.DataFrame({"source": cellname_edge_index_curbatch[Factor_envir_curbatch_largenum_nonzeroidnex, 0],
                                                "target": cellname_edge_index_curbatch[Factor_envir_curbatch_largenum_nonzeroidnex, 1],
                                                "interaction_num": Factor_envir_curbatch_largenum[Factor_envir_curbatch_largenum_nonzeroidnex]})
    MI_edge_index_all_choose.append(MI_edge_index_all_choose_df)
MI_edge_index_all_choose = pd.concat(MI_edge_index_all_choose, axis=0)
print(MI_edge_index_all_choose)
##save as csv
MI_edge_index_all_choose.to_csv(output_path(str(run_dirs['run_dir']) + "/MI_edge_index_all_choose.csv"), index=False)

## Optional age-focused downstream analyses

The next sections use SpiderNet outputs to quantify aging-related signals. They cover feature aggregation, cell-age prediction, coefficient inspection, cell-type-pair summaries, and spatial visualization of a selected MI.


## Optional B1. Predict cell age from aggregated MI features

In [ ]:
## Aggregate sender- and receiver-side MI features, then fit age-prediction models
from torch_scatter import scatter_mean

print("Performing MI aggregation...")
MI_SR_agg_all = []
for slice_index in range(len(results['factor_envir_list'])):
    Factor_envir_cur = results['factor_envir_list'][slice_index]
    Factor_envir_cur = torch.tensor(Factor_envir_cur, dtype=torch.float32, device=device)
    edge_index_cur = processed.spidernet_data[slice_index]['edge_index'].to(
        device=Factor_envir_cur.device, dtype=torch.int64
    )
    num_cell_cur = processed.spidernet_data[slice_index].x.shape[0]
    ##
    MI_receiver_agg_cur = scatter_mean(Factor_envir_cur,
                                   edge_index_cur[:, 1], dim=0,
                                   dim_size=num_cell_cur).to("cpu").numpy()
    MI_sender_agg_cur = scatter_mean(Factor_envir_cur,
                                 edge_index_cur[:, 0], dim=0,
                                 dim_size=num_cell_cur).to("cpu").numpy()
    MI_SR_agg_cur = np.hstack([MI_sender_agg_cur, MI_receiver_agg_cur])
    MI_SR_agg_all.append(MI_SR_agg_cur)
MI_SR_agg_all = np.vstack(MI_SR_agg_all)
print("MI aggregation completed.")

cellage = [np.array(processed.adata_list[i].obs['age']) for i in range(len(processed.adata_list))]
celltype_list = [np.array(processed.adata_list[i].obs['celltype']) for i in range(len(processed.adata_list))]
cellage_all = np.hstack(cellage)
celltype_all = np.hstack(celltype_list)
celltype_all_unique = np.sort(np.unique(celltype_all))


In [ ]:
## Baseline 1: PCA on gene expression
geneexp_all = [processed.adata_list[i].X.toarray() if isinstance(processed.adata_list[i].X, np.ndarray) == False else processed.adata_list[i].X for i in range(len(processed.adata_list))]
geneexp_all = np.vstack(geneexp_all)
##
from sklearn.decomposition import PCA
# PCA
print("Performing PCA...")
geneexp_all_zscore = (geneexp_all - np.mean(geneexp_all, axis=0)) / (np.std(geneexp_all, axis=0) + 1e-10)
pca_model = PCA(n_components=20)
geneexp_all_pca = pca_model.fit_transform(geneexp_all_zscore)
print("PCA completed.")

X_dict = {"SpiderNet": MI_SR_agg_all,
            "PCA": geneexp_all_pca}

In [ ]:
## Baseline 2: COMMOT pathway scores
# COMMOT_path_main = "D:/SpiderNet/Results/AgingBrain/COMMOT/"
COMMOT_path_main = OUTPUT_ROOT / "COMMOT"

import os
import numpy as np
import scanpy as sc
import scipy.sparse as sp
import torch

## -------------------------
## Step 1. Match SpiderNet slices to COMMOT files
## -------------------------
COMMOT_data_index = []
numcell_adata = []

for slice_index in range(len(processed.adata_list)):
    adata_sub = processed.adata_list[slice_index]
    numcell_adata.append(adata_sub.n_obs)

numcell_COMMOT_data = []
for slice_index in range(len(processed.adata_list)):
    time_cur = np.unique(processed.adata_list[slice_index].obs['age'])[0]
    COMMOT_LRscore_path = os.path.join(
        COMMOT_path_main,
        # f"Aging_sample{slice_index+1}_COMMOT_cellchat.h5ad"
        f"aging_coronal_age{time_cur}_commot.h5ad"
    )
    COMMOT_adata = sc.read_h5ad(input_path(COMMOT_LRscore_path))
    numcell_COMMOT_data.append(COMMOT_adata.n_obs)

for slice_index in range(len(processed.adata_list)):
    matched_idx = np.where(np.array(numcell_COMMOT_data) == numcell_adata[slice_index])[0][0]
    COMMOT_data_index.append(matched_idx)

print("COMMOT_data_index:", COMMOT_data_index)
print("Unique matched COMMOT files:", len(np.unique(COMMOT_data_index)))

## -------------------------
## Step 2. Collect the union of pathway columns across all matched COMMOT files
## -------------------------
def get_commot_pathway_keys(commot_adata):
    keys = list(commot_adata.obsp.keys())
    total_key = "commot-cellchat-total-total"
    pathway_keys = [
        k for k in keys
        if k.startswith("commot-cellchat-")
        and len(k.split("-")) == 3   # commot-cellchat-<PATHWAY>
        and k != total_key
    ]
    return sorted(pathway_keys)

COMMOT_pathway_union_set = set()

for slice_index in range(len(processed.adata_list)):
    # COMMOT_data_index_cur = COMMOT_data_index[slice_index]
    # COMMOT_LRscore_path = os.path.join(
    #     COMMOT_path_main,
    #     f"Aging_sample{COMMOT_data_index_cur+1}_COMMOT_cellchat.h5ad"
    # )
    time_cur = np.unique(processed.adata_list[slice_index].obs['age'])[0]
    COMMOT_LRscore_path = os.path.join(
        COMMOT_path_main,
        f"aging_coronal_age{time_cur}_commot.h5ad"
    )
    COMMOT_adata = sc.read_h5ad(input_path(COMMOT_LRscore_path))
    pathway_keys_cur = get_commot_pathway_keys(COMMOT_adata)
    COMMOT_pathway_union_set.update(pathway_keys_cur)

COMMOT_pathway_union = sorted(COMMOT_pathway_union_set)
COMMOT_pathway_to_col = {k: i for i, k in enumerate(COMMOT_pathway_union)}

print("Number of union pathway columns:", len(COMMOT_pathway_union))
print("First few pathway keys:", COMMOT_pathway_union[:10])

## Optional: keep pathway names for later interpretation
COMMOT_pathway_names_clean = [
    k.replace("commot-cellchat-", "") for k in COMMOT_pathway_union
]

## -------------------------
## Step 3. Build per-slice edge-level and cell-level matrices using the union columns
## -------------------------
COMMOT_LRscore_pathway_list = []          # per-slice: (E, P_union) edge-level
COMMOT_LRscore_pathway_agg_list = []      # per-slice: (Ncell, 2*P_union) [receiver_agg | sender_agg]

for slice_index in range(len(processed.adata_list)):
    COMMOT_data_index_cur = COMMOT_data_index[slice_index]
    time_cur = np.unique(processed.adata_list[slice_index].obs['age'])[0]
    # COMMOT_LRscore_path = os.path.join(
    #     COMMOT_path_main,
    #     f"Aging_sample{COMMOT_data_index_cur+1}_COMMOT_cellchat.h5ad"
    # )
    COMMOT_LRscore_path = os.path.join(
        COMMOT_path_main,
        f"aging_coronal_age{time_cur}_commot.h5ad"
    )
    COMMOT_adata = sc.read_h5ad(input_path(COMMOT_LRscore_path))

    # Edge index (E, 2) from SpiderNet graph
    edge_index_cur = processed.spidernet_data[slice_index]["edge_index"]
    edge_np = edge_index_cur.detach().cpu().numpy()
    rows = edge_np[:, 0].astype(np.int64, copy=False)
    cols = edge_np[:, 1].astype(np.int64, copy=False)
    E = rows.shape[0]

    # Pathway keys available in this particular COMMOT file
    pathway_keys_cur = get_commot_pathway_keys(COMMOT_adata)

    # Unified edge-level pathway score matrix: (E, P_union)
    cellpair_pathway_array = np.zeros(
        (E, len(COMMOT_pathway_union)),
        dtype=np.float32
    )

    # Fill only the columns present in this file; missing ones stay 0
    for pathway_key in pathway_keys_cur:
        j = COMMOT_pathway_to_col[pathway_key]
        A = COMMOT_adata.obsp[pathway_key]

        if sp.issparse(A):
            A = A.tocsr()
            cellpair_pathway_array[:, j] = A[rows, cols].A1.astype(np.float32, copy=False)
        else:
            cellpair_pathway_array[:, j] = A[rows, cols].astype(np.float32, copy=False)

    COMMOT_LRscore_pathway_list.append(cellpair_pathway_array)

    # Cell-level aggregation (receiver + sender): (Ncell, 2*P_union)
    num_cell_cur = processed.spidernet_data[slice_index].x.shape[0]

    edge_dst = edge_index_cur[:, 1].to(torch.int64).to(device)
    edge_src = edge_index_cur[:, 0].to(torch.int64).to(device)

    pathway_t = torch.tensor(cellpair_pathway_array, dtype=torch.float32, device=device)

    cell_pathway_receiver_agg = scatter_nanmean(
        pathway_t, edge_dst, dim=0, dim_size=num_cell_cur
    ).to("cpu").numpy()

    cell_pathway_sender_agg = scatter_nanmean(
        pathway_t, edge_src, dim=0, dim_size=num_cell_cur
    ).to("cpu").numpy()

    cell_pathway_agg_cur = np.hstack([
        cell_pathway_receiver_agg,
        cell_pathway_sender_agg
    ])
    COMMOT_LRscore_pathway_agg_list.append(cell_pathway_agg_cur)

## -------------------------
## Step 4. Concatenate all slices
## -------------------------
COMMOT_LRscore_pathway_agg_all = np.vstack(COMMOT_LRscore_pathway_agg_list)
COMMOT_LRscore_pathway_agg_all[np.isnan(COMMOT_LRscore_pathway_agg_all)] = 0.0

print("COMMOT_LRscore_pathway_agg_all shape:", COMMOT_LRscore_pathway_agg_all.shape)

X_dict["COMMOT"] = COMMOT_LRscore_pathway_agg_all

## Optional: save feature names
COMMOT_feature_names = (
    [f"receiver_{x}" for x in COMMOT_pathway_names_clean] +
    [f"sender_{x}" for x in COMMOT_pathway_names_clean]
)

print("Number of COMMOT features:", len(COMMOT_feature_names))
print("First few COMMOT features:", COMMOT_feature_names[:10])

In [ ]:
## Baseline 3: PCA on BANKSY embeddings
from pathlib import Path
import pandas as pd
from scipy.io import mmread
from scipy import sparse
Banksy_path_main = OUTPUT_ROOT / "Banksy"
Banksy_matrix_all = []
for slice_index in range(len(processed.adata_list)):
    print(slice_index)
    age_cur = np.unique(processed.adata_list[slice_index].obs['age'])[0]
    # Banksy_matrix = pd.read_csv("D:/SpiderNet/Results/AgingBrain/Banksy/Banksy_CellIdentity_age" + str(age_cur) + ".csv", index_col=0)
    ##
    mtx_path = Banksy_path_main / f"aging_coronal_age{age_cur}_Banksy_cellidentity_lambda0.2.mtx"
    genes_path = Banksy_path_main / f"aging_coronal_age{age_cur}_Banksy_cellidentity_lambda0.2_genes.csv"
    barcodes_path = Banksy_path_main / f"aging_coronal_age{age_cur}_Banksy_cellidentity_lambda0.2_barcodes.csv"
    
    mat = mmread(input_path(mtx_path)).tocsr()
    
    genes = pd.read_csv(input_path(genes_path))["gene"].tolist()
    barcodes = pd.read_csv(input_path(barcodes_path))["barcode"].tolist()
    
    Banksy_matrix = pd.DataFrame.sparse.from_spmatrix(
        mat,
        index=genes,
        columns=barcodes
    )
    Banksy_matrix = Banksy_matrix.T
    ##
    Banksy_matrix_all.append(Banksy_matrix.loc[processed.adata_list[slice_index].obs.index, :])
Banksy_matrix_all = pd.concat(Banksy_matrix_all, axis=0)
# ##
from sklearn.decomposition import PCA
# PCA
print("Performing PCA of Banksy matrix...")
Banksy_matrix_all_zscore = Banksy_matrix_all
pca_model = PCA(n_components=20)
Banksy_matrix_all_pca = pca_model.fit_transform(Banksy_matrix_all_zscore)
print("PCA of Banksy matrix completed.")

X_dict["Banksy"] = Banksy_matrix_all_pca

In [ ]:
## Baseline 4: NMF-LR on edge-level LR coexpression
# --------------------------------------------------------------
# Global NMF-LR behavior:
#   - DO NOT load existing Factor_LR_list.pkl.
#   - DO NOT overwrite or save Factor_LR_list.pkl.
#   - Always recompute NMF-LR in memory from the current processed.spidernet_data.
#   - Concatenate edge-level LR coexpression matrices from all slices.
#   - Fit one shared/global NMF model on the concatenated matrix.
#   - NMF rank is matched to SpiderNet rank, i.e. train_cfg.dim_envir = 30.
#   - Split global edge factors back to slices.
#   - Aggregate edge-level NMF-LR factors to sender/receiver cell-level features.

import numpy as np
import scipy.sparse as sp
import torch
from sklearn.decomposition import NMF

if "X_dict" not in globals():
    raise NameError("X_dict is not defined. Please run the SpiderNet/PCA baseline cells first.")

if "processed" not in globals():
    raise NameError("processed is not defined. Please run the data-loading cells first.")

if "train_cfg" not in globals():
    raise NameError("train_cfg is not defined. Please run the configuration-loading cells first.")

if "scatter_nanmean" not in globals():
    raise NameError("scatter_nanmean is not defined. Please import it before running this cell.")

NMF_LR_RANDOM_STATE = 0
NMF_LR_MAX_ITER = 1000

# Match NMF-LR rank to SpiderNet rank.
NMF_LR_N_COMPONENTS = int(train_cfg.dim_envir)

if NMF_LR_N_COMPONENTS != 30:
    print(f"[Warning] SpiderNet rank / NMF-LR rank is {NMF_LR_N_COMPONENTS}, not 30.")
else:
    print("Using NMF-LR rank = SpiderNet rank = 30")


def _clean_nonnegative_matrix_for_nmf(X):
    """
    Convert edge-level LR coexpression matrix to a non-negative matrix suitable for sklearn NMF.
    Supports torch.Tensor, scipy sparse matrix, and numpy array.
    """
    if torch.is_tensor(X):
        X = X.detach().cpu().numpy()

    if sp.issparse(X):
        X = X.tocsr(copy=True)
        X.data = np.nan_to_num(X.data, nan=0.0, posinf=0.0, neginf=0.0)
        X.data[X.data < 0] = 0.0
        return X

    X = np.asarray(X, dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X[X < 0] = 0.0
    return X


def build_global_nmflr_factors(spidernet_data_list, n_components, random_state=0, max_iter=1000):
    """
    Fit one shared/global NMF-LR basis after concatenating all slice LR coexpression matrices.

    Returns
    -------
    Factor_LR_list_local : list of np.ndarray
        One array per slice, shape = n_edges_in_slice x n_components.
    """
    lr_mats = []
    edge_counts = []

    for slice_index, data_cur in enumerate(spidernet_data_list):
        if "cellpair_LRpair_neigh" not in data_cur:
            raise KeyError(
                "processed.spidernet_data contains no 'cellpair_LRpair_neigh'. "
                "NMF-LR baseline requires edge-level LR coexpression features."
            )

        X_cur = _clean_nonnegative_matrix_for_nmf(data_cur["cellpair_LRpair_neigh"])
        lr_mats.append(X_cur)
        edge_counts.append(X_cur.shape[0])

        print(
            f"Collected LR coexpression from slice "
            f"{slice_index + 1}/{len(spidernet_data_list)}: {X_cur.shape}"
        )

    # Concatenate all slices at edge level.
    if any(sp.issparse(X) for X in lr_mats):
        lr_all = sp.vstack(lr_mats, format="csr")
    else:
        lr_all = np.vstack(lr_mats)

    print("Fitting global NMF-LR on concatenated edge-level LR coexpression matrix:", lr_all.shape)
    print("NMF-LR rank:", n_components)

    nmf_LR = NMF(
        n_components=n_components,
        init="nndsvda",
        random_state=random_state,
        max_iter=max_iter,
    )

    factor_all = nmf_LR.fit_transform(lr_all)

    # Global max-normalization.
    # This is done after fitting one shared NMF model on all slices.
    # It avoids per-slice normalization and keeps cross-slice comparability.
    factor_all_colmax = np.max(factor_all, axis=0)
    factor_all_colmax[factor_all_colmax == 0] = 1.0
    factor_all = factor_all / factor_all_colmax
    factor_all = factor_all.astype(np.float32, copy=False)

    # Split global edge factors back to slices.
    Factor_LR_list_local = []
    start = 0

    for slice_index, n_edges in enumerate(edge_counts):
        end = start + n_edges
        factor_cur = factor_all[start:end].copy()

        if factor_cur.shape[0] != n_edges:
            raise ValueError(
                f"Unexpected split size for slice {slice_index}: "
                f"{factor_cur.shape[0]} vs expected {n_edges}."
            )

        Factor_LR_list_local.append(factor_cur)
        print(f"Split global NMF-LR factors for slice {slice_index + 1}: {factor_cur.shape}")

        start = end

    if start != factor_all.shape[0]:
        raise ValueError(
            f"Split mismatch: used {start} rows, but factor_all has {factor_all.shape[0]} rows."
        )

    return Factor_LR_list_local


# --------------------------------------------------------------
# Always recompute global NMF-LR in memory.
# Do not load cache.
# Do not save cache.
# --------------------------------------------------------------
print("Recomputing global NMF-LR from current processed.spidernet_data.")
print("No existing Factor_LR_list.pkl will be loaded.")
print("No new Factor_LR_list.pkl will be saved.")

Factor_LR_list = build_global_nmflr_factors(
    processed.spidernet_data,
    n_components=NMF_LR_N_COMPONENTS,
    random_state=NMF_LR_RANDOM_STATE,
    max_iter=NMF_LR_MAX_ITER,
)


# --------------------------------------------------------------
# Aggregate NMF-LR edge factors to sender/receiver cell-level features.
# --------------------------------------------------------------
NMF_LR_SR_agg_all = []

for slice_index, Factor_LR_cur in enumerate(Factor_LR_list):
    edge_index_cur = processed.spidernet_data[slice_index]["edge_index"]
    num_cell_cur = processed.spidernet_data[slice_index].x.shape[0]

    Factor_LR_cur_t = torch.as_tensor(Factor_LR_cur, dtype=torch.float32, device=device)

    if not torch.is_tensor(edge_index_cur):
        edge_index_cur = torch.as_tensor(edge_index_cur)

    edge_index_cur = edge_index_cur.to(torch.int64).to(device)

    nmflr_receiver_agg_cur = scatter_nanmean(
        Factor_LR_cur_t,
        edge_index_cur[:, 1],
        dim=0,
        dim_size=num_cell_cur,
    ).to("cpu").numpy()

    nmflr_sender_agg_cur = scatter_nanmean(
        Factor_LR_cur_t,
        edge_index_cur[:, 0],
        dim=0,
        dim_size=num_cell_cur,
    ).to("cpu").numpy()

    nmflr_sr_agg_cur = np.hstack([nmflr_sender_agg_cur, nmflr_receiver_agg_cur])
    NMF_LR_SR_agg_all.append(nmflr_sr_agg_cur)

    print(
        f"Aggregated NMF-LR sender/receiver features for slice {slice_index + 1}: "
        f"{nmflr_sr_agg_cur.shape}"
    )

NMF_LR_SR_agg_all = np.vstack(NMF_LR_SR_agg_all)
NMF_LR_SR_agg_all = np.nan_to_num(NMF_LR_SR_agg_all, nan=0.0, posinf=0.0, neginf=0.0)

print("NMF_LR_SR_agg_all shape:", NMF_LR_SR_agg_all.shape)

X_dict["NMF-LR"] = NMF_LR_SR_agg_all

print("Age-prediction methods now in X_dict:", list(X_dict.keys()))

In [ ]:
## Baseline 5: scCChain edge-program scores
# --------------------------------------------------------------
# Reference: CCC_Coupling_benchmark.
#   - Load ScCChain per-edge communication-program scores.
#   - Align ScCChain sender/receiver indices to the SpiderNet edge_index.
#   - Aggregate edge-level scores to sender and receiver cell-level features.
#   - Add the combined sender+receiver features to X_dict for age prediction.
#
# Output:
#   X_dict["ScCChain"] = concatenated cell-level scCChain features.

from pathlib import Path
import os

import numpy as np
import pandas as pd
import scanpy as sc
import torch

if "X_dict" not in globals():
    raise NameError("X_dict is not defined. Please run the SpiderNet baseline / X_dict setup cell first.")

if "processed" not in globals():
    raise NameError("processed is not defined. Please run the processed-data loading cell first.")

if "scatter_nanmean" not in globals():
    raise NameError("scatter_nanmean is not defined. Please import it before running this cell.")

ScCChain_path_main = Path(OUTPUT_ROOT) / "ScCChain"
sccchain_h5ad_list_path = ScCChain_path_main / "h5ad_files.txt"

if not input_path(sccchain_h5ad_list_path).exists():
    raise FileNotFoundError(
        f"Cannot find ScCChain h5ad file list: {sccchain_h5ad_list_path}"
    )


def _resolve_sccchain_h5ad_path(path_value, base_dir):
    path_value = str(path_value).strip()
    p = Path(path_value)
    if input_path(p).exists():
        return p
    p2 = Path(base_dir) / path_value
    if input_path(p2).exists():
        return p2
    raise FileNotFoundError(f"Cannot resolve ScCChain h5ad path: {path_value}")


def _edge_index_to_numpy(edge_index):
    if torch.is_tensor(edge_index):
        edge_np = edge_index.detach().cpu().numpy()
    else:
        edge_np = np.asarray(edge_index)

    if edge_np.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_np.shape}")

    # Support both [E, 2] and PyG-style [2, E].
    if edge_np.shape[0] == 2 and edge_np.shape[1] != 2:
        edge_np = edge_np.T

    if edge_np.shape[1] != 2:
        raise ValueError(f"edge_index must have two columns after conversion, got shape {edge_np.shape}")

    return edge_np.astype(np.int64, copy=False)


def _align_sccchain_scores_to_spidernet_edges(score_df, edge_np, n_cells, program_cols):
    """
    Memory-safe alignment of ScCChain score table to SpiderNet edges.

    ScCChain score_df uses 1-based sender_index / receiver_index.
    SpiderNet edge_index uses 0-based indices.
    """
    sender_scc = score_df["sender_index"].to_numpy(dtype=np.int64) - 1
    receiver_scc = score_df["receiver_index"].to_numpy(dtype=np.int64) - 1

    valid_scc = (
        (sender_scc >= 0)
        & (sender_scc < n_cells)
        & (receiver_scc >= 0)
        & (receiver_scc < n_cells)
    )

    sender_scc = sender_scc[valid_scc]
    receiver_scc = receiver_scc[valid_scc]
    score_values = score_df.loc[valid_scc, program_cols].to_numpy(dtype=np.float32, copy=False)
    score_values = np.nan_to_num(score_values, nan=0.0, posinf=0.0, neginf=0.0)

    scc_keys = sender_scc * np.int64(n_cells) + receiver_scc
    order = np.argsort(scc_keys)
    scc_keys_sorted = scc_keys[order]
    score_values_sorted = score_values[order]

    rows = edge_np[:, 0].astype(np.int64, copy=False)
    cols = edge_np[:, 1].astype(np.int64, copy=False)
    edge_keys = rows * np.int64(n_cells) + cols

    pos = np.searchsorted(scc_keys_sorted, edge_keys)
    matched = (pos < scc_keys_sorted.shape[0])
    matched[matched] = scc_keys_sorted[pos[matched]] == edge_keys[matched]

    out = np.zeros((edge_np.shape[0], len(program_cols)), dtype=np.float32)
    if np.any(matched):
        out[matched, :] = score_values_sorted[pos[matched], :]

    match_rate = float(np.mean(matched)) if edge_np.shape[0] > 0 else np.nan
    return out, match_rate


def load_sccchain_outputs_for_age_prediction(adata_list, spidernet_data_list, sccchain_dir):
    """
    Load ScCChain outputs and return a list of edge-level score matrices
    aligned to the SpiderNet edge order for each slice.
    """
    h5ad_paths_raw = pd.read_csv(input_path(Path(sccchain_dir) / "h5ad_files.txt"), header=None)[0].tolist()
    h5ad_paths = [_resolve_sccchain_h5ad_path(p, sccchain_dir) for p in h5ad_paths_raw]

    # Match files to processed slices using n_obs, following CCC_Coupling_benchmark.
    n_obs_ref = [adata_sub.n_obs for adata_sub in adata_list]
    n_obs_scc = []
    for p in h5ad_paths:
        ad = sc.read_h5ad(input_path(p), backed="r")
        n_obs_scc.append(ad.n_obs)
        ad.file.close()

    index_map = []
    for n in n_obs_ref:
        matched = np.where(np.asarray(n_obs_scc) == n)[0]
        if len(matched) == 0:
            raise ValueError(f"Cannot match ScCChain result by n_obs={n}")
        index_map.append(int(matched[0]))

    ScCChain_CPscore_list = []
    ScCChain_program_cols = None
    match_rate_rows = []

    for slice_index in range(len(adata_list)):
        print(f"Loading ScCChain for slice {slice_index + 1}/{len(adata_list)}")

        matched_idx = index_map[slice_index]
        src_path = h5ad_paths[matched_idx]
        stem = src_path.stem

        score_path_candidates = [
            Path(sccchain_dir) / f"{stem}_ScCChain_edge_program_scores.csv",
            src_path.parent / f"{stem}_ScCChain_edge_program_scores.csv",
        ]
        score_path = next((p for p in score_path_candidates if input_path(p).exists()), None)
        if score_path is None:
            raise FileNotFoundError(
                "Cannot find ScCChain edge-program score CSV. Checked:\n"
                + "\n".join(str(p) for p in score_path_candidates)
            )

        score_df = pd.read_csv(input_path(score_path), index_col=None)
        required_cols = {"sender_index", "receiver_index"}
        missing_cols = required_cols.difference(score_df.columns)
        if len(missing_cols) > 0:
            raise KeyError(f"ScCChain score file is missing columns: {sorted(missing_cols)}")

        program_cols = score_df.columns.tolist()[2:]
        if len(program_cols) == 0:
            raise ValueError(f"No ScCChain program score columns found in {score_path}")

        if ScCChain_program_cols is None:
            ScCChain_program_cols = program_cols
        elif program_cols != ScCChain_program_cols:
            raise ValueError(
                "ScCChain program columns differ across slices. "
                "Please harmonize program columns before comparison."
            )

        edge_np = _edge_index_to_numpy(spidernet_data_list[slice_index]["edge_index"])
        n_cells = int(adata_list[slice_index].n_obs)

        edge_scores, match_rate = _align_sccchain_scores_to_spidernet_edges(
            score_df=score_df,
            edge_np=edge_np,
            n_cells=n_cells,
            program_cols=program_cols,
        )

        ScCChain_CPscore_list.append(edge_scores)
        match_rate_rows.append({
            "slice_index": slice_index,
            "matched_h5ad": str(src_path),
            "score_path": str(score_path),
            "n_cells": n_cells,
            "n_spidernet_edges": int(edge_np.shape[0]),
            "n_sccchain_score_rows": int(score_df.shape[0]),
            "match_rate_to_spidernet_edges": match_rate,
            "n_programs": int(len(program_cols)),
        })

        print(
            f"  score shape={edge_scores.shape}, "
            f"matched SpiderNet edge rate={match_rate:.3f}"
        )

    match_rate_df = pd.DataFrame(match_rate_rows)
    match_rate_path = Path(run_dirs["run_dir"]) / "ScCChain_edge_alignment_summary_for_age_prediction.csv"
    match_rate_df.to_csv(output_path(match_rate_path), index=False)
    print(f"Saved ScCChain edge-alignment summary to: {match_rate_path}")

    return ScCChain_CPscore_list, ScCChain_program_cols


ScCChain_CPscore_list, ScCChain_program_cols = load_sccchain_outputs_for_age_prediction(
    processed.adata_list,
    processed.spidernet_data,
    ScCChain_path_main,
)

# Aggregate ScCChain edge-program scores to sender/receiver cell-level features.
ScCChain_SR_agg_all = []

for slice_index, ScCChain_CPscore_cur in enumerate(ScCChain_CPscore_list):
    edge_index_cur = processed.spidernet_data[slice_index]["edge_index"]
    edge_index_cur = torch.as_tensor(_edge_index_to_numpy(edge_index_cur), dtype=torch.int64, device=device)
    num_cell_cur = int(processed.spidernet_data[slice_index].x.shape[0])

    score_t = torch.as_tensor(ScCChain_CPscore_cur, dtype=torch.float32, device=device)

    scc_sender_agg_cur = scatter_nanmean(
        score_t,
        edge_index_cur[:, 0],
        dim=0,
        dim_size=num_cell_cur,
    ).to("cpu").numpy()

    scc_receiver_agg_cur = scatter_nanmean(
        score_t,
        edge_index_cur[:, 1],
        dim=0,
        dim_size=num_cell_cur,
    ).to("cpu").numpy()

    scc_sr_agg_cur = np.hstack([scc_sender_agg_cur, scc_receiver_agg_cur])
    ScCChain_SR_agg_all.append(scc_sr_agg_cur)

    print(
        f"Aggregated ScCChain sender/receiver features for slice {slice_index + 1}: "
        f"{scc_sr_agg_cur.shape}"
    )

ScCChain_SR_agg_all = np.vstack(ScCChain_SR_agg_all)
ScCChain_SR_agg_all = np.nan_to_num(ScCChain_SR_agg_all, nan=0.0, posinf=0.0, neginf=0.0)

print("ScCChain_SR_agg_all shape:", ScCChain_SR_agg_all.shape)
print("Number of ScCChain programs:", len(ScCChain_program_cols))
print("First few ScCChain programs:", ScCChain_program_cols[:10])

X_dict["ScCChain"] = ScCChain_SR_agg_all

print("Age-prediction methods now in X_dict:", list(X_dict.keys()))


In [ ]:
from collections import defaultdict
import os

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression


# =========================================================
# Plot / style settings
# =========================================================
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,
})
sns.set_theme(style="white", context="paper")


# =========================================================
# Cross-validation and plot settings
# =========================================================
NUM_CV_SPLITS = 10

HIST2D_BINS = 20
HIST2D_CMIN = 0
USE_SQRT_DENSITY = True

REG_LINE_LS = "-"
REG_LINE_LW = 1.0
REG_LINE_COLOR = "#00F7FF"


# =========================================================
# Helper functions
# =========================================================
def make_age_balanced_splits(age_array, num_splits, random_seed=None):
    """
    Create age-balanced cross-validation splits by distributing cells
    from each age group as evenly as possible across folds.

    Parameters
    ----------
    age_array : np.ndarray
        One-dimensional array of age labels.
    num_splits : int
        Number of cross-validation folds.
    random_seed : int or None
        Random seed for reproducibility.

    Returns
    -------
    list of np.ndarray
        List of fold index arrays, each containing local indices.
    """
    rng = np.random.default_rng(random_seed)
    age_groups = defaultdict(list)

    for idx, age in enumerate(age_array):
        age_groups[age].append(idx)

    for age in age_groups:
        age_groups[age] = rng.permutation(age_groups[age])

    splits = [[] for _ in range(num_splits)]
    for indices in age_groups.values():
        grouped_splits = np.array_split(indices, num_splits)
        for split_idx, split_indices in enumerate(grouped_splits):
            splits[split_idx].extend(split_indices.tolist())

    splits = [rng.permutation(split).astype(int) for split in splits]
    return splits


def run_linear_regression_cv(X, y, num_splits):
    """
    Run age-balanced cross-validated linear regression.

    Parameters
    ----------
    X : np.ndarray
        Feature matrix with shape (n_cells, n_features).
    y : np.ndarray
        Target vector with shape (n_cells,).
    num_splits : int
        Number of cross-validation folds.

    Returns
    -------
    y_pred_all : np.ndarray
        Cross-validated predictions for all samples, shape (n_cells,).
    mean_pred_by_timepoint : pd.DataFrame
        Mean predicted age for each true age in each fold.
    """
    unique_timepoints = np.unique(y)
    cv_splits = make_age_balanced_splits(y, num_splits=num_splits)

    y_pred_all = np.zeros(len(y), dtype=float)
    mean_pred_by_timepoint = pd.DataFrame(
        np.nan,
        index=unique_timepoints,
        columns=[f"Split_{i}" for i in range(num_splits)]
    )

    for split_idx in range(num_splits):
        test_idx = cv_splits[split_idx]
        train_idx = np.concatenate([
            cv_splits[i] for i in range(num_splits) if i != split_idx
        ])

        X_train, y_train = X[train_idx], y[train_idx]
        X_test, y_test = X[test_idx], y[test_idx]

        model_linear = LinearRegression()
        model_linear.fit(X_train, y_train)
        y_pred_test = model_linear.predict(X_test)

        y_pred_all[test_idx] = y_pred_test

        for timepoint in unique_timepoints:
            mask_tp = (y_test == timepoint)
            if np.any(mask_tp):
                mean_pred_by_timepoint.loc[timepoint, f"Split_{split_idx}"] = np.mean(y_pred_test[mask_tp])

    return y_pred_all, mean_pred_by_timepoint


def compute_metrics(y_true, y_pred):
    """
    Compute Pearson correlation and mean squared error.

    Parameters
    ----------
    y_true : np.ndarray
        True values.
    y_pred : np.ndarray
        Predicted values.

    Returns
    -------
    pearson_corr : float
        Pearson correlation coefficient.
    mse : float
        Mean squared error.
    """
    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        pearson_corr = np.nan
    else:
        pearson_corr, _ = stats.pearsonr(y_true, y_pred)

    mse = np.mean((y_pred - y_true) ** 2)
    return pearson_corr, mse


def plot_hist2d_true_vs_pred(
    y_true,
    y_pred,
    celltype_name,
    method_name,
    pearson_corr,
    out_prefix,
    bins=20,
    cmin=0,
    use_sqrt_density=True,
    reg_line_ls="-",
    reg_line_lw=1.0,
    reg_line_color="#00F7FF",
):
    """
    Plot a 2D histogram of true age vs predicted age and save to disk.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid_mask = (~np.isnan(y_true)) & (~np.isnan(y_pred))
    y_true = y_true[valid_mask]
    y_pred = y_pred[valid_mask]

    if len(y_true) == 0:
        return

    xlo, xhi = float(y_true.min()), float(y_true.max())
    ylo, yhi = float(y_pred.min()), float(y_pred.max())

    H, xedges, yedges = np.histogram2d(
        y_true,
        y_pred,
        bins=bins,
        range=[[xlo, xhi], [ylo, yhi]]
    )

    if cmin is not None and cmin > 0:
        H_plot = H.copy()
        H_plot[H_plot < cmin] = np.nan
    else:
        H_plot = H

    H_color = np.sqrt(H_plot) if use_sqrt_density else H_plot

    fig, ax = plt.subplots(figsize=(3.2, 3.2))
    ax.pcolormesh(xedges, yedges, H_color.T, shading="auto")

    if len(y_true) >= 2 and np.std(y_true) > 0:
        slope, intercept, _, _, _ = stats.linregress(y_true, y_pred)
        xx = np.array([xlo, xhi], dtype=float)
        yy = intercept + slope * xx
        ax.plot(
            xx,
            yy,
            linestyle=reg_line_ls,
            linewidth=reg_line_lw,
            color=reg_line_color,
            zorder=3,
        )

    ax.set_xlim(xlo, xhi)
    ax.set_ylim(ylo, yhi)
    ax.set_xlabel("True age")
    ax.set_ylabel("Predicted age")

    corr_text = "r = NA" if np.isnan(pearson_corr) else f"r = {pearson_corr:.2f}"
    ax.set_title(f"{celltype_name} ({method_name})\n{corr_text}", pad=6)

    sns.despine(ax=ax)
    plt.tight_layout()

    plt.savefig(output_path(out_prefix + ".pdf"), bbox_inches="tight")
    plt.savefig(output_path(out_prefix + ".png"), dpi=300, bbox_inches="tight")
    plt.close(fig)


# =========================================================
# Main analysis
# =========================================================
pearson_corr_dict = {}
mse_dict = {}
agepred_SpiderNet_dict = {}
mean_age_pre_dict = {}

for method_name, X_mat in X_dict.items():
    print(f"\nProcessing method: {method_name}")
    pearson_corr_dict[method_name] = {}
    mse_dict[method_name] = {}

    for celltype_cur in celltype_all_unique:
        print(f"  Cell type: {celltype_cur}")

        cell_indices = np.where(celltype_all == celltype_cur)[0]
        n_cells = len(cell_indices)

        if n_cells < NUM_CV_SPLITS:
            print(f"  Skip {celltype_cur}: too few cells ({n_cells})")
            continue

        X = X_mat[cell_indices, :]
        y = np.asarray(cellage_all[cell_indices]).ravel()
        barcodes_cur = np.asarray(processed.adata_all.obs.index)[cell_indices]

        y_pred_all, mean_age_pre = run_linear_regression_cv(
            X=X,
            y=y,
            num_splits=NUM_CV_SPLITS,
        )

        pearson_corr, mse = compute_metrics(y_true=y, y_pred=y_pred_all)
        pearson_corr_dict[method_name][celltype_cur] = pearson_corr
        mse_dict[method_name][celltype_cur] = mse

        if method_name == "SpiderNet":
            agepred_SpiderNet_dict[celltype_cur] = pd.DataFrame(
                {"predicted_age": y_pred_all},
                index=barcodes_cur
            )
            mean_age_pre_dict[celltype_cur] = mean_age_pre

            out_prefix_hist2d = os.path.join(
                run_dirs['run_dir'],
                f"AgePred_Hist2D_{method_name}_{celltype_cur}_sqrtDensity_regLine_noColorbar"
            )

            pd.DataFrame({"TrueAge": y, "PredictedAge": y_pred_all}).to_csv(
                output_path(out_prefix_hist2d + "_predictions.csv"), index=False)

            plot_hist2d_true_vs_pred(
                y_true=y,
                y_pred=y_pred_all,
                celltype_name=celltype_cur,
                method_name=method_name,
                pearson_corr=pearson_corr,
                out_prefix=out_prefix_hist2d,
                bins=HIST2D_BINS,
                cmin=HIST2D_CMIN,
                use_sqrt_density=USE_SQRT_DENSITY,
                reg_line_ls=REG_LINE_LS,
                reg_line_lw=REG_LINE_LW,
                reg_line_color=REG_LINE_COLOR,
            )

print("\nCross-validation finished.")


# =========================================================
# Summarize results
# =========================================================
pearson_corr_dict_df = pd.DataFrame(pearson_corr_dict)
mse_dict_df = pd.DataFrame(mse_dict)

print("\nPearson correlation:")
print(pearson_corr_dict_df)

### Assemble and display the coronal-atlas age-prediction figure

This cell assembles the 18 SpiderNet true-age versus predicted-age panels from B1 into `Ageprediction_agingmousebrain.pdf`, with six columns and three rows in the manuscript cell-type order. It reads the existing per-cell-type PDF and PNG files from `run_dirs["run_dir"]`; B1 must have exported all panels from the same run. No regression or cross-validation is repeated. The PDF retains the original vector panels, including their titles, correlations, axes, density colors, and fitted lines. Individual panel labels are retained, so the layout is not an exact copy of the manually arranged manuscript figure with shared axis labels.

The assembled PDF and a compact PNG preview are saved in the current run directory, and the preview is displayed below. The manuscript copy in `overleaf_git/img` is not overwritten. This assembly-only step requires `pypdf` (install with `%pip install pypdf` in the notebook environment if needed) and Pillow, which is also a Matplotlib dependency. It can be rerun using existing panels after setting `run_dirs["run_dir"]`.


In [ ]:
from pathlib import Path

from IPython.display import Image, display
from PIL import Image as PILImage

try:
    from pypdf import PdfReader, PdfWriter, Transformation
except ImportError as exc:
    raise ImportError(
        "Vector figure assembly requires pypdf. Run %pip install pypdf "
        "in this notebook environment, then rerun this cell."
    ) from exc


# Row-major order of the 18 cell types in the coronal-atlas figure.
AGEPRED_FIGURE_CELL_TYPES = [
    "Astrocyte", "B cell", "Endothelial", "Ependymal", "Macrophage", "Microglia",
    "Neuroblast", "Neuron-Excitatory", "Neuron-Inhibitory", "Neuron-MSN", "Neutrophil", "NSC",
    "Oligodendrocyte", "OPC", "Pericyte", "T cell", "VLMC", "VSMC",
]


def assemble_ageprediction_panels(run_dir, cell_types, ncols=6):
    """Arrange existing B1 panels without recomputing predictions or rasterizing the PDF."""
    run_dir = Path(run_dir)
    if not cell_types or ncols < 1:
        raise ValueError("Provide at least one cell type and a positive column count.")

    panel_stems = [
        run_dir / f"AgePred_Hist2D_SpiderNet_{name}_sqrtDensity_regLine_noColorbar"
        for name in cell_types
    ]
    missing = [
        str(stem.with_suffix(ext))
        for stem in panel_stems for ext in (".pdf", ".png")
        if not input_path(stem.with_suffix(ext)).is_file()
    ]
    if missing:
        raise FileNotFoundError(
            "Run the B1 cross-validation and per-cell-type plotting cell first. "
            "All 18 PDF/PNG panel pairs must come from the same run. Missing files:\n"
            + "\n".join(missing)
        )

    readers = [PdfReader(input_path(str(stem.with_suffix(".pdf")))) for stem in panel_stems]
    if any(len(reader.pages) != 1 for reader in readers):
        raise ValueError("Each age-prediction panel must contain exactly one PDF page.")
    pages = [reader.pages[0] for reader in readers]
    for page in pages:
        if page.rotation:
            page.transfer_rotation_to_content()

    # Keep all panels at their original physical size, centered in equal-sized slots.
    slot_width = max(float(page.cropbox.width) for page in pages)
    slot_height = max(float(page.cropbox.height) for page in pages)
    gap = 10.0  # PDF points between panels.
    margin = 10.0
    nrows = (len(pages) + ncols - 1) // ncols
    page_width = 2 * margin + ncols * slot_width + (ncols - 1) * gap
    page_height = 2 * margin + nrows * slot_height + (nrows - 1) * gap

    writer = PdfWriter()
    combined_page = writer.add_blank_page(width=page_width, height=page_height)
    preview_scale = 2400 / page_width
    preview = PILImage.new(
        "RGB", (round(page_width * preview_scale), round(page_height * preview_scale)), "white"
    )

    for index, (page, stem) in enumerate(zip(pages, panel_stems)):
        row, col = divmod(index, ncols)
        width, height = float(page.cropbox.width), float(page.cropbox.height)
        x = margin + col * (slot_width + gap) + (slot_width - width) / 2
        top = margin + row * (slot_height + gap) + (slot_height - height) / 2
        y = page_height - top - height
        transform = Transformation().translate(
            tx=x - float(page.cropbox.left), ty=y - float(page.cropbox.bottom)
        )
        combined_page.merge_transformed_page(page, transform, expand=False)

        # The matching PNG is used only for the inline preview; the PDF stays vector.
        with PILImage.open(input_path(stem.with_suffix(".png"))) as panel_image:
            panel_preview = panel_image.convert("RGB").resize(
                (max(1, round(width * preview_scale)), max(1, round(height * preview_scale))),
                resample=PILImage.Resampling.LANCZOS,
            )
        preview.paste(panel_preview, (round(x * preview_scale), round(top * preview_scale)))

    pdf_path = run_dir / "Ageprediction_agingmousebrain.pdf"
    preview_path = run_dir / "Ageprediction_agingmousebrain_preview.png"
    writer.add_metadata({"/Title": "Cell-type-specific age prediction from SpiderNet MI profiles in the MERFISH coronal brain atlas"})
    with output_path(pdf_path).open("wb") as stream:
        writer.write(stream)
    preview.save(output_path(preview_path))
    return pdf_path, preview_path


ageprediction_pdf_path, ageprediction_preview_path = assemble_ageprediction_panels(
    run_dirs["run_dir"], AGEPRED_FIGURE_CELL_TYPES
)
print(f"Saved assembled age-prediction figure: {ageprediction_pdf_path}")
display(Image(filename=str(input_path(ageprediction_preview_path)), width=1400))


### Age-prediction Pearson-r comparison plot

This plot excludes the exploratory PCA baseline and compares SpiderNet with NMF-LR, COMMOT, ScCChain, and Banksy.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.patches import Patch

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["mathtext.fontset"] = "dejavuserif"
plt.rcParams["font.family"] = "arial"

df = pearson_corr_dict_df.copy()

# Do not show PCA results. Add ScCChain to the displayed method set.
# Display/order requirement:
#   SpiderNet, NMF-LR, COMMOT, ScCChain, Banksy
# methods = ["SpiderNet", "NMF-LR", "COMMOT", "ScCChain", "Banksy"]
methods = ["SpiderNet", "NMF-LR", "COMMOT", "ScCChain", "Banksy"][::-1]
baseline_cols = ["NMF-LR", "COMMOT", "ScCChain", "Banksy"]
required_cols = methods

missing = [c for c in required_cols if c not in df.columns]
if len(missing) > 0:
    raise ValueError(f"pearson_corr_dict_df missing columns: {missing}")

# ====== 1. Compute the improvement ratio for each cell type ======
def safe_div(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return a / np.where(b == 0, np.nan, b)

improve_stack = np.vstack([
    safe_div(df["SpiderNet"], df[baseline])
    for baseline in baseline_cols
])

# Minimum improvement ratio of SpiderNet relative to shown baselines
df["ImproveRatio"] = np.nanmin(improve_stack, axis=0)
df["ImproveRatio"] = df["ImproveRatio"].replace([np.inf, -np.inf], np.nan)
df["ImproveRatio"] = df["ImproveRatio"].fillna(0)

# ====== 2. Sort cell types ======
df_sorted = df.sort_values(by="ImproveRatio", ascending=False)

# ====== 3. Convert to long format ======
df_long = df_sorted.reset_index().melt(
    id_vars="index",
    value_vars=methods,
    var_name="Method",
    value_name="Pearson_r",
)

df_long.rename(columns={"index": "CellType"}, inplace=True)
df_long["Method"] = pd.Categorical(df_long["Method"], categories=methods, ordered=True)

# Save results without PCA and with ScCChain
df_sorted[methods + ["ImproveRatio"]].to_csv(
    output_path(str(run_dirs["run_dir"]) + "/Celltype_age_prediction_pearsonr_comparison_wide.csv")
)

df_long.to_csv(
    output_path(str(run_dirs["run_dir"]) + "/Celltype_age_prediction_pearsonr_comparison_long.csv"),
    index=False,
)

# ====== 4. Horizontal bar plot with grouped spacing and reversed display order ======
plt.figure(figsize=(5.45 * 1.05, 4.48 * 0.9), facecolor="white")
ax = plt.gca()
ax.set_facecolor("white")

# Match CCC_Coupling_benchmark fill/edge color scheme.
# Banksy is a new method here and uses the user-specified fill/edge colors.
method_colors = {
    "SpiderNet": {"edge": "#9F3B38", "fill": "#E1B6A7"},
    "NMF-LR": {"edge": "#82CCE2", "fill": "#D4ECF1"},
    "COMMOT": {"edge": "#519384", "fill": "#B9CEC7"},
    "ScCChain": {"edge": "#636491", "fill": "#A6A2B9"},
    "Banksy": {"edge": "#8A5A44", "fill": "#D9C1B0"},
}

# method_colors = {
#     "SpiderNet": {"edge": "#9F3B38", "fill": "#9F3B38"},
#     "NMF-LR": {"edge": "#82CCE2", "fill": "#82CCE2"},
#     "COMMOT": {"edge": "#519384", "fill": "#519384"},
#     "ScCChain": {"edge": "#636491", "fill": "#636491"},
#     "Banksy": {"edge": "#8A5A44", "fill": "#8A5A44"},
# }

# method_colors = {
#     "SpiderNet": {"edge": "#E1B6A7", "fill": "#E1B6A7"},
#     "NMF-LR": {"edge": "#D4ECF1", "fill": "#D4ECF1"},
#     "COMMOT": {"edge": "#B9CEC7", "fill": "#B9CEC7"},
#     "ScCChain": {"edge": "#A6A2B9", "fill": "#A6A2B9"},
#     "Banksy": {"edge": "#D9C1B0", "fill": "#D9C1B0"},
# }

# Reverse the cell-type order for plotting
celltypes = df_sorted.reset_index()["index"].tolist()[::-1]

mat = (
    df_sorted.reset_index()
    .set_index("index")[methods]
    .reindex(celltypes)
)

y = np.arange(len(celltypes))

n_hue = len(methods)
group_height = 0.82
inner_gap = 0.05

bar_h = (group_height - inner_gap * (n_hue - 1)) / n_hue
offsets = (
    (-group_height / 2)
    + (np.arange(n_hue) + 0.5) * bar_h
    + np.arange(n_hue) * inner_gap
)

for j, m in enumerate(methods):
    ax.barh(
        y + offsets[j],
        mat[m].values,
        height=bar_h,
        facecolor=method_colors[m]["fill"],
        edgecolor=method_colors[m]["edge"],
        linewidth=0.7,
        label=m,
    )

ax.set_yticks(y)
ax.set_yticklabels(celltypes)

sns.despine(ax=ax, top=True, right=True)

ax.spines["left"].set_visible(True)
ax.spines["bottom"].set_visible(True)
ax.spines["left"].set_color("black")
ax.spines["bottom"].set_color("black")
ax.spines["left"].set_linewidth(1.0)
ax.spines["bottom"].set_linewidth(1.0)

ax.set_xlim(left=0)

ax.set_ylabel("Cell type")
ax.set_xlabel("Pearson correlation")

legend_handles = [
    Patch(
        facecolor=method_colors[m]["fill"],
        edgecolor=method_colors[m]["edge"],
        linewidth=1.0,
        label=m,
    )
    for m in methods
]

ax.legend(
    handles=legend_handles,
    title="Method",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)

plt.tight_layout()

plt.savefig(
    output_path(str(run_dirs["run_dir"]) + "/Celltype_age_prediction_pearsonr_comparison.pdf"),
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close()

## Leave-one-section-out age-prediction benchmark

This analysis is separate from the age-stratified 10-fold analysis above. For each cell type, one complete coronal section is held out, a linear age-prediction model is trained using that cell type's cells from the other nine sections, and predictions are generated for that cell type in the held-out section.


In [ ]:
# ============================================================
# LOSO setup and helper function
# ============================================================
from pathlib import Path

LOSO_METHODS = [
    method for method in ["SpiderNet", "NMF-LR", "COMMOT", "ScCChain", "Banksy"]
    if method in X_dict
]
LOSO_EXPECTED_N_SECTIONS = 10
loso_outdir = Path(run_dirs["run_dir"]) / "AgePrediction_LOSO"
loso_outdir.mkdir(parents=True, exist_ok=True)

# Reconstruct one section label per cell in exactly the same slice order used
# to concatenate every feature matrix in X_dict. Include the positional index
# so labels remain unique even if external section names are duplicated.
section_label_parts = []
section_summary_rows = []
batch_names = getattr(processed, "batch_cell_unique", None)

for section_index, adata_section in enumerate(processed.adata_list):
    if batch_names is not None and section_index < len(batch_names):
        section_name = str(batch_names[section_index])
    elif "sample_name" in adata_section.obs.columns:
        section_name = str(adata_section.obs["sample_name"].iloc[0])
    else:
        section_name = f"section_{section_index + 1}"

    section_label = f"Section_{section_index + 1}:{section_name}"
    section_label_parts.append(
        np.repeat(section_label, adata_section.n_obs)
    )
    section_age = pd.to_numeric(
        pd.Series(adata_section.obs["age"]), errors="coerce"
    ).dropna().unique()
    section_summary_rows.append({
        "section_index": int(section_index),
        "section_label": section_label,
        "section_name": section_name,
        "age": float(section_age[0]) if len(section_age) == 1 else np.nan,
        "n_cells": int(adata_section.n_obs),
    })

section_all_loso = np.concatenate(section_label_parts)
section_order_loso = [row["section_label"] for row in section_summary_rows]
section_summary_loso = pd.DataFrame(section_summary_rows)

if len(section_order_loso) != LOSO_EXPECTED_N_SECTIONS:
    raise ValueError(
        f"Expected {LOSO_EXPECTED_N_SECTIONS} coronal sections, "
        f"but found {len(section_order_loso)}."
    )
if len(section_all_loso) != len(cellage_all):
    raise ValueError(
        "Section labels and cell-level age labels have different lengths: "
        f"{len(section_all_loso)} vs {len(cellage_all)}."
    )
for method_name in LOSO_METHODS:
    if X_dict[method_name].shape[0] != len(section_all_loso):
        raise ValueError(
            f"{method_name} has {X_dict[method_name].shape[0]} rows, but "
            f"{len(section_all_loso)} cells were found."
        )

section_summary_loso.to_csv(
    output_path(loso_outdir / "LOSO_section_summary.csv"), index=False
)
display(section_summary_loso)
print("LOSO methods:", LOSO_METHODS)


def run_linear_regression_loso(X, y, section_labels, section_order):
    """Predict every cell using a model that never sees its entire section."""
    X = np.asarray(X)
    y = np.asarray(y, dtype=float).ravel()
    section_labels = np.asarray(section_labels).astype(str)
    y_pred = np.full(y.shape[0], np.nan, dtype=float)
    fold_rows = []

    for held_out_section in section_order:
        test_mask = section_labels == str(held_out_section)
        train_mask = ~test_mask
        n_test = int(test_mask.sum())
        n_train = int(train_mask.sum())

        if n_test == 0:
            fold_rows.append({
                "held_out_section": held_out_section,
                "n_train_cells": n_train,
                "n_test_cells": 0,
                "status": "cell type absent from held-out section",
            })
            continue
        if n_train == 0:
            fold_rows.append({
                "held_out_section": held_out_section,
                "n_train_cells": 0,
                "n_test_cells": n_test,
                "status": "no training cells",
            })
            continue

        model_loso = LinearRegression()
        model_loso.fit(X[train_mask], y[train_mask])
        fold_prediction = model_loso.predict(X[test_mask])
        y_pred[test_mask] = fold_prediction
        fold_rows.append({
            "held_out_section": held_out_section,
            "n_train_cells": n_train,
            "n_test_cells": n_test,
            "held_out_true_age": float(np.mean(y[test_mask])),
            "mean_predicted_age": float(np.mean(fold_prediction)),
            "median_predicted_age": float(np.median(fold_prediction)),
            "fold_mse": float(np.mean((fold_prediction - y[test_mask]) ** 2)),
            "status": "ok",
        })

    return y_pred, pd.DataFrame(fold_rows)


In [ ]:
# ============================================================
# Run LOSO benchmark for every method and cell type
# ============================================================
pearson_corr_loso_dict = {}
mse_loso_dict = {}
agepred_SpiderNet_loso_dict = {}
loso_fold_summary_parts = []
loso_prediction_parts = []
all_barcodes_loso = np.concatenate([
    np.asarray(adata_section.obs_names).astype(str)
    for adata_section in processed.adata_list
])

for method_name in LOSO_METHODS:
    print(f"\nProcessing LOSO method: {method_name}")
    pearson_corr_loso_dict[method_name] = {}
    mse_loso_dict[method_name] = {}
    X_method = np.asarray(X_dict[method_name])

    for celltype_cur in celltype_all_unique:
        cell_indices = np.flatnonzero(celltype_all == celltype_cur)
        if len(cell_indices) < 2:
            print(f"  Skip {celltype_cur}: too few cells ({len(cell_indices)})")
            continue

        X_cur = X_method[cell_indices, :]
        y_cur = np.asarray(cellage_all[cell_indices], dtype=float).ravel()
        sections_cur = section_all_loso[cell_indices]
        barcodes_cur = all_barcodes_loso[cell_indices]

        y_pred_cur, fold_summary_cur = run_linear_regression_loso(
            X=X_cur,
            y=y_cur,
            section_labels=sections_cur,
            section_order=section_order_loso,
        )
        valid_prediction = np.isfinite(y_cur) & np.isfinite(y_pred_cur)
        if valid_prediction.sum() < 2:
            pearson_corr, mse = np.nan, np.nan
        else:
            pearson_corr, mse = compute_metrics(
                y_true=y_cur[valid_prediction],
                y_pred=y_pred_cur[valid_prediction],
            )

        pearson_corr_loso_dict[method_name][celltype_cur] = pearson_corr
        mse_loso_dict[method_name][celltype_cur] = mse

        fold_summary_cur.insert(0, "CellType", celltype_cur)
        fold_summary_cur.insert(0, "Method", method_name)
        loso_fold_summary_parts.append(fold_summary_cur)

        prediction_cur = pd.DataFrame({
            "Method": method_name,
            "CellType": celltype_cur,
            "Barcode": barcodes_cur,
            "Section": sections_cur,
            "TrueAge": y_cur,
            "PredictedAge": y_pred_cur,
        })
        loso_prediction_parts.append(prediction_cur)

        if method_name == "SpiderNet":
            agepred_SpiderNet_loso_dict[celltype_cur] = prediction_cur.set_index(
                "Barcode"
            )[["Section", "TrueAge", "PredictedAge"]]
            plot_hist2d_true_vs_pred(
                y_true=y_cur[valid_prediction],
                y_pred=y_pred_cur[valid_prediction],
                celltype_name=celltype_cur,
                method_name="SpiderNet LOSO",
                pearson_corr=pearson_corr,
                out_prefix=str(
                    loso_outdir
                    / f"AgePred_LOSO_Hist2D_SpiderNet_{celltype_cur}_sqrtDensity_regLine_noColorbar"
                ),
                bins=HIST2D_BINS,
                cmin=HIST2D_CMIN,
                use_sqrt_density=USE_SQRT_DENSITY,
                reg_line_ls=REG_LINE_LS,
                reg_line_lw=REG_LINE_LW,
                reg_line_color=REG_LINE_COLOR,
            )

        print(
            f"  {celltype_cur}: n={int(valid_prediction.sum())}, "
            f"Pearson r={pearson_corr:.4f}, MSE={mse:.4f}"
        )

pearson_corr_loso_df = pd.DataFrame(pearson_corr_loso_dict)
mse_loso_df = pd.DataFrame(mse_loso_dict)
loso_fold_summary_df = pd.concat(
    loso_fold_summary_parts, axis=0, ignore_index=True
)
loso_predictions_df = pd.concat(
    loso_prediction_parts, axis=0, ignore_index=True
)

pearson_corr_loso_df.to_csv(output_path(loso_outdir / "Celltype_age_prediction_LOSO_pearsonr.csv"))
mse_loso_df.to_csv(output_path(loso_outdir / "Celltype_age_prediction_LOSO_mse.csv"))
loso_fold_summary_df.to_csv(output_path(loso_outdir / "Celltype_age_prediction_LOSO_fold_summary.csv"), index=False)
loso_predictions_df.to_csv(output_path(loso_outdir / "Celltype_age_prediction_LOSO_cell_predictions.csv"), index=False)

print("\nLOSO Pearson correlation:")
display(pearson_corr_loso_df)
print("Saved LOSO outputs to:", loso_outdir)


In [ ]:
# ============================================================
# Plot the LOSO method comparison without altering the original plot
# ============================================================
from matplotlib.patches import Patch

loso_plot_methods = [
    method for method in ["SpiderNet", "NMF-LR", "COMMOT", "ScCChain", "Banksy"][::-1]
    if method in pearson_corr_loso_df.columns
]
loso_baseline_cols = [
    method for method in ["NMF-LR", "COMMOT", "ScCChain", "Banksy"]
    if method in pearson_corr_loso_df.columns
]
loso_plot_df = pearson_corr_loso_df.copy()

if "SpiderNet" not in loso_plot_df.columns:
    raise ValueError("SpiderNet is missing from pearson_corr_loso_df.")
if not loso_baseline_cols:
    raise ValueError("No LOSO baseline methods are available for comparison.")

loso_improvement_stack = np.vstack([
    safe_div(loso_plot_df["SpiderNet"], loso_plot_df[baseline])
    for baseline in loso_baseline_cols
])
loso_plot_df["ImproveRatio"] = np.nanmin(
    loso_improvement_stack, axis=0
)
loso_plot_df["ImproveRatio"] = (
    loso_plot_df["ImproveRatio"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)
loso_plot_df_sorted = loso_plot_df.sort_values(
    "ImproveRatio", ascending=False
)

loso_long_df = loso_plot_df_sorted.reset_index().melt(
    id_vars="index",
    value_vars=loso_plot_methods,
    var_name="Method",
    value_name="Pearson_r",
).rename(columns={"index": "CellType"})
loso_long_df["Method"] = pd.Categorical(
    loso_long_df["Method"], categories=loso_plot_methods, ordered=True
)

loso_plot_df_sorted[loso_plot_methods + ["ImproveRatio"]].to_csv(
    output_path(loso_outdir / "Celltype_age_prediction_LOSO_pearsonr_comparison_wide.csv")
)
loso_long_df.to_csv(
    output_path(loso_outdir / "Celltype_age_prediction_LOSO_pearsonr_comparison_long.csv"),
    index=False,
)

method_colors_loso = {
    "SpiderNet": {"edge": "#9F3B38", "fill": "#E1B6A7"},
    "NMF-LR": {"edge": "#82CCE2", "fill": "#D4ECF1"},
    "COMMOT": {"edge": "#519384", "fill": "#B9CEC7"},
    "ScCChain": {"edge": "#636491", "fill": "#A6A2B9"},
    "Banksy": {"edge": "#8A5A44", "fill": "#D9C1B0"},
}
celltypes_loso = loso_plot_df_sorted.index.tolist()[::-1]
mat_loso = loso_plot_df_sorted[loso_plot_methods].reindex(celltypes_loso)
y_loso = np.arange(len(celltypes_loso))
n_hue_loso = len(loso_plot_methods)
group_height_loso = 0.82
inner_gap_loso = 0.05
bar_h_loso = (
    group_height_loso - inner_gap_loso * (n_hue_loso - 1)
) / n_hue_loso
offsets_loso = (
    (-group_height_loso / 2)
    + (np.arange(n_hue_loso) + 0.5) * bar_h_loso
    + np.arange(n_hue_loso) * inner_gap_loso
)

fig, ax = plt.subplots(
    figsize=(5.45 * 1.05, 4.48 * 0.9), facecolor="white"
)
ax.set_facecolor("white")
for method_index, method_name in enumerate(loso_plot_methods):
    ax.barh(
        y_loso + offsets_loso[method_index],
        mat_loso[method_name].to_numpy(),
        height=bar_h_loso,
        facecolor=method_colors_loso[method_name]["fill"],
        edgecolor=method_colors_loso[method_name]["edge"],
        linewidth=0.7,
        label=method_name,
    )

ax.set_yticks(y_loso)
ax.set_yticklabels(celltypes_loso)
# LOSO correlations can be negative. Show the complete range instead of
# clipping negative bars at zero (which makes methods look missing).
finite_loso_values = mat_loso.to_numpy(dtype=float)
finite_loso_values = finite_loso_values[np.isfinite(finite_loso_values)]
if finite_loso_values.size > 0:
    loso_xmin = min(0.0, float(finite_loso_values.min()))
    loso_xmax = max(0.0, float(finite_loso_values.max()))
    loso_xrange = max(loso_xmax - loso_xmin, 1e-6)
    ax.set_xlim(
        loso_xmin - 0.04 * loso_xrange,
        loso_xmax + 0.04 * loso_xrange,
    )
ax.axvline(0, color="#58595B", linewidth=0.7, zorder=0)
ax.set_ylabel("Cell type")
ax.set_xlabel("LOSO Pearson correlation")
sns.despine(ax=ax, top=True, right=True)
ax.spines["left"].set_color("black")
ax.spines["bottom"].set_color("black")
ax.spines["left"].set_linewidth(1.0)
ax.spines["bottom"].set_linewidth(1.0)
legend_handles_loso = [
    Patch(
        facecolor=method_colors_loso[method]["fill"],
        edgecolor=method_colors_loso[method]["edge"],
        linewidth=1.0,
        label=method,
    )
    for method in loso_plot_methods
]
ax.legend(
    handles=legend_handles_loso,
    title="Method",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)
fig.tight_layout()
fig.savefig(
    output_path(loso_outdir / "Celltype_age_prediction_LOSO_pearsonr_comparison.pdf"),
    dpi=300, bbox_inches="tight",
)
fig.savefig(
    output_path(loso_outdir / "Celltype_age_prediction_LOSO_pearsonr_comparison.png"),
    dpi=300, bbox_inches="tight",
)
plt.show()
plt.close(fig)


## LOSO true-age versus predicted-age density plots

The following cells reuse the saved cell-level LOSO predictions and the existing `plot_hist2d_true_vs_pred` helper. They do not retrain any age-prediction model.

In [ ]:
# ============================================================
# Load and validate the saved cell-level LOSO predictions
# ============================================================
from pathlib import Path

loso_outdir = Path(run_dirs["run_dir"]) / "AgePrediction_LOSO"
loso_prediction_path = (
    loso_outdir / "Celltype_age_prediction_LOSO_cell_predictions.csv"
)

if not input_path(loso_prediction_path).exists():
    raise FileNotFoundError(
        f"Cannot find saved LOSO predictions: {loso_prediction_path}. "
        "Run the LOSO benchmark cell first."
    )

loso_density_columns = [
    "Method", "CellType", "TrueAge", "PredictedAge"
]
if (
    "loso_predictions_df" in globals()
    and set(loso_density_columns).issubset(loso_predictions_df.columns)
):
    loso_predictions_for_density = (
        loso_predictions_df[loso_density_columns].copy()
    )
else:
    # The saved table contains millions of rows. Read only the four
    # columns needed for these figures and use categorical labels to
    # avoid loading Barcode and Section strings into memory again.
    loso_predictions_for_density = pd.read_csv(
        input_path(loso_prediction_path),
        usecols=loso_density_columns,
        dtype={"Method": "category", "CellType": "category"},
    )
required_loso_prediction_columns = set(loso_density_columns)
missing_loso_prediction_columns = (
    required_loso_prediction_columns
    - set(loso_predictions_for_density.columns)
)
if missing_loso_prediction_columns:
    raise ValueError(
        "Saved LOSO prediction table is missing columns: "
        f"{sorted(missing_loso_prediction_columns)}"
    )

for value_column in ["TrueAge", "PredictedAge"]:
    loso_predictions_for_density[value_column] = pd.to_numeric(
        loso_predictions_for_density[value_column], errors="coerce"
    )

loso_predictions_for_density = (
    loso_predictions_for_density
    .dropna(subset=["Method", "CellType", "TrueAge", "PredictedAge"])
    .copy()
)

LOSO_DENSITY_METHOD_ORDER = [
    method
    for method in ["SpiderNet", "NMF-LR", "COMMOT", "ScCChain", "Banksy"]
    if method in loso_predictions_for_density["Method"].unique()
]
loso_density_outdir = loso_outdir / "Hist2D_TrueAge_vs_PredictedAge_AllMethods"
loso_density_outdir.mkdir(parents=True, exist_ok=True)

loso_density_group_counts = (
    loso_predictions_for_density
    .groupby(["Method", "CellType"], observed=True)
    .size()
    .rename("n_valid_cell_predictions")
    .reset_index()
)
print("Loaded LOSO predictions from:", loso_prediction_path)
print("Density plots will be saved to:", loso_density_outdir)
print("Methods:", LOSO_DENSITY_METHOD_ORDER)
display(loso_density_group_counts)


In [ ]:
# ============================================================
# Plot LOSO true age versus predicted age for every method
# and cell type using the same density-based plotting function
# as the original age-stratified 10-fold analysis.
# ============================================================
import re

if "plot_hist2d_true_vs_pred" not in globals():
    raise NameError(
        "plot_hist2d_true_vs_pred is not defined. Run the original "
        "age-prediction helper/10-fold cell first."
    )


def _safe_loso_density_filename(value):
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(value)).strip("_")


loso_density_summary_rows = []
for method_name in LOSO_DENSITY_METHOD_ORDER:
    method_predictions = loso_predictions_for_density.loc[
        loso_predictions_for_density["Method"].eq(method_name)
    ]
    method_celltypes = sorted(
        method_predictions["CellType"].astype(str).unique()
    )

    for celltype_name in method_celltypes:
        prediction_group = method_predictions.loc[
            method_predictions["CellType"].astype(str).eq(celltype_name)
        ].copy()
        y_true = prediction_group["TrueAge"].to_numpy(dtype=float)
        y_pred = prediction_group["PredictedAge"].to_numpy(dtype=float)
        valid_values = np.isfinite(y_true) & np.isfinite(y_pred)
        y_true = y_true[valid_values]
        y_pred = y_pred[valid_values]

        if len(y_true) < 2:
            loso_density_summary_rows.append({
                "Method": method_name,
                "CellType": celltype_name,
                "n_valid_cell_predictions": int(len(y_true)),
                "Pearson_r": np.nan,
                "MSE": np.nan,
                "status": "fewer than two valid predictions",
                "output_prefix": "",
            })
            continue

        pearson_corr, mse = compute_metrics(y_true=y_true, y_pred=y_pred)
        safe_method = _safe_loso_density_filename(method_name)
        safe_celltype = _safe_loso_density_filename(celltype_name)
        output_prefix = loso_density_outdir / (
            f"AgePred_LOSO_Hist2D_{safe_method}_{safe_celltype}"
            "_sqrtDensity_regLine_noColorbar"
        )

        plot_hist2d_true_vs_pred(
            y_true=y_true,
            y_pred=y_pred,
            celltype_name=celltype_name,
            method_name=f"{method_name} LOSO",
            pearson_corr=pearson_corr,
            out_prefix=str(output_prefix),
            bins=HIST2D_BINS,
            cmin=HIST2D_CMIN,
            use_sqrt_density=USE_SQRT_DENSITY,
            reg_line_ls=REG_LINE_LS,
            reg_line_lw=REG_LINE_LW,
            reg_line_color=REG_LINE_COLOR,
        )

        loso_density_summary_rows.append({
            "Method": method_name,
            "CellType": celltype_name,
            "n_valid_cell_predictions": int(len(y_true)),
            "Pearson_r": float(pearson_corr),
            "MSE": float(mse),
            "status": "saved",
            "output_prefix": str(output_prefix),
        })

loso_density_plot_summary = pd.DataFrame(loso_density_summary_rows)
loso_density_summary_path = (
    loso_density_outdir / "LOSO_Hist2D_true_vs_predicted_plot_summary.csv"
)
loso_density_plot_summary.to_csv(output_path(loso_density_summary_path), index=False)
print("Saved LOSO density plots to:", loso_density_outdir)
print("Saved plot summary to:", loso_density_summary_path)
display(loso_density_plot_summary)


## Optional B2. Inspect regression coefficients from the age-prediction models

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm


def get_mi_feature_names(n_features):
    if n_features % 2 != 0:
        raise ValueError(f"Expected an even number of features, but got {n_features}.")
    n_mi = n_features // 2
    return (
        [f"Sender_MI_{i + 1}" for i in range(n_mi)] +
        [f"Receiver_MI_{i + 1}" for i in range(n_mi)]
    )


celltype_all = np.asarray(celltype_all)
cellage_all = np.asarray(cellage_all)
cellclass_unique = np.unique(celltype_all)
feature_names = get_mi_feature_names(MI_SR_agg_all.shape[1])

# =========================================================
# 1. Fit sklearn linear regression for each cell type
# =========================================================
reg_scale_dict = {}
coef_dict_sklearn = {}

for cellclass_cur in cellclass_unique:
    print(f"Training sklearn LinearRegression: {cellclass_cur}")

    idx = np.where(celltype_all == cellclass_cur)[0]
    X = MI_SR_agg_all[idx, :]
    y = cellage_all[idx]

    model_linear = LinearRegression().fit(X, y)
    reg_scale_dict[cellclass_cur] = model_linear
    coef_dict_sklearn[cellclass_cur] = model_linear.coef_

coef_df_sklearn = pd.DataFrame(coef_dict_sklearn).T
coef_df_sklearn.columns = feature_names

# =========================================================
# 2. Fit OLS for each cell type and extract p-values
# =========================================================
ols_model_dict = {}
coef_dict_ols = {}
pval_dict = {}

for cellclass_cur in cellclass_unique:
    print(f"Training statsmodels OLS: {cellclass_cur}")

    idx = np.where(celltype_all == cellclass_cur)[0]
    X = MI_SR_agg_all[idx, :]
    y = cellage_all[idx]

    X_sm = sm.add_constant(X)
    model_linear = sm.OLS(y, X_sm).fit()

    ols_model_dict[cellclass_cur] = model_linear
    coef_dict_ols[cellclass_cur] = model_linear.params[1:]   # exclude intercept
    pval_dict[cellclass_cur] = model_linear.pvalues[1:]      # exclude intercept

coef_df = pd.DataFrame(coef_dict_ols).T
pval_df = pd.DataFrame(pval_dict).T

coef_df.columns = feature_names
pval_df.columns = feature_names
# Save the exact fitted coefficients and P values for plotting without refitting.
coef_df.to_csv(output_path(run_dirs["run_dir"] / "AgePrediction_RegressionCoefficients.csv"))
pval_df.to_csv(output_path(run_dirs["run_dir"] / "AgePrediction_RegressionPvalues.csv"))


In [ ]:
## Save the fitted age-prediction models
import pickle
with open(output_path(str(run_dirs['run_dir']) + '/reg_scale_dict.pkl'), 'wb') as f:
    pickle.dump(reg_scale_dict, f)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import pickle


celltype_all = np.asarray(celltype_all)
cellage_all = np.asarray(cellage_all)
cellclass_unique = np.unique(celltype_all)

# =========================================================
# Fit sklearn linear regression on geneexp_all_pca for each cell type
# =========================================================
reg_scale_exppca_dict = {}
coef_exppca_dict = {}
intercept_exppca_dict = {}

for cellclass_cur in cellclass_unique:
    print(f"Training sklearn LinearRegression on geneexp_all_pca: {cellclass_cur}")

    idx = np.where(celltype_all == cellclass_cur)[0]
    X = geneexp_all_pca[idx, :]
    y = cellage_all[idx]

    model_linear = LinearRegression().fit(X, y)

    reg_scale_exppca_dict[cellclass_cur] = model_linear
    coef_exppca_dict[cellclass_cur] = model_linear.coef_
    intercept_exppca_dict[cellclass_cur] = model_linear.intercept_

# Convert coefficients to a DataFrame
coef_exppca_df = pd.DataFrame(coef_exppca_dict).T
coef_exppca_df.columns = [f"PC_{i + 1}" for i in range(coef_exppca_df.shape[1])]

# Convert intercepts to a Series
intercept_exppca_series = pd.Series(intercept_exppca_dict, name="intercept")

# =========================================================
# Save the fitted regression models
# =========================================================
with open(output_path(str(run_dirs['run_dir']) + '/reg_scale_exppca_dict.pkl'), "wb") as f:
    pickle.dump(reg_scale_exppca_dict, f)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import pickle


celltype_all = np.asarray(celltype_all)
cellage_all = np.asarray(cellage_all)
cellclass_unique = np.unique(celltype_all)

# =========================================================
# Fit sklearn linear regression on COMMOT_LRscore_pathway_agg_all for each cell type
# =========================================================
reg_scale_commot_dict = {}
coef_commot_dict = {}
intercept_commot_dict = {}

for cellclass_cur in cellclass_unique:
    print(f"Training sklearn LinearRegression on COMMOT_LRscore_pathway_agg_all: {cellclass_cur}")

    idx = np.where(celltype_all == cellclass_cur)[0]
    X = COMMOT_LRscore_pathway_agg_all[idx, :]
    y = cellage_all[idx]

    model_linear = LinearRegression().fit(X, y)

    reg_scale_commot_dict[cellclass_cur] = model_linear
    coef_commot_dict[cellclass_cur] = model_linear.coef_
    intercept_commot_dict[cellclass_cur] = model_linear.intercept_

# Convert coefficients to a DataFrame
coef_commot_df = pd.DataFrame(coef_commot_dict).T
coef_commot_df.columns = [f"PC_{i + 1}" for i in range(coef_commot_df.shape[1])]

# Convert intercepts to a Series
intercept_commot_series = pd.Series(intercept_commot_dict, name="intercept")

# =========================================================
# Save the fitted regression models
# =========================================================
with open(output_path(str(run_dirs['run_dir']) + '/reg_scale_commot_dict.pkl'), "wb") as f:
    pickle.dump(reg_scale_commot_dict, f)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import pickle


celltype_all = np.asarray(celltype_all)
cellage_all = np.asarray(cellage_all)
cellclass_unique = np.unique(celltype_all)

# =========================================================
# Fit sklearn linear regression on Banksy_matrix_all_pca for each cell type
# =========================================================
reg_scale_banksy_dict = {}
coef_banksy_dict = {}
intercept_banksy_dict = {}

for cellclass_cur in cellclass_unique:
    print(f"Training sklearn LinearRegression on Banksy_matrix_all_pca: {cellclass_cur}")

    idx = np.where(celltype_all == cellclass_cur)[0]
    X = Banksy_matrix_all_pca[idx, :]
    y = cellage_all[idx]

    model_linear = LinearRegression().fit(X, y)

    reg_scale_banksy_dict[cellclass_cur] = model_linear
    coef_banksy_dict[cellclass_cur] = model_linear.coef_
    intercept_banksy_dict[cellclass_cur] = model_linear.intercept_

# Convert coefficients to a DataFrame
coef_banksy_df = pd.DataFrame(coef_banksy_dict).T
coef_banksy_df.columns = [f"PC_{i + 1}" for i in range(coef_banksy_df.shape[1])]

# Convert intercepts to a Series
intercept_banksy_series = pd.Series(intercept_banksy_dict, name="intercept")

# =========================================================
# Save the fitted regression models
# =========================================================
with open(output_path(str(run_dirs['run_dir']) + '/reg_scale_banksy_dict.pkl'), "wb") as f:
    pickle.dump(reg_scale_banksy_dict, f)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Rectangle


# =========================================================
# Prepare display tables
# Rename Sender_MI_x / Receiver_MI_x to MI-x for visualization
# =========================================================
coef_df_show = coef_df.copy()
pval_df_show = pval_df.copy()

display_columns = []
for col in coef_df_show.columns:
    if col.startswith("Receiver_MI_"):
        display_col = col.replace("Receiver_MI_", "MI-")
    elif col.startswith("Sender_MI_"):
        display_col = col.replace("Sender_MI_", "MI-")
    else:
        display_col = col
    display_columns.append(display_col)

coef_df_show.columns = display_columns
pval_df_show.columns = display_columns


# =========================================================
# Plot heatmap of regression coefficients
# =========================================================
fig, ax = plt.subplots(figsize=(20, 8))

sns.heatmap(
    coef_df_show,
    cmap="coolwarm",
    center=0,
    cbar=False,
    annot=False,
    ax=ax
)

nrows, ncols = coef_df_show.shape


# =========================================================
# Add light gray borders to each heatmap cell
# =========================================================
for i in range(nrows):
    for j in range(ncols):
        rect = Rectangle(
            (j, i),
            1,
            1,
            fill=False,
            edgecolor="#D0D0D0",
            linewidth=0.8
        )
        ax.add_patch(rect)


# =========================================================
# Mark significant cells with an asterisk (p < 0.05)
# =========================================================
for i in range(nrows):
    for j in range(ncols):
        if pval_df_show.iloc[i, j] < 0.05:
            ax.text(
                j + 0.5,
                i + 0.75,
                "*",
                ha="center",
                va="center",
                color="black",
                fontsize=14,
                fontweight="bold"
            )


# =========================================================
# Format axes
# =========================================================
ax.set_xlabel("Aggregated MI predictor", fontsize=24)
ax.set_ylabel("Cell type", fontsize=24)
ax.set_xticklabels(ax.get_xticklabels(), fontsize=20, rotation=60, ha="center")
ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)


# =========================================================
# Add a horizontal colorbar with a fixed range
# =========================================================
# norm = plt.Normalize(vmin=-1, vmax=1)
norm = plt.Normalize(vmin=-20, vmax=20)
sm = plt.cm.ScalarMappable(cmap="coolwarm", norm=norm)
sm.set_array([])

cbar = fig.colorbar(
    sm,
    ax=ax,
    orientation="horizontal",
    fraction=0.08,
    pad=0.2
)
cbar.set_label("Coefficient", fontsize=15)


# =========================================================
# Save and show figure
# =========================================================
plt.tight_layout()
plt.savefig(output_path(str(run_dirs['run_dir']) + "/AgePrediction_RegressionCoefficients_Heatmap.png"), dpi=300)
plt.savefig(output_path(str(run_dirs['run_dir']) + "/AgePrediction_RegressionCoefficients_Heatmap.pdf"), dpi=300)
plt.show()
plt.close()

In [ ]:
df_pos = np.sum((coef_df > 0) * (pval_df < 0.05),axis = 0)
##Sort df_pos
df_pos_sorted = df_pos.sort_values(ascending=False)
print(df_pos_sorted)

## Optional B3. Summarize mean MI strength across sender--receiver cell-type pairs

In [ ]:
import numpy as np
import pandas as pd

# =========================================================
# Collect edge-level metadata
# =========================================================
age_edge = []
sender_cellclass = []
receiver_cellclass = []

for batch_index in range(len(processed.spidernet_data)):
    edge_index_cur = processed.spidernet_data[batch_index]["edge_index"].cpu().numpy()
    cellclass_cursample = np.asarray(processed.adata_list[batch_index].obs["celltype"])
    age_cursample = np.asarray(processed.adata_list[batch_index].obs["age"])[edge_index_cur[:, 0]]

    age_edge.extend(age_cursample.tolist())
    sender_cellclass.extend(cellclass_cursample[edge_index_cur[:, 0]].tolist())
    receiver_cellclass.extend(cellclass_cursample[edge_index_cur[:, 1]].tolist())

age_edge = np.asarray(age_edge)
sender_cellclass = np.asarray(sender_cellclass)
receiver_cellclass = np.asarray(receiver_cellclass)


# =========================================================
# Build edge-level DataFrame
# =========================================================
num_mi = Factor_envir_use.shape[1]
mi_columns = [f"MI-{i + 1}" for i in range(num_mi)]

edge_df = pd.DataFrame({
    "Sender": sender_cellclass,
    "Receiver": receiver_cellclass,
})

# Add MI values
mi_df = pd.DataFrame(Factor_envir_use, columns=mi_columns)
edge_df = pd.concat([edge_df, mi_df], axis=1)

# Keep only cross-cell-type edges
edge_df = edge_df[edge_df["Sender"] != edge_df["Receiver"]].copy()

# Create pair label
edge_df["Pair"] = edge_df["Sender"] + "_to_" + edge_df["Receiver"]


# =========================================================
# Compute mean MI for each sender-receiver cell-class pair
# =========================================================
mean_MI_cellclasspair_all = (
    edge_df.groupby("Pair")[mi_columns]
    .mean()
    .T
)

print(mean_MI_cellclasspair_all.shape)
mean_MI_cellclasspair_all.head()

In [ ]:
df_plot_all = pd.DataFrame({
    'MI': np.repeat(mean_MI_cellclasspair_all.index, mean_MI_cellclasspair_all.shape[1]),
    'Pair': np.tile(mean_MI_cellclasspair_all.columns, mean_MI_cellclasspair_all.shape[0]),
    'Mean': mean_MI_cellclasspair_all.values.flatten()
})

MI_OI = 'MI-29'
# MI_OI = 'MI-24'
# MI_OI = 'MI-26'
# ---- 1) Filter for MI of interest and T cell → other cell type ----
df_MIOI_all = df_plot_all[(df_plot_all['MI'] == MI_OI)]

# ---- 2) Clean: extract receiver cell type ----
df_MIOI_all['Receiver'] = df_MIOI_all['Pair'].str.split("_to_").str[1]
df_MIOI_all['Sender'] = df_MIOI_all['Pair'].str.split("_to_").str[0]

# ---- 3) Save CSV ----
csv_path = str(run_dirs['run_dir']) + "/" +MI_OI + "_all_links.csv"
df_MIOI_all.to_csv(output_path(csv_path), index=False)
print("Saved CSV:", csv_path)


## Optional B4. Relate cell age to mean MI strength for selected cell-type pairs

In [ ]:
target_cellclass = "T cell"
# target_cellclass = "NSC"
target_cellclass_pair_df = pd.DataFrame({"Sender": np.repeat(target_cellclass, len(cellclass_unique)).tolist() + cellclass_unique.tolist(),
                                         "Receiver": cellclass_unique.tolist() + np.repeat(target_cellclass, len(cellclass_unique)).tolist()})
##Remove the rows where Sender and Receiver are the same
target_cellclass_pair_df = target_cellclass_pair_df[target_cellclass_pair_df['Sender'] != target_cellclass_pair_df['Receiver']]

In [ ]:
corr_MI_cellclasspair = pd.DataFrame(0,index = ["MI-" + str(i + 1) for i in range(Factor_envir_use.shape[1])],
                                     columns = [target_cellclass_pair_df['Sender'].values[i] + "_to_" + target_cellclass_pair_df['Receiver'].values[i] for i in range(target_cellclass_pair_df.shape[0])])
pvaluecorr_MI_cellclasspair = pd.DataFrame(0,index = ["MI-" + str(i + 1) for i in range(Factor_envir_use.shape[1])],
                                     columns = [target_cellclass_pair_df['Sender'].values[i] + "_to_" + target_cellclass_pair_df['Receiver'].values[i] for i in range(target_cellclass_pair_df.shape[0])])
mean_MI_cellclasspair = pd.DataFrame(0,index = ["MI-" + str(i + 1) for i in range(Factor_envir_use.shape[1])],
                                     columns = [target_cellclass_pair_df['Sender'].values[i] + "_to_" + target_cellclass_pair_df['Receiver'].values[i] for i in range(target_cellclass_pair_df.shape[0])])
samplesize_MI_cellclasspair = pd.DataFrame(0,index = ["MI-" + str(i + 1) for i in range(Factor_envir_use.shape[1])],
                                     columns = [target_cellclass_pair_df['Sender'].values[i] + "_to_" + target_cellclass_pair_df['Receiver'].values[i] for i in range(target_cellclass_pair_df.shape[0])])
# print(corr_MI_cellclasspair.shape)
for MI_index in range(Factor_envir_use.shape[1]):
    # print(MI_index)
    for cellclass_pair_cur in corr_MI_cellclasspair.columns:
        sendercellclass_cur = cellclass_pair_cur.split("_to_")[0]
        receivercellclass_cur = cellclass_pair_cur.split("_to_")[1]
        edge_index_choose = np.where((sender_cellclass == sendercellclass_cur) & (receiver_cellclass == receivercellclass_cur))[0]
        Factor_envir_use_choose = Factor_envir_use[edge_index_choose, MI_index]
        age_edge_choose = age_edge[edge_index_choose]
        age_edge_choose = age_edge_choose.astype(np.float32)
        samplesize_MI_cellclasspair.iloc[MI_index, samplesize_MI_cellclasspair.columns.get_loc(cellclass_pair_cur)] = len(edge_index_choose)
        ##Calculate the spearman correlation
        if len(edge_index_choose) < 10:
            corr_MI_cellclasspair.iloc[MI_index, corr_MI_cellclasspair.columns.get_loc(cellclass_pair_cur)] = np.nan
            pvaluecorr_MI_cellclasspair.iloc[MI_index, pvaluecorr_MI_cellclasspair.columns.get_loc(cellclass_pair_cur)] = np.nan
        else:
            spearmanr_res = stats.spearmanr(Factor_envir_use_choose, age_edge_choose)
            corr_MI_cellclasspair.iloc[MI_index, corr_MI_cellclasspair.columns.get_loc(cellclass_pair_cur)] = spearmanr_res.correlation
            pvaluecorr_MI_cellclasspair.iloc[MI_index, pvaluecorr_MI_cellclasspair.columns.get_loc(cellclass_pair_cur)] = spearmanr_res.pvalue
        mean_MI_cellclasspair.iloc[MI_index, mean_MI_cellclasspair.columns.get_loc(cellclass_pair_cur)] = np.mean(Factor_envir_use_choose)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# ---------- 1) Order MI rows based on mean corr ----------
MI_order = corr_MI_cellclasspair.mean(axis=1).sort_values(ascending=False).index

corr_MI_cellclasspair = corr_MI_cellclasspair.loc[MI_order, :]
mean_MI_cellclasspair = mean_MI_cellclasspair.loc[MI_order, :]
pvaluecorr_MI_cellclasspair = pvaluecorr_MI_cellclasspair.loc[MI_order, :]

# ---------- 2) Build long-format df based on new order ----------
df_plot = pd.DataFrame({
    'MI': np.repeat(mean_MI_cellclasspair.index, mean_MI_cellclasspair.shape[1]),
    'Pair': np.tile(mean_MI_cellclasspair.columns, mean_MI_cellclasspair.shape[0]),
    'Mean': mean_MI_cellclasspair.values.flatten(),
    'Corr': corr_MI_cellclasspair.values.flatten(),
    'P-value': pvaluecorr_MI_cellclasspair.values.flatten()
})

# Normalize sizes
size_min = 10
size_max = 500
mean_norm = (df_plot['Mean'] - df_plot['Mean'].min()) / (df_plot['Mean'].max() - df_plot['Mean'].min() + 1e-12)
df_plot['Size'] = size_min + mean_norm * (size_max - size_min)

# ---------- 3) Plot ----------
fig, ax = plt.subplots(figsize=(20, 10))

scatter = ax.scatter(
    x=df_plot['Pair'],
    y=df_plot['MI'],
    s=df_plot['Size'],
    c=df_plot['Corr'],
    cmap='bwr',
    alpha=0.8,
    vmin=-0.8,
    vmax=0.8,
    edgecolor='k',
    linewidth=0.3
)

cbar = fig.colorbar(scatter, ax=ax)
cbar.set_label("Spearman Correlation")

ax.set_xticklabels(df_plot['Pair'].unique(), rotation=75, ha='right')
ax.set_xlabel("Cell Type Pair (Sender → Receiver)")
ax.set_ylabel("MI Dimensions")
ax.set_title("Bubble Plot: Corr (color) & Mean MI Level (size)")

fig.tight_layout()

fig.savefig(
    output_path(str(run_dirs['run_dir']) + '/Bubble_MI_CellclassPair_' + target_cellclass + '.png'),
    dpi=300
)
plt.close()


In [ ]:
# ---- 1) Filter for MI-29 and T cell → other cell type ----
df_MI29 = df_plot[(df_plot['MI'] == 'MI-29') &
                  (df_plot['Pair'].str.startswith('T cell_to_'))]
# df_MI29 = df_plot[(df_plot['MI'] == 'MI-24') &
#                   (df_plot['Pair'].str.startswith('T cell_to_'))]

# ---- 2) Clean: extract receiver cell type ----
df_MI29['Receiver'] = df_MI29['Pair'].str.replace('T cell_to_', '', regex=False)

# ---- 3) Save CSV ----
csv_path = str(run_dirs['run_dir']) + "/MI-29_Tcell_links.csv"
df_MI29.to_csv(output_path(csv_path), index=False)
print("Saved CSV:", csv_path)


## Optional B5. Visualize a selected meta-interaction in situ

In [ ]:
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import LinearSegmentedColormap, Normalize
##Show the in-situ meta-interaction plot
save_path_insituMI = str(run_dirs['run_dir']) + "/In_situ_meta_interaction/"
if not os.path.exists(save_path_insituMI):
    os.makedirs(save_path_insituMI)
##
cellclass_use_unique = np.unique(np.hstack([processed.spidernet_data[i]['cell_class'] for i in range(len(processed.spidernet_data))]))
##


In [ ]:
## Select the MI to visualize
MI_OI = 'MI-29'
MI_index = int(MI_OI.split("-")[1]) - 1
##
gray_other_cells = True 
##
# vis_mode = 1
vis_mode = 2

In [ ]:


if vis_mode == 1:
    base_size = 2
    ratio_size = 2
    arrow_head_length = 0.2
    arrow_head_width = 0.2
    arrow_lw = 10
    dpi = 500
elif vis_mode == 2:
    base_size = 2
    ratio_size = 1.5
    arrow_head_length = 0.03
    arrow_head_width = 0.05
    arrow_lw = 0.7
    dpi = 1500


batch_cell_unique = pd.read_pickle(input_path(str(run_dirs['run_dir']) + '/batch_cell_unique.pkl'))
sample_unique = batch_cell_unique

for sample_cur in sample_unique:

    sample_index_cur = np.where(batch_cell_unique == sample_cur)[0][0]

    print(f"Processing sample: {sample_cur} ({sample_index_cur+1}/{len(sample_unique)})")

    adata_cursample = processed.adata_list[sample_index_cur]
    CellFlowMap_data_pyg_cursample = processed.spidernet_data[sample_index_cur]

    MI_use_cursample = results['factor_envir_list'][sample_index_cur]
    spatial_location_use_cursample = adata_cursample.obsm['spatial']
    edge_index_use_cursample = CellFlowMap_data_pyg_cursample['edge_index'].cpu().detach().numpy()

    cellclass_use_cursample = np.array(CellFlowMap_data_pyg_cursample['cell_class'])

    # Encode celltype categories
    cat_series = pd.Categorical(cellclass_use_cursample,
                                categories=cellclass_use_unique,
                                ordered=True)
    cellclass_use_cursample_numerical = cat_series.codes

    # Color map for cell types (final order)
    colorct_ref = [
        "#4F6AB5", "#C176AE", "#EB2029", "#909190", "#7D287C",
        "#6BCCD7", "#F48B20", "#F7A45C", "#258B42", "#9FCF89",
        "#98CB3B", "#9750B4", "#3FB7E7", "#8ACEEC", "#7F2321",
        "#F321F8", "#D25C5A", "#F67F70"
    ]

    # Gray option
    if gray_other_cells:
        gray_color = "#D0D0D0"
        tcell_color = "#F321F8"
        colorct_ref = [(tcell_color if cls == "T cell" else gray_color) for cls in cellclass_use_unique]

    cmap_celltype = ListedColormap(colorct_ref)
    cmap_cells = ListedColormap([cmap_celltype(i) for i in range(
        len(np.unique(cellclass_use_cursample_numerical)))])

    factor_method_cur = MI_use_cursample[:, MI_index]

    # T cell → all other cell types
    sender_label = "T cell"
    receiver_list = [ct for ct in cellclass_use_unique if ct != sender_label]

    Avg_MI_cellclass_pair_merge_use_cur = pd.DataFrame({
        "Sender": [sender_label] * len(receiver_list),
        "Receiver": receiver_list
    })
    Avg_MI_cellclass_pair_merge_use_cur["SR_merge"] = (
        Avg_MI_cellclass_pair_merge_use_cur["Sender"] + " → " +
        Avg_MI_cellclass_pair_merge_use_cur["Receiver"]
    )
    SR_merge_list = Avg_MI_cellclass_pair_merge_use_cur["SR_merge"].values

    cmap = LinearSegmentedColormap.from_list("white_to_red", ["white", "red"], N=256)
    norm = Normalize(vmin=0, vmax=1)

    plt.close()
    fig, ax = plt.subplots(figsize=(31, 25), facecolor='white')
    ax.set_facecolor('white')

    factor_method_cur_q95 = 0.2

    # Draw arrows using the parameter set above
    for edge_index0 in range(edge_index_use_cursample.shape[0]):

        sender_cellclass = cellclass_use_cursample[edge_index_use_cursample[edge_index0, 0]]
        receiver_cellclass = cellclass_use_cursample[edge_index_use_cursample[edge_index0, 1]]

        sr_merge = sender_cellclass + " → " + receiver_cellclass

        if (sr_merge in SR_merge_list) and (factor_method_cur[edge_index0] > factor_method_cur_q95):

            sender_i = int(edge_index_use_cursample[edge_index0, 0])
            receiver_i = int(edge_index_use_cursample[edge_index0, 1])

            edge_color = cmap(norm(factor_method_cur[edge_index0]))

            ax.annotate(
                '',
                xy=(spatial_location_use_cursample[receiver_i, 0],
                    spatial_location_use_cursample[receiver_i, 1]),
                xytext=(spatial_location_use_cursample[sender_i, 0],
                        spatial_location_use_cursample[sender_i, 1]),
                arrowprops=dict(
                    color=edge_color,
                    arrowstyle=f"->,head_length={arrow_head_length},head_width={arrow_head_width}",
                    lw=arrow_lw,
                    zorder=1
                )
            )

    # Draw cells, enlarging T cells for visibility
    sizes = np.array([base_size] * len(cellclass_use_cursample))
    tcell_idx = np.where(cellclass_use_cursample == "T cell")[0]
    sizes[tcell_idx] = base_size * ratio_size

    ax.scatter(
        spatial_location_use_cursample[:, 0],
        spatial_location_use_cursample[:, 1],
        c=cellclass_use_cursample_numerical,
        s=sizes,
        cmap=cmap_cells,
        zorder=2,
        rasterized=True
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_title('')

    for spine in ax.spines.values():
        spine.set_visible(False)

    # plt.savefig(
    #     save_path_insituMI + f"Insitu_sample{sample_cur}_MI29_celltype_Mode{str(vis_mode)}.png",
    #     dpi=dpi, bbox_inches='tight',
    #     facecolor=fig.get_facecolor()
    # )
    plt.savefig(
        output_path(save_path_insituMI + f"Insitu_sample{sample_cur}_MI29_celltype_Mode{str(vis_mode)}.pdf"),
        dpi=dpi, bbox_inches='tight',
        facecolor=fig.get_facecolor()
    )
    plt.close()


## Optional B6. Compare aging module scores across MI-29 activity groups

In [ ]:
# ==============================================================
# Check aging genes log2 fold change: Old slices vs Young slices
# --------------------------------------------------------------
# Goal:
#   1. Build aging/senescence gene set from MERFISH gene panel.
#   2. Sort slices by age.
#   3. Define 5 youngest slices as Young and 5 oldest slices as Old.
#   4. For each aging gene, compute log2FC = log2(mean_old / mean_young).
#
# Main log2FC definition:
#   - First compute mean expression of each gene within each slice.
#   - Then average slice-level means across 5 Old slices and 5 Young slices.
#   - log2FC_old_vs_young = log2((old_slice_mean + pseudocount) /
#                                (young_slice_mean + pseudocount))
#
# Required upstream objects:
#   processed
#   DATA_ROOT
#   OUTPUT_ROOT
# ==============================================================

import re
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import mannwhitneyu


AGING_GENE_LFC_N_YOUNG_SLICES = 5
AGING_GENE_LFC_N_OLD_SLICES = 5
AGING_GENE_LFC_PSEUDOCOUNT = 1e-6


def _extract_first_number(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(value))
    if match is None:
        return np.nan
    try:
        return float(match.group(0))
    except Exception:
        return np.nan


def _resolve_gene_names_from_varnames_for_lfc(var_names, genes):
    """Resolve requested gene symbols to actual var_names, case-insensitively."""
    var_names = pd.Index([str(x) for x in var_names])
    upper_to_actual = {}
    for g in var_names:
        upper_to_actual.setdefault(str(g).upper(), str(g))

    resolved = {}
    missing = []
    for gene in genes:
        gene = str(gene).strip()
        if len(gene) == 0:
            continue

        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)

    return resolved, missing


def _get_dense_gene_matrix_for_lfc(adata, genes_actual):
    gene_idx = [adata.var_names.get_loc(g) for g in genes_actual]
    X_sub = adata.X[:, gene_idx]
    if sparse.issparse(X_sub):
        X_sub = X_sub.toarray()
    else:
        X_sub = np.asarray(X_sub)
    return X_sub.astype(float, copy=False)


# -----------------------------
# 1. Build aging gene set
# -----------------------------
genepanel_path = DATA_ROOT / "Supp_table" / "2023-12-22736D-TableS1_MERFISHGenePanel.xlsx"
genepanel_table = pd.read_excel(input_path(genepanel_path))

required_cols = {"Rationale for inclusion", "Vizgen Gene"}
missing_cols = required_cols.difference(genepanel_table.columns)
if len(missing_cols) > 0:
    raise KeyError(
        f"MERFISH gene-panel table is missing columns: {sorted(missing_cols)}"
    )

rationale = genepanel_table["Rationale for inclusion"].fillna("").astype(str)
aging_mask = rationale.str.contains("aging|senescence", case=False, regex=True)

aging_genes_lfc_requested = (
    genepanel_table.loc[aging_mask, "Vizgen Gene"]
    .dropna()
    .astype(str)
    .map(lambda x: x.strip())
)
aging_genes_lfc_requested = [g for g in aging_genes_lfc_requested if len(g) > 0]
aging_genes_lfc_requested = list(dict.fromkeys(aging_genes_lfc_requested))

if len(aging_genes_lfc_requested) == 0:
    raise ValueError(
        "No aging/senescence genes were found from the MERFISH gene-panel rationale column."
    )

# -----------------------------
# 2. Resolve aging genes in expression matrix
# -----------------------------
ref_var_names = processed.adata_list[0].var_names.astype(str)
resolved_genes_lfc, missing_aging_genes_lfc = _resolve_gene_names_from_varnames_for_lfc(
    ref_var_names,
    aging_genes_lfc_requested,
)

aging_genes_lfc_used_requested = [
    g for g in aging_genes_lfc_requested
    if g in resolved_genes_lfc
]
aging_genes_lfc_actual = [
    resolved_genes_lfc[g]
    for g in aging_genes_lfc_used_requested
]

if len(aging_genes_lfc_actual) == 0:
    raise ValueError(
        "None of the aging/senescence genes are present in processed.adata_list[0].var_names."
    )

print(f"Aging genes used for log2FC: {len(aging_genes_lfc_actual)} / {len(aging_genes_lfc_requested)}")
if len(missing_aging_genes_lfc) > 0:
    print(f"[Warning] Missing aging genes skipped: {missing_aging_genes_lfc}")


# -----------------------------
# 3. Build slice-level expression table
# -----------------------------
slice_mean_rows = []
cell_level_parts = []

for slice_index, adata_cur in enumerate(processed.adata_list):
    missing_cur = [g for g in aging_genes_lfc_actual if g not in adata_cur.var_names]
    if len(missing_cur) > 0:
        raise ValueError(
            f"Slice {slice_index} is missing aging genes present in slice 0: {missing_cur}"
        )

    if "age" not in adata_cur.obs.columns:
        raise KeyError("adata.obs must contain an 'age' column for slice age grouping.")

    age_values = (
        pd.Series(adata_cur.obs["age"])
        .dropna()
        .astype(str)
        .str.strip()
    )
    age_values = age_values[(age_values != "") & (age_values.str.lower() != "nan")]

    if len(age_values) == 0:
        age_value = np.nan
    else:
        age_value = age_values.iloc[0]

    age_numeric = _extract_first_number(age_value)

    X_gene_cur = _get_dense_gene_matrix_for_lfc(adata_cur, aging_genes_lfc_actual)

    # Slice-level mean for each gene
    slice_gene_mean = np.nanmean(X_gene_cur, axis=0)

    for requested_gene, actual_gene, mean_expr in zip(
        aging_genes_lfc_used_requested,
        aging_genes_lfc_actual,
        slice_gene_mean,
    ):
        slice_mean_rows.append({
            "slice_index": slice_index,
            "age": age_value,
            "age_numeric": age_numeric,
            "n_cells": int(adata_cur.n_obs),
            "requested_gene": requested_gene,
            "actual_gene": actual_gene,
            "slice_mean_expression": float(mean_expr),
        })

    # Keep cell-level values as a secondary check
    cell_df_cur = pd.DataFrame(
        X_gene_cur,
        columns=aging_genes_lfc_actual,
    )
    cell_df_cur["slice_index"] = slice_index
    cell_df_cur["age"] = age_value
    cell_df_cur["age_numeric"] = age_numeric
    cell_level_parts.append(cell_df_cur)

aging_gene_slice_mean_df = pd.DataFrame(slice_mean_rows)

slice_age_df = (
    aging_gene_slice_mean_df[
        ["slice_index", "age", "age_numeric", "n_cells"]
    ]
    .drop_duplicates()
    .sort_values(["age_numeric", "slice_index"])
    .reset_index(drop=True)
)

if slice_age_df["age_numeric"].isna().any():
    bad_slices = slice_age_df.loc[
        slice_age_df["age_numeric"].isna(),
        ["slice_index", "age"],
    ]
    raise ValueError(
        "Some slices have non-numeric age values and cannot be sorted into young/old groups:\n"
        + str(bad_slices)
    )

n_slices_total = slice_age_df.shape[0]
required_n = AGING_GENE_LFC_N_YOUNG_SLICES + AGING_GENE_LFC_N_OLD_SLICES
if n_slices_total < required_n:
    raise ValueError(
        f"Need at least {required_n} slices to select "
        f"{AGING_GENE_LFC_N_YOUNG_SLICES} young + {AGING_GENE_LFC_N_OLD_SLICES} old slices, "
        f"but found only {n_slices_total} slices."
    )

young_slice_ids = (
    slice_age_df
    .head(AGING_GENE_LFC_N_YOUNG_SLICES)["slice_index"]
    .astype(int)
    .tolist()
)

old_slice_ids = (
    slice_age_df
    .tail(AGING_GENE_LFC_N_OLD_SLICES)["slice_index"]
    .astype(int)
    .tolist()
)

slice_group_map = {
    **{sid: "Young" for sid in young_slice_ids},
    **{sid: "Old" for sid in old_slice_ids},
}

slice_age_df["AgeGroup_5young_5old"] = slice_age_df["slice_index"].map(slice_group_map)
selected_slice_age_df = slice_age_df[
    slice_age_df["AgeGroup_5young_5old"].isin(["Young", "Old"])
].copy()

print("Selected 5 young + 5 old slices:")
display(selected_slice_age_df)


# -----------------------------
# 4. Compute log2FC old vs young for each aging gene
# -----------------------------
aging_gene_slice_mean_df["AgeGroup_5young_5old"] = aging_gene_slice_mean_df["slice_index"].map(slice_group_map)

selected_slice_mean_df = aging_gene_slice_mean_df[
    aging_gene_slice_mean_df["AgeGroup_5young_5old"].isin(["Young", "Old"])
].copy()

lfc_rows = []

for actual_gene, df_gene in selected_slice_mean_df.groupby("actual_gene", sort=False):
    requested_gene_values = df_gene["requested_gene"].dropna().astype(str).unique()
    requested_gene = requested_gene_values[0] if len(requested_gene_values) > 0 else actual_gene

    young_vals = pd.to_numeric(
        df_gene.loc[
            df_gene["AgeGroup_5young_5old"] == "Young",
            "slice_mean_expression",
        ],
        errors="coerce",
    ).dropna()

    old_vals = pd.to_numeric(
        df_gene.loc[
            df_gene["AgeGroup_5young_5old"] == "Old",
            "slice_mean_expression",
        ],
        errors="coerce",
    ).dropna()

    mean_young_slice = float(np.nanmean(young_vals)) if len(young_vals) > 0 else np.nan
    mean_old_slice = float(np.nanmean(old_vals)) if len(old_vals) > 0 else np.nan

    log2fc_slice_mean_old_vs_young = np.log2(
        (mean_old_slice + AGING_GENE_LFC_PSEUDOCOUNT)
        / (mean_young_slice + AGING_GENE_LFC_PSEUDOCOUNT)
    )

    if len(young_vals) > 0 and len(old_vals) > 0:
        stat, pval = mannwhitneyu(young_vals, old_vals, alternative="two-sided")
    else:
        stat, pval = np.nan, np.nan

    lfc_rows.append({
        "requested_gene": requested_gene,
        "actual_gene": actual_gene,
        "n_young_slices": int(len(young_vals)),
        "n_old_slices": int(len(old_vals)),
        "mean_young_slice_mean_expression": mean_young_slice,
        "mean_old_slice_mean_expression": mean_old_slice,
        "log2FC_old_vs_young_slice_mean": float(log2fc_slice_mean_old_vs_young),
        "old_greater_than_young": bool(log2fc_slice_mean_old_vs_young > 0),
        "mannwhitneyu_stat_slice_means": stat,
        "pvalue_slice_means": pval,
    })

aging_gene_old_vs_young_lfc_df = pd.DataFrame(lfc_rows)


# -----------------------------
# 5. Optional secondary check:
#    cell-level mean expression pooled across selected young/old cells
# -----------------------------
cell_level_df = pd.concat(cell_level_parts, axis=0, ignore_index=True)
cell_level_df["AgeGroup_5young_5old"] = cell_level_df["slice_index"].map(slice_group_map)
cell_level_df = cell_level_df[
    cell_level_df["AgeGroup_5young_5old"].isin(["Young", "Old"])
].copy()

cell_level_lfc_rows = []

for requested_gene, actual_gene in zip(aging_genes_lfc_used_requested, aging_genes_lfc_actual):
    young_cell_vals = pd.to_numeric(
        cell_level_df.loc[cell_level_df["AgeGroup_5young_5old"] == "Young", actual_gene],
        errors="coerce",
    ).dropna()

    old_cell_vals = pd.to_numeric(
        cell_level_df.loc[cell_level_df["AgeGroup_5young_5old"] == "Old", actual_gene],
        errors="coerce",
    ).dropna()

    mean_young_cell = float(np.nanmean(young_cell_vals)) if len(young_cell_vals) > 0 else np.nan
    mean_old_cell = float(np.nanmean(old_cell_vals)) if len(old_cell_vals) > 0 else np.nan

    log2fc_cell_mean_old_vs_young = np.log2(
        (mean_old_cell + AGING_GENE_LFC_PSEUDOCOUNT)
        / (mean_young_cell + AGING_GENE_LFC_PSEUDOCOUNT)
    )

    cell_level_lfc_rows.append({
        "actual_gene": actual_gene,
        "mean_young_cell_expression": mean_young_cell,
        "mean_old_cell_expression": mean_old_cell,
        "log2FC_old_vs_young_cell_mean": float(log2fc_cell_mean_old_vs_young),
    })

aging_gene_cell_level_lfc_df = pd.DataFrame(cell_level_lfc_rows)

aging_gene_old_vs_young_lfc_df = aging_gene_old_vs_young_lfc_df.merge(
    aging_gene_cell_level_lfc_df,
    on="actual_gene",
    how="left",
)

aging_gene_old_vs_young_lfc_df = aging_gene_old_vs_young_lfc_df.sort_values(
    "log2FC_old_vs_young_slice_mean",
    ascending=False,
).reset_index(drop=True)


# -----------------------------
# 6. Save and display
# -----------------------------
slice_selection_path = OUTPUT_ROOT / "aging_gene_log2FC_5young_5old_selected_slices.csv"
slice_mean_path = OUTPUT_ROOT / "aging_gene_log2FC_5young_5old_slice_mean_expression_long.csv"
lfc_path = OUTPUT_ROOT / "aging_gene_log2FC_5young_5old_old_vs_young.csv"

selected_slice_age_df.to_csv(output_path(slice_selection_path), index=False)
selected_slice_mean_df.to_csv(output_path(slice_mean_path), index=False)
aging_gene_old_vs_young_lfc_df.to_csv(output_path(lfc_path), index=False)

print(f"Saved selected slice table to: {slice_selection_path}")
print(f"Saved slice-level expression table to: {slice_mean_path}")
print(f"Saved aging gene log2FC table to: {lfc_path}")

n_total_genes = aging_gene_old_vs_young_lfc_df.shape[0]
n_positive = int(aging_gene_old_vs_young_lfc_df["old_greater_than_young"].sum())
n_nonpositive = n_total_genes - n_positive

print(
    f"Aging genes with log2FC(old vs young) > 0 based on slice-level means: "
    f"{n_positive} / {n_total_genes}"
)

if n_nonpositive > 0:
    print("Aging genes with log2FC(old vs young) <= 0:")
    display(
        aging_gene_old_vs_young_lfc_df.loc[
            ~aging_gene_old_vs_young_lfc_df["old_greater_than_young"],
            [
                "requested_gene",
                "actual_gene",
                "mean_young_slice_mean_expression",
                "mean_old_slice_mean_expression",
                "log2FC_old_vs_young_slice_mean",
                "pvalue_slice_means",
            ],
        ]
    )
else:
    print("All aging genes have log2FC(old vs young) > 0 based on slice-level means.")

display(aging_gene_old_vs_young_lfc_df)

In [ ]:
# ==============================================================
# Aging module score: good aging genes only, global z-scored + mean
# --------------------------------------------------------------
# This version filters aging genes before module-score calculation:
#
#   good aging genes =
#       aging_gene_old_vs_young_lfc_df["log2FC_old_vs_young_cell_mean"] > 0.2
#
# Score definition:
#   1. Build aging/senescence gene set from the MERFISH gene panel.
#   2. Resolve genes in current expression matrix.
#   3. Keep only genes with cell-level log2FC(old vs young) > 0.2.
#   4. Concatenate all cells across all slices.
#   5. For each selected good aging gene, compute one global mean/std.
#   6. For each cell, z-score each gene using that global reference.
#   7. Average the global z-scored genes to obtain one aging module score.
#
# Required upstream object:
#   aging_gene_old_vs_young_lfc_df
#
# The resulting vector keeps the same cell order as:
#   processed.adata_list[0], processed.adata_list[1], ...
# ==============================================================

import os
import numpy as np
import pandas as pd
from scipy import sparse

MODULE_SCORE_METHOD = "zscore_mean_global_good_aging_genes_cellLFC_gt0.2"
GOOD_AGING_GENE_CELL_LFC_THRESHOLD = 0.2


def _resolve_gene_names_from_varnames(var_names, genes):
    """Resolve requested gene symbols to actual var_names, case-insensitively."""
    var_names = pd.Index([str(x) for x in var_names])
    upper_to_actual = {}
    for g in var_names:
        upper_to_actual.setdefault(str(g).upper(), str(g))

    resolved = {}
    missing = []
    for gene in genes:
        gene = str(gene).strip()
        if len(gene) == 0:
            continue
        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)
    return resolved, missing


def _get_dense_gene_matrix(adata, genes_actual):
    """Return dense expression matrix for selected genes only."""
    gene_idx = [adata.var_names.get_loc(g) for g in genes_actual]
    X_sub = adata.X[:, gene_idx]
    if sparse.issparse(X_sub):
        X_sub = X_sub.toarray()
    else:
        X_sub = np.asarray(X_sub)
    return X_sub.astype(float, copy=False)


if "aging_gene_old_vs_young_lfc_df" not in globals():
    raise NameError(
        "Please run the previous aging-gene old-vs-young log2FC cell first, "
        "so that aging_gene_old_vs_young_lfc_df exists."
    )

required_lfc_cols = {"actual_gene", "log2FC_old_vs_young_cell_mean"}
missing_lfc_cols = required_lfc_cols.difference(aging_gene_old_vs_young_lfc_df.columns)
if len(missing_lfc_cols) > 0:
    raise KeyError(
        "aging_gene_old_vs_young_lfc_df is missing required columns: "
        f"{sorted(missing_lfc_cols)}"
    )


# -----------------------------
# 1. Build aging gene set from MERFISH gene panel
# -----------------------------
genepanel_path = DATA_ROOT / "Supp_table" / "2023-12-22736D-TableS1_MERFISHGenePanel.xlsx"
genepanel_table = pd.read_excel(input_path(genepanel_path))

required_cols = {"Rationale for inclusion", "Vizgen Gene"}
missing_cols = required_cols.difference(genepanel_table.columns)
if len(missing_cols) > 0:
    raise KeyError(
        f"MERFISH gene-panel table is missing columns: {sorted(missing_cols)}"
    )

rationale = genepanel_table["Rationale for inclusion"].fillna("").astype(str)
aging_mask = rationale.str.contains("aging|senescence", case=False, regex=True)

aging_genes = (
    genepanel_table.loc[aging_mask, "Vizgen Gene"]
    .dropna()
    .astype(str)
    .map(lambda x: x.strip())
)
aging_genes = [g for g in aging_genes if len(g) > 0]
aging_genes = list(dict.fromkeys(aging_genes))

if len(aging_genes) == 0:
    raise ValueError(
        "No aging/senescence genes were found from the MERFISH gene-panel rationale column."
    )


# -----------------------------
# 2. Resolve genes in current expression matrix
# -----------------------------
ref_var_names = processed.adata_list[0].var_names.astype(str)
resolved_genes, missing_aging_genes = _resolve_gene_names_from_varnames(
    ref_var_names,
    aging_genes,
)

aging_genes_used_requested_all = [g for g in aging_genes if g in resolved_genes]
aging_genes_actual_all = [resolved_genes[g] for g in aging_genes_used_requested_all]

if len(aging_genes_actual_all) == 0:
    raise ValueError(
        "None of the aging/senescence genes are present in processed.adata_list[0].var_names."
    )

print(f"Aging genes resolved before log2FC filtering: {len(aging_genes_actual_all)} / {len(aging_genes)}")
if len(missing_aging_genes) > 0:
    print(f"[Warning] Missing aging genes skipped before log2FC filtering: {missing_aging_genes}")


# -----------------------------
# 3. Keep only good aging genes:
#    cell-level log2FC(old vs young) > 0.2
# -----------------------------
lfc_df = aging_gene_old_vs_young_lfc_df.copy()
lfc_df["actual_gene"] = lfc_df["actual_gene"].astype(str)
lfc_df["log2FC_old_vs_young_cell_mean"] = pd.to_numeric(
    lfc_df["log2FC_old_vs_young_cell_mean"],
    errors="coerce",
)

good_lfc_df = lfc_df[
    np.isfinite(lfc_df["log2FC_old_vs_young_cell_mean"])
    & (lfc_df["log2FC_old_vs_young_cell_mean"] > GOOD_AGING_GENE_CELL_LFC_THRESHOLD)
].copy()

good_actual_gene_set = set(good_lfc_df["actual_gene"].astype(str))

aging_genes_used_requested = []
aging_genes_actual = []

for requested_gene, actual_gene in zip(aging_genes_used_requested_all, aging_genes_actual_all):
    if str(actual_gene) in good_actual_gene_set:
        aging_genes_used_requested.append(requested_gene)
        aging_genes_actual.append(actual_gene)

if len(aging_genes_actual) == 0:
    raise ValueError(
        "No aging genes passed the filter: "
        f"log2FC_old_vs_young_cell_mean > {GOOD_AGING_GENE_CELL_LFC_THRESHOLD}."
    )

good_lfc_keep_df = good_lfc_df[
    good_lfc_df["actual_gene"].isin(aging_genes_actual)
].copy()

print(
    "Good aging genes used after cell-level log2FC filtering: "
    f"{len(aging_genes_actual)} / {len(aging_genes_actual_all)} resolved aging genes"
)
print(
    f"Filter: log2FC_old_vs_young_cell_mean > {GOOD_AGING_GENE_CELL_LFC_THRESHOLD}"
)

display(
    good_lfc_keep_df[
        [
            "requested_gene",
            "actual_gene",
            "mean_young_cell_expression",
            "mean_old_cell_expression",
            "log2FC_old_vs_young_cell_mean",
            "log2FC_old_vs_young_slice_mean",
        ]
    ].sort_values("log2FC_old_vs_young_cell_mean", ascending=False)
)


# -----------------------------
# 4. Concatenate selected good-aging-gene expression across slices
# -----------------------------
X_aging_parts = []
barcode_parts = []
age_parts = []
celltype_parts = []
slice_parts = []

for slice_index, adata_cur in enumerate(processed.adata_list):
    missing_cur = [g for g in aging_genes_actual if g not in adata_cur.var_names]
    if len(missing_cur) > 0:
        raise ValueError(
            f"Slice {slice_index} is missing selected good aging genes: {missing_cur}"
        )

    X_aging_cur = _get_dense_gene_matrix(adata_cur, aging_genes_actual)
    X_aging_parts.append(X_aging_cur)

    barcode_parts.extend(adata_cur.obs_names.astype(str).tolist())

    if "age" in adata_cur.obs.columns:
        age_parts.extend(adata_cur.obs["age"].astype(str).tolist())
    else:
        age_parts.extend([np.nan] * adata_cur.n_obs)

    if "celltype" in adata_cur.obs.columns:
        celltype_parts.extend(adata_cur.obs["celltype"].astype(str).tolist())
    else:
        celltype_parts.extend([np.nan] * adata_cur.n_obs)

    slice_parts.extend([slice_index] * adata_cur.n_obs)

X_aging_all = np.vstack(X_aging_parts).astype(float, copy=False)


# -----------------------------
# 5. Global z-score per selected good aging gene, then mean per cell
# -----------------------------
gene_mean = np.nanmean(X_aging_all, axis=0)
gene_std = np.nanstd(X_aging_all, axis=0, ddof=1)
gene_std[(~np.isfinite(gene_std)) | (gene_std == 0)] = np.nan

X_aging_global_z = (X_aging_all - gene_mean) / gene_std
X_aging_global_z = np.nan_to_num(
    X_aging_global_z,
    nan=0.0,
    posinf=0.0,
    neginf=0.0,
)

exp_nor_all_aginggenes_norm_mean = np.mean(X_aging_global_z, axis=1)
exp_nor_all_aginggenes_norm_mean = np.asarray(
    exp_nor_all_aginggenes_norm_mean,
    dtype=float,
)

print("Good-aging-gene module score vector shape:", exp_nor_all_aginggenes_norm_mean.shape)
print(
    "Good-aging-gene module score summary:",
    {
        "mean": float(np.nanmean(exp_nor_all_aginggenes_norm_mean)),
        "std": float(np.nanstd(exp_nor_all_aginggenes_norm_mean, ddof=1)),
        "min": float(np.nanmin(exp_nor_all_aginggenes_norm_mean)),
        "max": float(np.nanmax(exp_nor_all_aginggenes_norm_mean)),
    }
)


# -----------------------------
# 6. Save outputs for reproducibility and backward compatibility
# -----------------------------
aging_module_score_df = pd.DataFrame({
    "barcode": barcode_parts,
    "slice_index": slice_parts,
    "age": age_parts,
    "celltype": celltype_parts,
    "AgingGeneExp": exp_nor_all_aginggenes_norm_mean,
    "score_method": MODULE_SCORE_METHOD,
    "gene_filter": f"log2FC_old_vs_young_cell_mean>{GOOD_AGING_GENE_CELL_LFC_THRESHOLD}",
})
aging_module_score_df.index = barcode_parts

aging_gene_used_df = pd.DataFrame({
    "requested_gene": aging_genes_used_requested,
    "actual_gene": aging_genes_actual,
    "gene_mean_global": gene_mean,
    "gene_std_global": gene_std,
})

aging_gene_used_df = aging_gene_used_df.merge(
    good_lfc_keep_df[
        [
            "actual_gene",
            "mean_young_cell_expression",
            "mean_old_cell_expression",
            "log2FC_old_vs_young_cell_mean",
            "mean_young_slice_mean_expression",
            "mean_old_slice_mean_expression",
            "log2FC_old_vs_young_slice_mean",
        ]
    ],
    on="actual_gene",
    how="left",
)

aging_module_score_path = OUTPUT_ROOT / "aging_module_score_good_aging_genes_cellLFC_gt0p2_zscore_mean_global.csv"
aging_gene_used_path = OUTPUT_ROOT / "aging_module_score_good_aging_genes_cellLFC_gt0p2_zscore_mean_global_genes_used.csv"

aging_module_score_df.to_csv(output_path(aging_module_score_path))
aging_gene_used_df.to_csv(output_path(aging_gene_used_path), index=False)

# Keep the original filename as a compatibility output, but now with the updated filtered-gene score.
pd.DataFrame(
    {"AgingGeneExp": exp_nor_all_aginggenes_norm_mean},
    index=barcode_parts,
).to_csv(output_path(OUTPUT_ROOT / "aging_module_score.csv"))

print(f"Saved good-aging-gene module scores to: {aging_module_score_path}")
print(f"Saved good-aging-gene list to: {aging_gene_used_path}")

display(aging_module_score_df.head())
display(aging_gene_used_df.sort_values("log2FC_old_vs_young_cell_mean", ascending=False))

In [ ]:
from torch_scatter import scatter_sum
##Calculate the mean receiving MI-29 stength from T cells
mean_receiver_MI29_fromTcell_Aging_list = []
startindex = 0
for batch_index in range(len(processed.spidernet_data)):
    SpiderNet_data_pyg_cur = processed.spidernet_data[batch_index]
    edge_index_cur = SpiderNet_data_pyg_cur['edge_index'].cpu().numpy()
    edge_index_cur_torch = torch.tensor(edge_index_cur, dtype=torch.long, device=device)
    celltype_all_cur = np.array(processed.adata_list[batch_index].obs['celltype'])
    num_cell_cur = SpiderNet_data_pyg_cur.x.shape[0]
    Factor_envir_cur = results['factor_envir_list'][batch_index].copy()
    age_cur = np.unique(np.array(processed.adata_list[batch_index].obs['age']))[0]
    ##Update the startindex
    if batch_index > 0:
        startindex += processed.spidernet_data[batch_index-1].x.shape[0]
    endindex = startindex + processed.spidernet_data[batch_index].x.shape[0]
    exp_nor_all_aginggenes_norm_mean_cur = exp_nor_all_aginggenes_norm_mean[startindex:endindex]
    ##
    edge_index_receivefrom_Tcell = np.where(celltype_all_cur[edge_index_cur[:,0]] == "T cell")[0]
    # edge_index_receivefrom_Tcell = np.array(range(edge_index_cur.shape[0]))
    ##Set the Factor_envir_cur's rows out of the edge_index_receivefrom_Tcell to 0
    Factor_envir_cur_Tcellonly = np.zeros(Factor_envir_cur.shape) * np.nan
    Factor_envir_cur_Tcellonly[edge_index_receivefrom_Tcell,:] = Factor_envir_cur[edge_index_receivefrom_Tcell,:]
    # print(Factor_envir_cur_Tcellonly.shape)
    # print(np.sum(Factor_envir_cur_Tcellonly))
    # print(Factor_envir_cur_Tcellonly)
    Factor_envir_cur_Tcellonly = torch.tensor(Factor_envir_cur_Tcellonly, dtype=torch.float32, device=device)
    # mask of valid (non-NaN) elements
    valid_mask = ~torch.isnan(Factor_envir_cur_Tcellonly)

    # Replace NaN with 0 for summing
    values_no_nan = torch.nan_to_num(Factor_envir_cur_Tcellonly, nan=0.0)
    # 1. sum of values ignoring NaN
    sum_values = scatter_sum(
        values_no_nan,
        # edge_index_cur_torch[:, 1].to(torch.int64),
        edge_index_cur_torch[:, 0].to(torch.int64),
        dim=0,
        dim_size=num_cell_cur
    )

    # 2. count of valid entries
    count_values = scatter_sum(
        valid_mask.float(),
        # edge_index_cur_torch[:, 1].to(torch.int64),
        edge_index_cur_torch[:, 0].to(torch.int64),
        dim=0,
        dim_size=num_cell_cur
    )

    # 3. compute mean: sum / count (where count=0 → NaN)
    # mean_values = sum_values / count_values
    mean_values = sum_values
    mean_values[count_values == 0] = float('nan')

    Factor_envir_cur_Tcellonly_receiver_agg = mean_values.cpu().numpy()
    # Factor_envir_cur_Tcellonly_receiver_agg = scatter_nanmax(Factor_envir_cur_Tcellonly,
    #                                      edge_index_cur_torch[:, 1].to(torch.int64), dim=0,
    #                                      dim_size=num_cell_cur).to("cpu").numpy()
    # print(Factor_envir_cur_Tcellonly_receiver_agg)
    # print(np.nanmean(Factor_envir_cur_Tcellonly_receiver_agg))
    ##Set the nan in Factor_envir_cur_Tcellonly_receiver_agg to zero
    Factor_envir_cur_Tcellonly_receiver_agg_nonan = np.nan_to_num(Factor_envir_cur_Tcellonly_receiver_agg, nan=0.0)
    # print(Factor_envir_cur_Tcellonly_receiver_agg_nonan.shape)
    Factor_envir_cur_Tcellonly_receiver_agg_nonan_MI29 = Factor_envir_cur_Tcellonly_receiver_agg_nonan[:,28]
    #
    cellindex_Factor_envir_cur_Tcellonly_receiver_agg_nonan_MI29_nonzero = np.where(Factor_envir_cur_Tcellonly_receiver_agg_nonan_MI29 > 0)[0]
    Factor_envir_cur_Tcellonly_receiver_agg_nonan_MI29_choose = Factor_envir_cur_Tcellonly_receiver_agg_nonan_MI29[cellindex_Factor_envir_cur_Tcellonly_receiver_agg_nonan_MI29_nonzero]
    exp_nor_all_aginggenes_norm_mean_cur_choose = exp_nor_all_aginggenes_norm_mean_cur[cellindex_Factor_envir_cur_Tcellonly_receiver_agg_nonan_MI29_nonzero]
    exp_nor_all_aginggenes_norm_mean_cur_receivermean = []
    for cellindex_cur in cellindex_Factor_envir_cur_Tcellonly_receiver_agg_nonan_MI29_nonzero:
        receiver_cur = edge_index_cur[edge_index_cur[:,0] == cellindex_cur,1]
        exp_nor_all_aginggenes_norm_mean_cur_receivercur = exp_nor_all_aginggenes_norm_mean_cur[receiver_cur]
        exp_nor_all_aginggenes_norm_mean_cur_receivermean.append(np.mean(exp_nor_all_aginggenes_norm_mean_cur_receivercur))
    df_cur = pd.DataFrame({'Mean_Receiver_MI29_fromTcell': Factor_envir_cur_Tcellonly_receiver_agg_nonan_MI29_choose,
                           'AgingGeneExp': exp_nor_all_aginggenes_norm_mean_cur_choose,
                           'Age': age_cur,
                           'AgingGeneExp_neighbor': exp_nor_all_aginggenes_norm_mean_cur_receivermean})
    mean_receiver_MI29_fromTcell_Aging_list.append(df_cur)
mean_receiver_MI29_fromTcell_Aging_all = pd.concat(mean_receiver_MI29_fromTcell_Aging_list)
    

In [ ]:
## Group cells by receiver-side MI-29 input from T cells
mean_receiver_MI29_fromTcell_Aging_all['Mean_Receiver_MI29_fromTcell_group'] = pd.qcut(mean_receiver_MI29_fromTcell_Aging_all['Mean_Receiver_MI29_fromTcell'],
                                                                                       # q=3, labels=['Low', 'Medium', 'High'])
                                                                                       q=2, labels=['Low', 'High'])

mean_receiver_MI29_fromTcell_Aging_all.to_csv(output_path(run_dirs["run_dir"] / "Tcell_MI29_aging_plot_data.csv"), index=False)


In [ ]:
from scipy.stats import mannwhitneyu
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def _compute_smd(high_vals, low_vals):
    """Standardized mean difference: (High - Low) / pooled SD."""
    high_vals = pd.to_numeric(pd.Series(high_vals), errors="coerce").dropna().to_numpy(dtype=float)
    low_vals = pd.to_numeric(pd.Series(low_vals), errors="coerce").dropna().to_numpy(dtype=float)

    if len(high_vals) == 0 or len(low_vals) == 0:
        return np.nan

    if len(high_vals) > 1 and len(low_vals) > 1:
        pooled_var = (
            (len(high_vals) - 1) * np.var(high_vals, ddof=1)
            + (len(low_vals) - 1) * np.var(low_vals, ddof=1)
        ) / (len(high_vals) + len(low_vals) - 2)
        pooled_sd = np.sqrt(pooled_var)
    else:
        pooled_sd = np.nan

    if not np.isfinite(pooled_sd) or pooled_sd == 0:
        return np.nan

    return (np.mean(high_vals) - np.mean(low_vals)) / pooled_sd


def _p_label(p):
    if not np.isfinite(p):
        return "p = NA"
    if p < 1e-4:
        return "p < 1e-4"
    return f"p = {p:.2e}"


def _style_boxplot_edges_and_medians(ax, order, color_map):
    """
    Make each box's fill color and edge color match group color.
    Make median lines white.
    Remove upper/lower cap horizontal lines.
    """
    # Box fill and edge colors
    for patch, group in zip(ax.patches, order):
        patch.set_facecolor(color_map[group])
        patch.set_edgecolor(color_map[group])
        patch.set_linewidth(1.4)

    # Matplotlib/seaborn boxplot line order per box:
    # whisker1, whisker2, cap1, cap2, median, flier
    lines_per_box = 6

    for i, group in enumerate(order):
        group_color = color_map[group]
        start = i * lines_per_box

        for k in range(start, min(start + lines_per_box, len(ax.lines))):
            line = ax.lines[k]

            # Remove cap lines: upper/lower horizontal whisker caps
            if k in [start + 2, start + 3]:
                line.set_visible(False)
                line.set_linewidth(0)

            # Median line
            elif k == start + 4:
                line.set_color("white")
                line.set_linewidth(1.8)

            # Fliers
            elif k == start + 5:
                line.set_markerfacecolor("black")
                line.set_markeredgecolor("black")
                line.set_color("black")

            # Vertical whisker lines
            else:
                line.set_color(group_color)
                line.set_linewidth(1.4)


# --------------------------------------------------------------
# Compute Wilcoxon rank-sum / Mann–Whitney U p-value
# --------------------------------------------------------------
group_order = ["High", "Low"]

color_map = {
    "High": "#3d91cf",
    "Low": "#83b7de",
}

low_vals = pd.to_numeric(
    mean_receiver_MI29_fromTcell_Aging_all.loc[
        mean_receiver_MI29_fromTcell_Aging_all["Mean_Receiver_MI29_fromTcell_group"] == "Low",
        "AgingGeneExp",
    ],
    errors="coerce",
).dropna()

high_vals = pd.to_numeric(
    mean_receiver_MI29_fromTcell_Aging_all.loc[
        mean_receiver_MI29_fromTcell_Aging_all["Mean_Receiver_MI29_fromTcell_group"] == "High",
        "AgingGeneExp",
    ],
    errors="coerce",
).dropna()

if len(low_vals) == 0 or len(high_vals) == 0:
    stat, pval = np.nan, np.nan
else:
    stat, pval = mannwhitneyu(low_vals, high_vals, alternative="two-sided")

smd = _compute_smd(high_vals, low_vals)

aging_boxplot_stats = pd.DataFrame([{
    "score": "AgingGeneExp",
    "group_high": "High",
    "group_low": "Low",
    "n_high": len(high_vals),
    "n_low": len(low_vals),
    "mean_high": float(np.mean(high_vals)) if len(high_vals) > 0 else np.nan,
    "mean_low": float(np.mean(low_vals)) if len(low_vals) > 0 else np.nan,
    "smd_high_minus_low": smd,
    "mannwhitneyu_stat": stat,
    "pvalue": pval,
}])

aging_boxplot_stats.to_csv(
    output_path(str(run_dirs["run_dir"]) + "/AgingGeneExp_by_MeanReceiverMI29_fromTcell_Group_Boxplot_stats.csv"),
    index=False,
)

display(aging_boxplot_stats)


# --------------------------------------------------------------
# Plot
# --------------------------------------------------------------
plt.figure(figsize=(3, 4))
ax = plt.gca()

sns.boxplot(
    data=mean_receiver_MI29_fromTcell_Aging_all,
    x="Mean_Receiver_MI29_fromTcell_group",
    y="AgingGeneExp",
    order=group_order,
    palette=color_map,
    width=0.7,
    linewidth=1.4,
    medianprops=dict(color="white", linewidth=1.8),
    capprops=dict(linewidth=0),
    flierprops=dict(
        marker="o",
        markersize=3,
        markerfacecolor="black",
        markeredgecolor="black",
        linestyle="none",
    ),
    ax=ax,
)

_style_boxplot_edges_and_medians(ax, group_order, color_map)

plt.xlabel("T cell group \n(Sending MI-29)", fontsize=16)
plt.ylabel("Aging module score", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

ax.text(
    0.5,
    0.98,
    f"{_p_label(pval)}\nSMD = {smd:.2f}",
    ha="center",
    va="top",
    fontsize=13,
    transform=ax.transAxes,
)

sns.despine(top=True, right=True)

plt.tight_layout()

plt.savefig(
    output_path(str(run_dirs["run_dir"]) + "/AgingGeneExp_by_MeanReceiverMI29_fromTcell_Group_Boxplot.pdf"),
    dpi=300,
)

plt.show()
plt.close()

In [ ]:
from scipy.stats import mannwhitneyu
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns







# --------------------------------------------------------------
# Compute Wilcoxon rank-sum / Mann–Whitney U p-value
# --------------------------------------------------------------
group_order = ["High", "Low"]

color_map = {
    "High": "#e93732",
    "Low": "#d9adac",
}

low_vals = pd.to_numeric(
    mean_receiver_MI29_fromTcell_Aging_all.loc[
        mean_receiver_MI29_fromTcell_Aging_all["Mean_Receiver_MI29_fromTcell_group"] == "Low",
        "AgingGeneExp_neighbor",
    ],
    errors="coerce",
).dropna()

high_vals = pd.to_numeric(
    mean_receiver_MI29_fromTcell_Aging_all.loc[
        mean_receiver_MI29_fromTcell_Aging_all["Mean_Receiver_MI29_fromTcell_group"] == "High",
        "AgingGeneExp_neighbor",
    ],
    errors="coerce",
).dropna()

if len(low_vals) == 0 or len(high_vals) == 0:
    stat, pval = np.nan, np.nan
else:
    stat, pval = mannwhitneyu(low_vals, high_vals, alternative="two-sided")

smd = _compute_smd(high_vals, low_vals)

aging_neighbor_boxplot_stats = pd.DataFrame([{
    "score": "AgingGeneExp_neighbor",
    "group_high": "High",
    "group_low": "Low",
    "n_high": len(high_vals),
    "n_low": len(low_vals),
    "mean_high": float(np.mean(high_vals)) if len(high_vals) > 0 else np.nan,
    "mean_low": float(np.mean(low_vals)) if len(low_vals) > 0 else np.nan,
    "smd_high_minus_low": smd,
    "mannwhitneyu_stat": stat,
    "pvalue": pval,
}])

aging_neighbor_boxplot_stats.to_csv(
    output_path(str(run_dirs["run_dir"]) + "/AgingGeneExp_neighbor_by_MeanReceiverMI29_fromTcell_Group_Boxplot_stats.csv"),
    index=False,
)

display(aging_neighbor_boxplot_stats)


# --------------------------------------------------------------
# Plot
# --------------------------------------------------------------
plt.figure(figsize=(3, 4))
ax = plt.gca()

sns.boxplot(
    data=mean_receiver_MI29_fromTcell_Aging_all,
    x="Mean_Receiver_MI29_fromTcell_group",
    y="AgingGeneExp_neighbor",
    order=group_order,
    palette=color_map,
    width=0.7,
    linewidth=1.4,
    medianprops=dict(color="white", linewidth=1.8),
    capprops=dict(linewidth=0),
    flierprops=dict(
        marker="o",
        markersize=3,
        markerfacecolor="black",
        markeredgecolor="black",
        linestyle="none",
    ),
    ax=ax,
)

_style_boxplot_edges_and_medians(ax, group_order, color_map)

plt.xlabel("Neighbors of T Cells\n(by MI-29 Sending group)", fontsize=16)
plt.ylabel("Aging module score", fontsize=16)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

ax.text(
    0.5,
    0.98,
    f"{_p_label(pval)}\nSMD = {smd:.2f}",
    ha="center",
    va="top",
    fontsize=13,
    transform=ax.transAxes,
)

sns.despine(top=True, right=True)

plt.tight_layout()

plt.savefig(
    output_path(str(run_dirs["run_dir"]) + "/AgingGeneExp_neighbor_by_MeanReceiverMI29_fromTcell_Group_Boxplot.pdf"),
    dpi=300,
)

plt.show()
plt.close()

In [ ]:
# ==============================================================
# Receiver-cell-type-specific version of the neighbor Aging score boxplot
# --------------------------------------------------------------
# Extension of:
#   AgingGeneExp_neighbor_by_MeanReceiverMI29_fromTcell_Group_Boxplot
#
# Logic:
#   1. Reconstruct the same T-cell MI-29 sending group as above:
#        - sender is T cell
#        - aggregate outgoing MI-29 by SUM over outgoing T cell -> neighbor edges
#        - keep T cells with MI-29 sum > 0
#        - split T cells into Low / High by pd.qcut(q=2)
#   2. For each T cell and each receiver cell type:
#        - collect outgoing neighbors of that receiver cell type
#        - compute mean Aging module score among those receiver cells
#   3. Plot High vs Low boxplot separately for each receiver cell type.
# ==============================================================

from pathlib import Path
import re
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu


# -----------------------------
# Settings
# -----------------------------
group_order = ["High", "Low"]

receiver_color_map = {
    "High": "#e93732",
    "Low": "#d9adac",
}

MIN_TCELLS_PER_GROUP_PER_RECEIVER_TYPE = 3
MAX_RECEIVER_TYPES_TO_PLOT = None  # set to an integer, e.g. 12, if too many panels

receiver_ct_outdir = Path(str(run_dirs["run_dir"])) / "ReceiverCelltype_AgingGeneExp_neighbor_by_Tcell_MI29_Group"
receiver_ct_outdir.mkdir(parents=True, exist_ok=True)


def _safe_filename(x):
    x = str(x)
    x = re.sub(r"[^\w\-.]+", "_", x)
    x = re.sub(r"_+", "_", x).strip("_")
    return x if len(x) > 0 else "NA"


def _compute_smd(high_vals, low_vals):
    """Standardized mean difference: (High - Low) / pooled SD."""
    high_vals = pd.to_numeric(pd.Series(high_vals), errors="coerce").dropna().to_numpy(dtype=float)
    low_vals = pd.to_numeric(pd.Series(low_vals), errors="coerce").dropna().to_numpy(dtype=float)

    if len(high_vals) == 0 or len(low_vals) == 0:
        return np.nan

    if len(high_vals) > 1 and len(low_vals) > 1:
        pooled_var = (
            (len(high_vals) - 1) * np.var(high_vals, ddof=1)
            + (len(low_vals) - 1) * np.var(low_vals, ddof=1)
        ) / (len(high_vals) + len(low_vals) - 2)
        pooled_sd = np.sqrt(pooled_var)
    else:
        pooled_sd = np.nan

    if not np.isfinite(pooled_sd) or pooled_sd == 0:
        return np.nan

    return (np.mean(high_vals) - np.mean(low_vals)) / pooled_sd


def _p_label(p):
    if not np.isfinite(p):
        return "p = NA"
    if p < 1e-4:
        return "p < 1e-4"
    return f"p = {p:.2e}"


def _style_boxplot_edges_and_medians(ax, order, color_map):
    """
    Make box fill and edge colors match group colors.
    Make median lines white.
    Remove upper/lower cap horizontal lines.
    """
    for patch, group in zip(ax.patches, order):
        patch.set_facecolor(color_map[group])
        patch.set_edgecolor(color_map[group])
        patch.set_linewidth(1.4)

    lines_per_box = 6
    for i, group in enumerate(order):
        group_color = color_map[group]
        start = i * lines_per_box

        for k in range(start, min(start + lines_per_box, len(ax.lines))):
            line = ax.lines[k]

            if k in [start + 2, start + 3]:
                line.set_visible(False)
                line.set_linewidth(0)
            elif k == start + 4:
                line.set_color("white")
                line.set_linewidth(1.8)
            elif k == start + 5:
                line.set_markerfacecolor("black")
                line.set_markeredgecolor("black")
                line.set_color("black")
            else:
                line.set_color(group_color)
                line.set_linewidth(1.4)


# -----------------------------
# 1. Build T-cell x receiver-cell-type table
# -----------------------------
receiver_ct_rows = []

startindex = 0

for batch_index in range(len(processed.spidernet_data)):
    SpiderNet_data_pyg_cur = processed.spidernet_data[batch_index]

    edge_index_cur = SpiderNet_data_pyg_cur["edge_index"]
    if hasattr(edge_index_cur, "detach"):
        edge_index_cur = edge_index_cur.detach().cpu().numpy()
    else:
        edge_index_cur = np.asarray(edge_index_cur)

    # Support either [n_edges, 2] or PyG-style [2, n_edges]
    if edge_index_cur.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got shape {edge_index_cur.shape}")
    if edge_index_cur.shape[0] == 2 and edge_index_cur.shape[1] != 2:
        edge_index_cur = edge_index_cur.T
    if edge_index_cur.shape[1] != 2:
        raise ValueError(f"edge_index must have two columns after conversion, got shape {edge_index_cur.shape}")

    edge_index_cur = edge_index_cur.astype(int, copy=False)

    adata_cur = processed.adata_list[batch_index]
    celltype_all_cur = np.asarray(adata_cur.obs["celltype"].astype(str))
    barcode_all_cur = np.asarray(adata_cur.obs_names.astype(str))

    num_cell_cur = SpiderNet_data_pyg_cur.x.shape[0]

    Factor_envir_cur = results["factor_envir_list"][batch_index].copy()
    Factor_envir_cur = np.asarray(Factor_envir_cur, dtype=float)

    if Factor_envir_cur.shape[1] <= 28:
        raise IndexError(
            f"MI-29 requires column index 28, but factor_envir has shape {Factor_envir_cur.shape}"
        )

    if batch_index > 0:
        startindex += processed.spidernet_data[batch_index - 1].x.shape[0]
    endindex = startindex + processed.spidernet_data[batch_index].x.shape[0]

    exp_nor_all_aginggenes_norm_mean_cur = np.asarray(
        exp_nor_all_aginggenes_norm_mean[startindex:endindex],
        dtype=float,
    )

    age_values = np.unique(np.asarray(adata_cur.obs["age"]))
    age_cur = age_values[0] if len(age_values) > 0 else np.nan

    # Only outgoing edges whose sender is T cell
    edge_is_from_tcell = celltype_all_cur[edge_index_cur[:, 0]] == "T cell"

    if np.sum(edge_is_from_tcell) == 0:
        continue

    edge_df_cur = pd.DataFrame({
        "batch_index": batch_index,
        "sender_index": edge_index_cur[edge_is_from_tcell, 0].astype(int),
        "receiver_index": edge_index_cur[edge_is_from_tcell, 1].astype(int),
        "MI29_strength": Factor_envir_cur[edge_is_from_tcell, 28].astype(float),
    })

    # Valid receiver only
    edge_df_cur = edge_df_cur[
        (edge_df_cur["receiver_index"] >= 0)
        & (edge_df_cur["receiver_index"] < num_cell_cur)
    ].copy()

    if edge_df_cur.shape[0] == 0:
        continue

    # This matches the original notebook:
    # mean_values = sum_values, then MI29 > 0, then pd.qcut(q=2)
    sender_mi29_sum = (
        edge_df_cur
        .groupby("sender_index", sort=False)["MI29_strength"]
        .sum()
        .rename("Mean_Receiver_MI29_fromTcell")
        .reset_index()
    )

    sender_mi29_sum = sender_mi29_sum[
        np.isfinite(sender_mi29_sum["Mean_Receiver_MI29_fromTcell"])
        & (sender_mi29_sum["Mean_Receiver_MI29_fromTcell"] > 0)
    ].copy()

    if sender_mi29_sum.shape[0] == 0:
        continue

    edge_df_cur = edge_df_cur[
        edge_df_cur["sender_index"].isin(sender_mi29_sum["sender_index"])
    ].copy()

    if edge_df_cur.shape[0] == 0:
        continue

    edge_df_cur["sender_barcode"] = barcode_all_cur[edge_df_cur["sender_index"].to_numpy(dtype=int)]
    edge_df_cur["receiver_barcode"] = barcode_all_cur[edge_df_cur["receiver_index"].to_numpy(dtype=int)]
    edge_df_cur["receiver_celltype"] = celltype_all_cur[edge_df_cur["receiver_index"].to_numpy(dtype=int)]
    edge_df_cur["AgingGeneExp_receiver"] = exp_nor_all_aginggenes_norm_mean_cur[
        edge_df_cur["receiver_index"].to_numpy(dtype=int)
    ]

    edge_df_cur = edge_df_cur.merge(
        sender_mi29_sum,
        on="sender_index",
        how="left",
    )

    edge_df_cur["Age"] = age_cur

    # One point = one T cell x one receiver cell type
    # Value = mean Aging module score of receiver cells of that type around this T cell
    sender_receiver_ct_df_cur = (
        edge_df_cur
        .groupby(
            [
                "batch_index",
                "Age",
                "sender_index",
                "sender_barcode",
                "Mean_Receiver_MI29_fromTcell",
                "receiver_celltype",
            ],
            sort=False,
            observed=True,
        )
        .agg(
            AgingGeneExp_neighbor_receiver_celltype=("AgingGeneExp_receiver", "mean"),
            AgingGeneExp_neighbor_receiver_celltype_median=("AgingGeneExp_receiver", "median"),
            n_receiver_edges_this_type=("AgingGeneExp_receiver", "size"),
            n_unique_receiver_cells_this_type=("receiver_barcode", "nunique"),
        )
        .reset_index()
    )

    receiver_ct_rows.append(sender_receiver_ct_df_cur)


if len(receiver_ct_rows) == 0:
    raise ValueError("No T cell -> receiver-cell edges found for receiver-cell-type-specific analysis.")

receiver_ct_neighbor_aging_df = pd.concat(receiver_ct_rows, axis=0, ignore_index=True)

receiver_ct_neighbor_aging_df = receiver_ct_neighbor_aging_df[
    np.isfinite(receiver_ct_neighbor_aging_df["Mean_Receiver_MI29_fromTcell"])
    & np.isfinite(receiver_ct_neighbor_aging_df["AgingGeneExp_neighbor_receiver_celltype"])
].copy()


# -----------------------------
# 2. Assign the same global T-cell MI-29 High/Low group
# -----------------------------
# Important:
#   Grouping is based on each T cell's summed outgoing MI-29.
#   The same T cell can appear in multiple receiver-cell-type rows.
#   Therefore qcut should be assigned once per T cell, then merged back.

tcell_group_df = (
    receiver_ct_neighbor_aging_df[
        [
            "batch_index",
            "sender_index",
            "sender_barcode",
            "Mean_Receiver_MI29_fromTcell",
        ]
    ]
    .drop_duplicates()
    .copy()
)

try:
    tcell_group_df["Mean_Receiver_MI29_fromTcell_group"] = pd.qcut(
        tcell_group_df["Mean_Receiver_MI29_fromTcell"],
        q=2,
        labels=["Low", "High"],
        duplicates="drop",
    )

    if tcell_group_df["Mean_Receiver_MI29_fromTcell_group"].nunique(dropna=True) < 2:
        raise ValueError("qcut produced fewer than two groups.")

except Exception:
    threshold = float(np.nanmedian(tcell_group_df["Mean_Receiver_MI29_fromTcell"]))
    tcell_group_df["Mean_Receiver_MI29_fromTcell_group"] = np.where(
        tcell_group_df["Mean_Receiver_MI29_fromTcell"] > threshold,
        "High",
        "Low",
    )

receiver_ct_neighbor_aging_df = receiver_ct_neighbor_aging_df.drop(
    columns=["Mean_Receiver_MI29_fromTcell_group"],
    errors="ignore",
).merge(
    tcell_group_df[
        [
            "batch_index",
            "sender_index",
            "Mean_Receiver_MI29_fromTcell_group",
        ]
    ],
    on=["batch_index", "sender_index"],
    how="left",
)

receiver_ct_neighbor_aging_df["Mean_Receiver_MI29_fromTcell_group"] = pd.Categorical(
    receiver_ct_neighbor_aging_df["Mean_Receiver_MI29_fromTcell_group"],
    categories=group_order,
    ordered=True,
)

receiver_ct_table_path = (
    receiver_ct_outdir
    / "AgingGeneExp_neighbor_by_MeanReceiverMI29_fromTcell_Group_receiver_celltype_table.csv"
)
receiver_ct_neighbor_aging_df.to_csv(output_path(receiver_ct_table_path), index=False)
print(f"Saved receiver-cell-type table to: {receiver_ct_table_path}")

display(receiver_ct_neighbor_aging_df.head())


# -----------------------------
# 3. Stats per receiver cell type
# -----------------------------
receiver_ct_stats_rows = []

for receiver_celltype, df_ct in receiver_ct_neighbor_aging_df.groupby("receiver_celltype", sort=False):
    df_ct = df_ct[
        df_ct["Mean_Receiver_MI29_fromTcell_group"].isin(group_order)
        & np.isfinite(pd.to_numeric(
            df_ct["AgingGeneExp_neighbor_receiver_celltype"],
            errors="coerce",
        ))
    ].copy()

    low_vals = pd.to_numeric(
        df_ct.loc[
            df_ct["Mean_Receiver_MI29_fromTcell_group"] == "Low",
            "AgingGeneExp_neighbor_receiver_celltype",
        ],
        errors="coerce",
    ).dropna()

    high_vals = pd.to_numeric(
        df_ct.loc[
            df_ct["Mean_Receiver_MI29_fromTcell_group"] == "High",
            "AgingGeneExp_neighbor_receiver_celltype",
        ],
        errors="coerce",
    ).dropna()

    if len(low_vals) == 0 or len(high_vals) == 0:
        stat, pval = np.nan, np.nan
    else:
        stat, pval = mannwhitneyu(low_vals, high_vals, alternative="two-sided")

    smd = _compute_smd(high_vals, low_vals)

    receiver_ct_stats_rows.append({
        "score": "AgingGeneExp_neighbor_receiver_celltype",
        "receiver_celltype": receiver_celltype,
        "group_high": "High",
        "group_low": "Low",
        "n_high": int(len(high_vals)),
        "n_low": int(len(low_vals)),
        "mean_high": float(np.mean(high_vals)) if len(high_vals) > 0 else np.nan,
        "mean_low": float(np.mean(low_vals)) if len(low_vals) > 0 else np.nan,
        "smd_high_minus_low": smd,
        "mannwhitneyu_stat": stat,
        "pvalue": pval,
    })

receiver_ct_stats = pd.DataFrame(receiver_ct_stats_rows)

receiver_ct_stats_path = (
    receiver_ct_outdir
    / "AgingGeneExp_neighbor_by_MeanReceiverMI29_fromTcell_Group_receiver_celltype_stats.csv"
)
receiver_ct_stats.to_csv(output_path(receiver_ct_stats_path), index=False)
print(f"Saved receiver-cell-type stats to: {receiver_ct_stats_path}")

display(receiver_ct_stats.sort_values("smd_high_minus_low", ascending=False))


# -----------------------------
# 4. Plot multi-panel receiver-cell-type boxplots
# -----------------------------
receiver_ct_stats_plot = receiver_ct_stats[
    (receiver_ct_stats["n_high"] >= MIN_TCELLS_PER_GROUP_PER_RECEIVER_TYPE)
    & (receiver_ct_stats["n_low"] >= MIN_TCELLS_PER_GROUP_PER_RECEIVER_TYPE)
].copy()

receiver_ct_stats_plot = receiver_ct_stats_plot.sort_values(
    ["smd_high_minus_low", "receiver_celltype"],
    ascending=[False, True],
)

if MAX_RECEIVER_TYPES_TO_PLOT is not None:
    receiver_ct_stats_plot = receiver_ct_stats_plot.head(int(MAX_RECEIVER_TYPES_TO_PLOT)).copy()

receiver_celltype_order = receiver_ct_stats_plot["receiver_celltype"].astype(str).tolist()

if len(receiver_celltype_order) == 0:
    print(
        "[Warning] No receiver cell type has enough T cells in both groups. "
        f"Current minimum per group = {MIN_TCELLS_PER_GROUP_PER_RECEIVER_TYPE}."
    )
else:
    plot_df = receiver_ct_neighbor_aging_df[
        receiver_ct_neighbor_aging_df["receiver_celltype"].astype(str).isin(receiver_celltype_order)
        & receiver_ct_neighbor_aging_df["Mean_Receiver_MI29_fromTcell_group"].isin(group_order)
        & np.isfinite(pd.to_numeric(
            receiver_ct_neighbor_aging_df["AgingGeneExp_neighbor_receiver_celltype"],
            errors="coerce",
        ))
    ].copy()

    n_panels = len(receiver_celltype_order)
    n_cols = min(4, n_panels)
    n_rows = int(math.ceil(n_panels / n_cols))

    fig_width = max(3.2 * n_cols, 4.0)
    fig_height = max(3.7 * n_rows, 3.8)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(fig_width, fig_height),
        squeeze=False,
        sharey=False,
    )

    axes_flat = axes.ravel()
    stats_lookup = receiver_ct_stats_plot.set_index("receiver_celltype")

    for ax_i, receiver_celltype in enumerate(receiver_celltype_order):
        ax = axes_flat[ax_i]

        df_ct = plot_df[
            plot_df["receiver_celltype"].astype(str) == str(receiver_celltype)
        ].copy()

        sns.boxplot(
            data=df_ct,
            x="Mean_Receiver_MI29_fromTcell_group",
            y="AgingGeneExp_neighbor_receiver_celltype",
            order=group_order,
            palette=receiver_color_map,
            width=0.7,
            linewidth=1.2,
            medianprops=dict(color="white", linewidth=1.6),
            capprops=dict(linewidth=0),
            flierprops=dict(
                marker="o",
                markersize=2.5,
                markerfacecolor="black",
                markeredgecolor="black",
                linestyle="none",
            ),
            ax=ax,
        )

        _style_boxplot_edges_and_medians(ax, group_order, receiver_color_map)

        sns.stripplot(
            data=df_ct,
            x="Mean_Receiver_MI29_fromTcell_group",
            y="AgingGeneExp_neighbor_receiver_celltype",
            order=group_order,
            color="black",
            size=1.5,
            jitter=0.18,
            alpha=0.25,
            ax=ax,
        )

        stat_row = stats_lookup.loc[receiver_celltype]
        pval = stat_row["pvalue"]
        smd = stat_row["smd_high_minus_low"]

        ax.text(
            0.5,
            0.98,
            f"{_p_label(pval)}\nSMD = {smd:.2f}",
            ha="center",
            va="top",
            fontsize=9.5,
            transform=ax.transAxes,
        )

        ax.set_title(str(receiver_celltype), fontsize=12)
        ax.set_xlabel("Neighbors of T Cells\n(by MI-29 group)", fontsize=10)
        ax.set_ylabel("Aging module score", fontsize=10)
        ax.tick_params(axis="both", labelsize=9)
        sns.despine(ax=ax, top=True, right=True)

    for j in range(n_panels, len(axes_flat)):
        axes_flat[j].axis("off")

    plt.tight_layout()

    combined_plot_path = (
        receiver_ct_outdir
        / "AgingGeneExp_neighbor_by_MeanReceiverMI29_fromTcell_Group_receiver_celltype_boxplots.pdf"
    )
    plt.savefig(output_path(combined_plot_path), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    print(f"Saved combined receiver-cell-type boxplots to: {combined_plot_path}")


# -----------------------------
# 5. Save one separate PDF per receiver cell type
# -----------------------------
separate_plot_paths = []

for receiver_celltype in receiver_celltype_order:
    df_ct = receiver_ct_neighbor_aging_df[
        (receiver_ct_neighbor_aging_df["receiver_celltype"].astype(str) == str(receiver_celltype))
        & receiver_ct_neighbor_aging_df["Mean_Receiver_MI29_fromTcell_group"].isin(group_order)
        & np.isfinite(pd.to_numeric(
            receiver_ct_neighbor_aging_df["AgingGeneExp_neighbor_receiver_celltype"],
            errors="coerce",
        ))
    ].copy()

    if df_ct.shape[0] == 0:
        continue

    stat_row = receiver_ct_stats_plot.set_index("receiver_celltype").loc[receiver_celltype]
    pval = stat_row["pvalue"]
    smd = stat_row["smd_high_minus_low"]

    plt.figure(figsize=(3.0, 4.0))
    ax = plt.gca()

    sns.boxplot(
        data=df_ct,
        x="Mean_Receiver_MI29_fromTcell_group",
        y="AgingGeneExp_neighbor_receiver_celltype",
        order=group_order,
        palette=receiver_color_map,
        width=0.7,
        linewidth=1.4,
        medianprops=dict(color="white", linewidth=1.8),
        capprops=dict(linewidth=0),
        flierprops=dict(
            marker="o",
            markersize=3,
            markerfacecolor="black",
            markeredgecolor="black",
            linestyle="none",
        ),
        ax=ax,
    )

    _style_boxplot_edges_and_medians(ax, group_order, receiver_color_map)

    sns.stripplot(
        data=df_ct,
        x="Mean_Receiver_MI29_fromTcell_group",
        y="AgingGeneExp_neighbor_receiver_celltype",
        order=group_order,
        color="black",
        size=1.7,
        jitter=0.18,
        alpha=0.28,
        ax=ax,
    )

    ax.text(
        0.5,
        0.98,
        f"{_p_label(pval)}\nSMD = {smd:.2f}",
        ha="center",
        va="top",
        fontsize=11,
        transform=ax.transAxes,
    )

    ax.set_title(str(receiver_celltype), fontsize=13)
    ax.set_xlabel("Neighbors of T Cells\n(by MI-29 group)", fontsize=12)
    ax.set_ylabel("Aging module score", fontsize=12)
    ax.tick_params(axis="both", labelsize=11)
    sns.despine(top=True, right=True)

    plt.tight_layout()

    separate_plot_path = (
        receiver_ct_outdir
        / f"AgingGeneExp_neighbor_by_MeanReceiverMI29_fromTcell_Group_receiver_{_safe_filename(receiver_celltype)}.pdf"
    )
    plt.savefig(output_path(separate_plot_path), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

    separate_plot_paths.append(separate_plot_path)

print(f"Saved {len(separate_plot_paths)} separate receiver-cell-type boxplots to: {receiver_ct_outdir}")

## Optional B7. Compare T-cell-sender aging associations across CCC methods

This section incorporates the complete analysis previously provided in `AgingBrain_TcellSender_CCC_SMD_comparison`. It applies to the same ten coronal sections used above, fixes SpiderNet to MI-29, scans the available dimensions of NMF-LR, COMMOT, and ScCChain, and compares two standardized mean-difference outcomes:

1. ageing-module scores of T cells grouped by outgoing T-cell-to-neighbor communication strength; and
2. ageing-module scores of receiver cells grouped independently by incoming T-cell-to-receiver communication strength.

The section reloads the exported SpiderNet and processed-data objects from disk so that the comparison exactly follows the standalone analysis and verifies its file dependencies explicitly. It also recomputes and saves the comparison-specific ageing-gene diagnostics and module-score tables. Spacia remains disabled by default because outputs are not available for this dataset. This additional CCC-method comparison is not explicitly shown in the current SpiderNet manuscript.


In [ ]:
# ============================================================
# B7.1. CCC comparison settings
# ============================================================

# Reuse the dataset and run paths configured above.
RUN_DIR = Path(run_dirs["run_dir"])

# SpiderNet is fixed to MI-29 to match the ageing-brain analysis.
SPIDERNET_FIXED_DIM_ONE_BASED = 29
SPIDERNET_FIXED_FEATURE_NAME = f"MI-{SPIDERNET_FIXED_DIM_ONE_BASED}"

# Baseline result directories used by the ageing mouse brain CCC benchmark.
COMMOT_PATH_MAIN = OUTPUT_ROOT / "COMMOT"
SC_CCHAIN_PATH_MAIN = OUTPUT_ROOT / "ScCChain"
SPACIA_PATH_MAIN = OUTPUT_ROOT / "Spacia"

# Spacia is disabled because outputs are generally unavailable for this dataset.
METHODS_TO_RUN = ["SpiderNet", "NMF-LR", "COMMOT", "ScCChain"]
INCLUDE_SPACIA = False
if INCLUDE_SPACIA and "Spacia" not in METHODS_TO_RUN:
    METHODS_TO_RUN.append("Spacia")

# Required methods stop the comparison if their inputs cannot be loaded.
REQUIRE_METHODS_TO_LOAD = ["SpiderNet", "ScCChain"]
SKIP_MISSING_BASELINES = True

# T-cell grouping filters.
SPIDERNET_TCELL_EVAL_FILTER = "positive_outgoing"
BASELINE_TCELL_EVAL_FILTER = "all_finite"

CELLTYPE_COL = "celltype"
AGE_COL = "age"
T_CELL_LABEL = "T cell"

# Aggregate edge-level communication features by summation.
T_CELL_OUTGOING_AGG = "sum"  # options: "sum", "mean", "max"
RECEIVER_INCOMING_AGG = T_CELL_OUTGOING_AGG

# Receiver-cell evaluation filters.
SPIDERNET_RECEIVER_EVAL_FILTER = "positive_incoming"
BASELINE_RECEIVER_EVAL_FILTER = "all_finite"

# Median-rank splitting avoids pathological group sizes when values tie at the median.
SPLIT_RULE = "median_rank_split_upper_half"
MIN_CELLS_PER_GROUP = 5

# Retained in output tables for compatibility with the standalone analysis.
AMBIGUOUS_RECEIVER_POLICY = "not_used_receiver_grouped_by_incoming_strength"

# NMF-LR is recomputed globally in memory with the original comparison settings.
NMF_RANDOM_STATE = 0
NMF_MAX_ITER = 1000
NMF_RANK = DIM_ENVIR

# SpiderNet is fixed to MI-29; baseline dimensions are selected by mean SMD rank.
FEATURE_SELECTION_MODE = "baselines_mean_SMD_rank_across_two_outcomes__SpiderNet_fixed_MI29"
BASELINE_FEATURE_SELECTION_MODE = "mean_SMD_rank_across_two_outcomes"
REQUIRE_BOTH_OUTCOMES_FOR_SELECTION = True

OUT_DIR = RUN_DIR / "Aging_Tcell_sender_CCC_method_SMD_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT)
print("PROCESSED_DATA_DIR:", PROCESSED_DATA_DIR)
print("RUN_DIR:", RUN_DIR)
print("OUT_DIR:", OUT_DIR)
print("METHODS_TO_RUN:", METHODS_TO_RUN)
print("SPIDERNET_FIXED_FEATURE_NAME:", SPIDERNET_FIXED_FEATURE_NAME)
print("REQUIRE_METHODS_TO_LOAD:", REQUIRE_METHODS_TO_LOAD)
print("SPIDERNET_TCELL_EVAL_FILTER:", SPIDERNET_TCELL_EVAL_FILTER)
print("BASELINE_TCELL_EVAL_FILTER:", BASELINE_TCELL_EVAL_FILTER)
print("RECEIVER_INCOMING_AGG:", RECEIVER_INCOMING_AGG)
print("SPIDERNET_RECEIVER_EVAL_FILTER:", SPIDERNET_RECEIVER_EVAL_FILTER)
print("BASELINE_RECEIVER_EVAL_FILTER:", BASELINE_RECEIVER_EVAL_FILTER)


In [ ]:
# ============================================================
# B7.2. Imports
# ============================================================
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
import torch
from matplotlib import rcParams
from scipy.stats import mannwhitneyu
from sklearn.decomposition import NMF

pd.set_option("display.max_columns", 200)


In [ ]:
# ============================================================
# B7.3. Helper functions
# ============================================================

def p_to_star(p):
    if pd.isna(p):
        return "NA"
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


def edge_index_to_e2(edge_index):
    """Return edge_index as an E x 2 numpy integer array."""
    if torch is not None and hasattr(edge_index, "detach"):
        arr = edge_index.detach().cpu().numpy()
    else:
        arr = np.asarray(edge_index)

    if arr.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got {arr.shape}")
    if arr.shape[1] == 2:
        out = arr
    elif arr.shape[0] == 2:
        out = arr.T
    else:
        raise ValueError(f"Cannot interpret edge_index shape as E x 2 or 2 x E: {arr.shape}")
    return out.astype(np.int64, copy=False)


def get_data_item(data, key):
    try:
        return data[key]
    except Exception:
        return getattr(data, key)


def get_num_cells_from_data(data, adata_sub):
    try:
        return int(get_data_item(data, "x").shape[0])
    except Exception:
        return int(adata_sub.n_obs)




def edge_matrix_from_square(square_mat, rows, cols):
    if sp.issparse(square_mat):
        square_mat = square_mat.tocsr()
        return square_mat[rows, cols].A1.astype(np.float32, copy=False)
    return np.asarray(square_mat)[rows, cols].astype(np.float32, copy=False)


def _clean_nonnegative_matrix_for_nmf(X):
    """Convert edge-level LR matrix to a non-negative matrix suitable for sklearn NMF."""
    if torch is not None and torch.is_tensor(X):
        X = X.detach().cpu().numpy()

    if sp.issparse(X):
        X = X.tocsr(copy=True)
        X.data = np.nan_to_num(X.data, nan=0.0, posinf=0.0, neginf=0.0)
        X.data[X.data < 0] = 0.0
        return X

    X = np.asarray(X, dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    X[X < 0] = 0.0
    return X


def build_global_nmflr_factors(spidernet_data_list, n_components, random_state=0, max_iter=1000):
    """Fit one shared NMF-LR basis after concatenating all slice LR coexpression matrices."""
    lr_mats = []
    edge_counts = []

    for slice_index, data_cur in enumerate(spidernet_data_list):
        if "cellpair_LRpair_neigh" not in data_cur:
            raise KeyError(
                "processed graph contains no 'cellpair_LRpair_neigh'. "
                "NMF-LR requires edge-level LR coexpression features."
            )
        X_cur = _clean_nonnegative_matrix_for_nmf(data_cur["cellpair_LRpair_neigh"])
        lr_mats.append(X_cur)
        edge_counts.append(X_cur.shape[0])
        print(f"Collected LR coexpression slice {slice_index + 1}/{len(spidernet_data_list)}: {X_cur.shape}")

    if any(sp.issparse(X) for X in lr_mats):
        lr_all = sp.vstack(lr_mats, format="csr")
    else:
        lr_all = np.vstack(lr_mats)

    print("Fitting global NMF-LR:", lr_all.shape, "rank=", n_components)
    nmf = NMF(
        n_components=n_components,
        init="nndsvda",
        random_state=random_state,
        max_iter=max_iter,
    )
    factor_all = nmf.fit_transform(lr_all)

    colmax = np.max(factor_all, axis=0)
    colmax[colmax <= 0] = 1.0
    factor_all = (factor_all / colmax.reshape(1, -1)).astype(np.float32, copy=False)

    factor_list = []
    start = 0
    for n_edges in edge_counts:
        end = start + n_edges
        factor_list.append(factor_all[start:end].copy())
        start = end
    return factor_list


def _high_mask_from_split_rule(values, split_rule="median_rank_split_upper_half"):
    values = np.asarray(values, dtype=float)
    high_mask = np.zeros(values.shape[0], dtype=bool)
    finite_mask = np.isfinite(values)

    if not np.any(finite_mask):
        return high_mask

    if split_rule == "median_rank_split_upper_half":
        finite_idx = np.where(finite_mask)[0]
        vals = values[finite_idx]
        # primary key = value, secondary key = original index for deterministic ties
        order = np.lexsort((finite_idx, vals))
        high_rel = order[len(order) // 2:]
        high_mask[finite_idx[high_rel]] = True
        return high_mask

    if split_rule == "median_ge":
        med = np.nanmedian(values[finite_mask])
        high_mask[finite_mask] = values[finite_mask] >= med
        return high_mask

    raise ValueError(f"Unknown split_rule={split_rule}")


def compute_smd_from_groups(high_scores, low_scores, min_cells_per_group=5):
    high = pd.to_numeric(pd.Series(high_scores), errors="coerce").dropna().to_numpy(dtype=float)
    low = pd.to_numeric(pd.Series(low_scores), errors="coerce").dropna().to_numpy(dtype=float)

    n_high = int(high.size)
    n_low = int(low.size)
    mean_high = float(np.mean(high)) if n_high else np.nan
    mean_low = float(np.mean(low)) if n_low else np.nan
    sd_high = float(np.std(high, ddof=1)) if n_high > 1 else np.nan
    sd_low = float(np.std(low, ddof=1)) if n_low > 1 else np.nan

    pooled_sd = np.nan
    smd = np.nan
    if n_high > 1 and n_low > 1 and np.isfinite(sd_high) and np.isfinite(sd_low):
        pooled_var = ((n_high - 1) * sd_high**2 + (n_low - 1) * sd_low**2) / (n_high + n_low - 2)
        if np.isfinite(pooled_var) and pooled_var > 0:
            pooled_sd = float(np.sqrt(pooled_var))
            smd = float((mean_high - mean_low) / pooled_sd)

    p_val = np.nan
    if n_high >= min_cells_per_group and n_low >= min_cells_per_group:
        try:
            p_val = mannwhitneyu(high, low, alternative="two-sided", method="asymptotic").pvalue
        except TypeError:
            p_val = mannwhitneyu(high, low, alternative="two-sided").pvalue

    return {
        "n_high": n_high,
        "n_low": n_low,
        "mean_high": mean_high,
        "mean_low": mean_low,
        "sd_high": sd_high,
        "sd_low": sd_low,
        "pooled_sd": pooled_sd,
        "SMD": smd,
        "p_wilcox": p_val,
        "significance": p_to_star(p_val),
    }




def set_plot_style():
    plt.close("all")
    plt.style.use("default")
    rcParams.update({
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "font.family": "Arial",
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "axes.linewidth": 0.8,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "figure.dpi": 150,
        "savefig.dpi": 300,
    })


METHOD_COLORS = {
    "SpiderNet": {"edge": "#9F3B38", "fill": "#E1B6A7"},
    "NMF-LR": {"edge": "#82CCE2", "fill": "#D4ECF1"},
    "COMMOT": {"edge": "#519384", "fill": "#B9CEC7"},
    "ScCChain": {"edge": "#636491", "fill": "#A6A2B9"},
    "Spacia": {"edge": "#FED881", "fill": "#FFF2D2"},
}

In [ ]:
# ============================================================
# B7.4. Load processed data and SpiderNet outputs
# ============================================================
required_processed_files = [
    "adata_list.pkl",
    "SpiderNet_data_pyg_list.pkl",
    "LR_list.pkl",
]
required_result_files = [
    "Factor_envir_list.pkl",
]

missing_processed = [f for f in required_processed_files if not (input_path(PROCESSED_DATA_DIR / f)).exists()]
missing_result = [f for f in required_result_files if not (input_path(RUN_DIR / f)).exists()]

if missing_processed:
    raise FileNotFoundError(f"Missing processed files under {PROCESSED_DATA_DIR}: {missing_processed}")
if missing_result:
    raise FileNotFoundError(f"Missing SpiderNet result files under {RUN_DIR}: {missing_result}")

adata_list = pd.read_pickle(input_path(PROCESSED_DATA_DIR / "adata_list.pkl"))
SpiderNet_data_pyg_list = pd.read_pickle(input_path(PROCESSED_DATA_DIR / "SpiderNet_data_pyg_list.pkl"))
Factor_envir_list = pd.read_pickle(input_path(RUN_DIR / "Factor_envir_list.pkl"))

adata_all_path = PROCESSED_DATA_DIR / "adata_all.h5ad"
if input_path(adata_all_path).exists():
    adata_all = sc.read_h5ad(input_path(adata_all_path))
else:
    adata_all = sc.concat(adata_list, join="inner", merge="same") if len(adata_list) > 1 else adata_list[0].copy()

# Build global row offsets matching concatenated adata_list order.
slice_offsets = np.cumsum([0] + [ad.n_obs for ad in adata_list[:-1]])
cell_global_index_parts = []
celltype_parts = []
age_parts = []
barcode_parts = []
slice_parts = []

for slice_index, adata_sub in enumerate(adata_list):
    n = adata_sub.n_obs
    cell_global_index_parts.append(np.arange(slice_offsets[slice_index], slice_offsets[slice_index] + n))
    barcode_parts.extend(adata_sub.obs_names.astype(str).tolist())
    slice_parts.extend([slice_index] * n)
    if CELLTYPE_COL in adata_sub.obs.columns:
        celltype_parts.extend(adata_sub.obs[CELLTYPE_COL].astype(str).tolist())
    else:
        raise KeyError(f"{CELLTYPE_COL!r} not found in adata_list[{slice_index}].obs")
    if AGE_COL in adata_sub.obs.columns:
        age_parts.extend(adata_sub.obs[AGE_COL].astype(str).tolist())
    else:
        age_parts.extend([np.nan] * n)

cell_meta_df = pd.DataFrame({
    "global_cell_index": np.concatenate(cell_global_index_parts),
    "barcode": barcode_parts,
    "slice_index": slice_parts,
    "celltype": celltype_parts,
    "age": age_parts,
})

print("n_slices:", len(adata_list))
print("n_cells:", len(cell_meta_df))
print("n_T_cells:", int((cell_meta_df["celltype"] == T_CELL_LABEL).sum()))
print("First SpiderNet factor shape:", np.asarray(Factor_envir_list[0]).shape)
display(cell_meta_df.head())

In [ ]:
# ============================================================
import re
# B7.5. Check ageing-gene log2 fold changes: five old versus five young slices
# ------------------------------------------------------------
# Purpose:
#   Before building the Aging module score, verify which aging/senescence genes
#   are actually upregulated in old slices.
#
# Definition:
#   1. Build aging/senescence genes from the MERFISH gene panel.
#   2. Sort slices by numeric age.
#   3. Use the 5 youngest slices as Young and 5 oldest slices as Old.
#   4. For each aging gene, compute:
#        a) slice-level log2FC: mean over slices after per-slice gene means
#        b) cell-level log2FC: pooled cell mean in Old vs Young selected slices
#
# The next cell uses:
#   aging_gene_old_vs_young_lfc_df['log2FC_old_vs_young_cell_mean'] > 0.2
# to select good aging genes for the module score.
# ============================================================

AGING_GENE_LFC_N_YOUNG_SLICES = 5
AGING_GENE_LFC_N_OLD_SLICES = 5
AGING_GENE_LFC_PSEUDOCOUNT = 1e-6


def _extract_first_number_for_age(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"[-+]?\d*\.?\d+", str(value))
    if match is None:
        return np.nan
    try:
        return float(match.group(0))
    except Exception:
        return np.nan


def _resolve_gene_names_from_varnames_for_lfc(var_names, genes):
    """Resolve requested gene symbols to actual var_names, case-insensitively."""
    var_names = pd.Index([str(x) for x in var_names])
    upper_to_actual = {}
    for g in var_names:
        upper_to_actual.setdefault(str(g).upper(), str(g))

    resolved = {}
    missing = []
    for gene in genes:
        gene = str(gene).strip()
        if len(gene) == 0:
            continue
        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)
    return resolved, missing


def _get_dense_gene_matrix_for_lfc(adata, genes_actual):
    gene_idx = [adata.var_names.get_loc(g) for g in genes_actual]
    X_sub = adata.X[:, gene_idx]
    if sp.issparse(X_sub):
        X_sub = X_sub.toarray()
    else:
        X_sub = np.asarray(X_sub)
    return X_sub.astype(float, copy=False)


# -----------------------------
# 1. Build aging/senescence gene set from MERFISH gene panel
# -----------------------------
genepanel_path = DATA_ROOT / "Supp_table" / "2023-12-22736D-TableS1_MERFISHGenePanel.xlsx"
if not input_path(genepanel_path).exists():
    raise FileNotFoundError(f"Cannot find MERFISH gene panel: {genepanel_path}")

genepanel_table = pd.read_excel(input_path(genepanel_path))
required_cols = {"Rationale for inclusion", "Vizgen Gene"}
missing_cols = required_cols.difference(genepanel_table.columns)
if len(missing_cols) > 0:
    raise KeyError(f"MERFISH gene-panel table is missing columns: {sorted(missing_cols)}")

rationale = genepanel_table["Rationale for inclusion"].fillna("").astype(str)
aging_mask = rationale.str.contains("aging|senescence", case=False, regex=True)

aging_genes_lfc_requested = (
    genepanel_table.loc[aging_mask, "Vizgen Gene"]
    .dropna()
    .astype(str)
    .map(lambda x: x.strip())
)
aging_genes_lfc_requested = [g for g in aging_genes_lfc_requested if len(g) > 0]
aging_genes_lfc_requested = list(dict.fromkeys(aging_genes_lfc_requested))

if len(aging_genes_lfc_requested) == 0:
    raise ValueError("No aging/senescence genes were found from the MERFISH gene-panel rationale column.")


# -----------------------------
# 2. Resolve aging genes in the expression matrix
# -----------------------------
ref_var_names = adata_list[0].var_names.astype(str)
resolved_genes_lfc, missing_aging_genes_lfc = _resolve_gene_names_from_varnames_for_lfc(
    ref_var_names,
    aging_genes_lfc_requested,
)

aging_genes_lfc_used_requested = [
    g for g in aging_genes_lfc_requested
    if g in resolved_genes_lfc
]
aging_genes_lfc_actual = [
    resolved_genes_lfc[g]
    for g in aging_genes_lfc_used_requested
]

if len(aging_genes_lfc_actual) == 0:
    raise ValueError("None of the aging/senescence genes are present in adata_list[0].var_names.")

print(f"Aging genes used for log2FC: {len(aging_genes_lfc_actual)} / {len(aging_genes_lfc_requested)}")
if len(missing_aging_genes_lfc) > 0:
    print(f"[Warning] Missing aging genes skipped: {missing_aging_genes_lfc}")


# -----------------------------
# 3. Build slice-level expression table
# -----------------------------
slice_mean_rows = []
cell_level_parts = []

for slice_index, adata_sub in enumerate(adata_list):
    missing_cur = [g for g in aging_genes_lfc_actual if g not in adata_sub.var_names]
    if len(missing_cur) > 0:
        raise ValueError(f"Slice {slice_index} is missing aging genes present in slice 0: {missing_cur}")

    if AGE_COL not in adata_sub.obs.columns:
        raise KeyError(f"adata_list[{slice_index}].obs must contain {AGE_COL!r} for slice age grouping.")

    age_values = (
        pd.Series(adata_sub.obs[AGE_COL])
        .dropna()
        .astype(str)
        .str.strip()
    )
    age_values = age_values[(age_values != "") & (age_values.str.lower() != "nan")]
    age_value = age_values.iloc[0] if len(age_values) > 0 else np.nan
    age_numeric = _extract_first_number_for_age(age_value)

    X_gene_cur = _get_dense_gene_matrix_for_lfc(adata_sub, aging_genes_lfc_actual)
    slice_gene_mean = np.nanmean(X_gene_cur, axis=0)

    for requested_gene, actual_gene, mean_expr in zip(
        aging_genes_lfc_used_requested,
        aging_genes_lfc_actual,
        slice_gene_mean,
    ):
        slice_mean_rows.append({
            "slice_index": slice_index,
            "age": age_value,
            "age_numeric": age_numeric,
            "n_cells": int(adata_sub.n_obs),
            "requested_gene": requested_gene,
            "actual_gene": actual_gene,
            "slice_mean_expression": float(mean_expr),
        })

    # Secondary cell-level check. Number of aging genes is small, so this is memory-safe enough here.
    cell_df_cur = pd.DataFrame(X_gene_cur, columns=aging_genes_lfc_actual)
    cell_df_cur["slice_index"] = slice_index
    cell_df_cur["age"] = age_value
    cell_df_cur["age_numeric"] = age_numeric
    cell_level_parts.append(cell_df_cur)

aging_gene_slice_mean_df = pd.DataFrame(slice_mean_rows)

slice_age_df = (
    aging_gene_slice_mean_df[["slice_index", "age", "age_numeric", "n_cells"]]
    .drop_duplicates()
    .sort_values(["age_numeric", "slice_index"])
    .reset_index(drop=True)
)

if slice_age_df["age_numeric"].isna().any():
    bad_slices = slice_age_df.loc[slice_age_df["age_numeric"].isna(), ["slice_index", "age"]]
    raise ValueError(
        "Some slices have non-numeric age values and cannot be sorted into young/old groups:\n"
        + str(bad_slices)
    )

n_slices_total = slice_age_df.shape[0]
required_n = AGING_GENE_LFC_N_YOUNG_SLICES + AGING_GENE_LFC_N_OLD_SLICES
if n_slices_total < required_n:
    raise ValueError(
        f"Need at least {required_n} slices to select "
        f"{AGING_GENE_LFC_N_YOUNG_SLICES} young + {AGING_GENE_LFC_N_OLD_SLICES} old slices, "
        f"but found only {n_slices_total} slices."
    )

young_slice_ids = slice_age_df.head(AGING_GENE_LFC_N_YOUNG_SLICES)["slice_index"].astype(int).tolist()
old_slice_ids = slice_age_df.tail(AGING_GENE_LFC_N_OLD_SLICES)["slice_index"].astype(int).tolist()

slice_group_map = {
    **{sid: "Young" for sid in young_slice_ids},
    **{sid: "Old" for sid in old_slice_ids},
}

slice_age_df["AgeGroup_5young_5old"] = slice_age_df["slice_index"].map(slice_group_map)
selected_slice_age_df = slice_age_df[
    slice_age_df["AgeGroup_5young_5old"].isin(["Young", "Old"])
].copy()

print("Selected 5 young + 5 old slices:")
display(selected_slice_age_df)


# -----------------------------
# 4. Compute slice-level log2FC old vs young for each aging gene
# -----------------------------
aging_gene_slice_mean_df["AgeGroup_5young_5old"] = aging_gene_slice_mean_df["slice_index"].map(slice_group_map)
selected_slice_mean_df = aging_gene_slice_mean_df[
    aging_gene_slice_mean_df["AgeGroup_5young_5old"].isin(["Young", "Old"])
].copy()

lfc_rows = []

for actual_gene, df_gene in selected_slice_mean_df.groupby("actual_gene", sort=False):
    requested_gene_values = df_gene["requested_gene"].dropna().astype(str).unique()
    requested_gene = requested_gene_values[0] if len(requested_gene_values) > 0 else actual_gene

    young_vals = pd.to_numeric(
        df_gene.loc[df_gene["AgeGroup_5young_5old"] == "Young", "slice_mean_expression"],
        errors="coerce",
    ).dropna()
    old_vals = pd.to_numeric(
        df_gene.loc[df_gene["AgeGroup_5young_5old"] == "Old", "slice_mean_expression"],
        errors="coerce",
    ).dropna()

    mean_young_slice = float(np.nanmean(young_vals)) if len(young_vals) > 0 else np.nan
    mean_old_slice = float(np.nanmean(old_vals)) if len(old_vals) > 0 else np.nan
    log2fc_slice_mean_old_vs_young = np.log2(
        (mean_old_slice + AGING_GENE_LFC_PSEUDOCOUNT)
        / (mean_young_slice + AGING_GENE_LFC_PSEUDOCOUNT)
    )

    if len(young_vals) > 0 and len(old_vals) > 0:
        try:
            stat, pval = mannwhitneyu(young_vals, old_vals, alternative="two-sided", method="asymptotic")
        except TypeError:
            stat, pval = mannwhitneyu(young_vals, old_vals, alternative="two-sided")
    else:
        stat, pval = np.nan, np.nan

    lfc_rows.append({
        "requested_gene": requested_gene,
        "actual_gene": actual_gene,
        "n_young_slices": int(len(young_vals)),
        "n_old_slices": int(len(old_vals)),
        "mean_young_slice_mean_expression": mean_young_slice,
        "mean_old_slice_mean_expression": mean_old_slice,
        "log2FC_old_vs_young_slice_mean": float(log2fc_slice_mean_old_vs_young),
        "old_greater_than_young_slice_mean": bool(log2fc_slice_mean_old_vs_young > 0),
        "mannwhitneyu_stat_slice_means": stat,
        "pvalue_slice_means": pval,
    })

aging_gene_old_vs_young_lfc_df = pd.DataFrame(lfc_rows)


# -----------------------------
# 5. Secondary check: cell-level mean expression pooled across selected old/young cells
# -----------------------------
cell_level_df = pd.concat(cell_level_parts, axis=0, ignore_index=True)
cell_level_df["AgeGroup_5young_5old"] = cell_level_df["slice_index"].map(slice_group_map)
cell_level_df = cell_level_df[cell_level_df["AgeGroup_5young_5old"].isin(["Young", "Old"])].copy()

cell_level_lfc_rows = []

for requested_gene, actual_gene in zip(aging_genes_lfc_used_requested, aging_genes_lfc_actual):
    young_cell_vals = pd.to_numeric(
        cell_level_df.loc[cell_level_df["AgeGroup_5young_5old"] == "Young", actual_gene],
        errors="coerce",
    ).dropna()
    old_cell_vals = pd.to_numeric(
        cell_level_df.loc[cell_level_df["AgeGroup_5young_5old"] == "Old", actual_gene],
        errors="coerce",
    ).dropna()

    mean_young_cell = float(np.nanmean(young_cell_vals)) if len(young_cell_vals) > 0 else np.nan
    mean_old_cell = float(np.nanmean(old_cell_vals)) if len(old_cell_vals) > 0 else np.nan
    log2fc_cell_mean_old_vs_young = np.log2(
        (mean_old_cell + AGING_GENE_LFC_PSEUDOCOUNT)
        / (mean_young_cell + AGING_GENE_LFC_PSEUDOCOUNT)
    )

    cell_level_lfc_rows.append({
        "actual_gene": actual_gene,
        "mean_young_cell_expression": mean_young_cell,
        "mean_old_cell_expression": mean_old_cell,
        "log2FC_old_vs_young_cell_mean": float(log2fc_cell_mean_old_vs_young),
    })

aging_gene_cell_level_lfc_df = pd.DataFrame(cell_level_lfc_rows)
aging_gene_old_vs_young_lfc_df = aging_gene_old_vs_young_lfc_df.merge(
    aging_gene_cell_level_lfc_df,
    on="actual_gene",
    how="left",
)

aging_gene_old_vs_young_lfc_df = aging_gene_old_vs_young_lfc_df.sort_values(
    "log2FC_old_vs_young_slice_mean",
    ascending=False,
).reset_index(drop=True)


# -----------------------------
# 6. Save and display
# -----------------------------
slice_selection_path = OUT_DIR / "aging_gene_log2FC_5young_5old_selected_slices.csv"
slice_mean_path = OUT_DIR / "aging_gene_log2FC_5young_5old_slice_mean_expression_long.csv"
lfc_path = OUT_DIR / "aging_gene_log2FC_5young_5old_old_vs_young.csv"

selected_slice_age_df.to_csv(output_path(slice_selection_path), index=False)
selected_slice_mean_df.to_csv(output_path(slice_mean_path), index=False)
aging_gene_old_vs_young_lfc_df.to_csv(output_path(lfc_path), index=False)

print(f"Saved selected slice table to: {slice_selection_path}")
print(f"Saved slice-level expression table to: {slice_mean_path}")
print(f"Saved aging gene log2FC table to: {lfc_path}")

n_total_genes = aging_gene_old_vs_young_lfc_df.shape[0]
n_positive_cell_lfc = int((aging_gene_old_vs_young_lfc_df["log2FC_old_vs_young_cell_mean"] > 0).sum())
n_good_cell_lfc = int((aging_gene_old_vs_young_lfc_df["log2FC_old_vs_young_cell_mean"] > 0.2).sum())

print(f"Aging genes with cell-level log2FC(old vs young) > 0: {n_positive_cell_lfc} / {n_total_genes}")
print(f"Aging genes with cell-level log2FC(old vs young) > 0.2: {n_good_cell_lfc} / {n_total_genes}")

display(aging_gene_old_vs_young_lfc_df)


In [ ]:
# ============================================================
# B7.6. Compute the ageing-module score from filtered ageing genes
# ------------------------------------------------------------
# This replaces the previous all-aging-gene module score.
#
# good aging genes =
#   aging_gene_old_vs_young_lfc_df['log2FC_old_vs_young_cell_mean'] > 0.2
#
# Score definition:
#   1. Build aging/senescence gene set from the MERFISH gene panel.
#   2. Resolve genes in current expression matrix.
#   3. Keep only genes with cell-level log2FC(old vs young) > 0.2.
#   4. Concatenate all cells across all slices.
#   5. For each selected good aging gene, compute one global mean/std.
#   6. For each cell, z-score each gene using that global reference.
#   7. Average the global z-scored genes to obtain one Aging module score.
#
# Downstream CCC/SMD cells keep using:
#   aging_score_all
#   cell_meta_df['AgingGeneExp']
# ============================================================

MODULE_SCORE_METHOD = "zscore_mean_global_good_aging_genes_cellLFC_gt0.2"
GOOD_AGING_GENE_CELL_LFC_THRESHOLD = 0.2


def _resolve_gene_names_from_varnames(var_names, genes):
    """Resolve requested gene symbols to actual var_names, case-insensitively."""
    var_names = pd.Index([str(x) for x in var_names])
    upper_to_actual = {}
    for g in var_names:
        upper_to_actual.setdefault(str(g).upper(), str(g))

    resolved = {}
    missing = []
    for gene in genes:
        gene = str(gene).strip()
        if len(gene) == 0:
            continue
        if gene in var_names:
            resolved[gene] = gene
        elif gene.upper() in upper_to_actual:
            resolved[gene] = upper_to_actual[gene.upper()]
        else:
            missing.append(gene)
    return resolved, missing


def _get_dense_gene_matrix(adata, genes_actual):
    gene_idx = [adata.var_names.get_loc(g) for g in genes_actual]
    X_sub = adata.X[:, gene_idx]
    if sp.issparse(X_sub):
        X_sub = X_sub.toarray()
    else:
        X_sub = np.asarray(X_sub)
    return X_sub.astype(float, copy=False)


if "aging_gene_old_vs_young_lfc_df" not in globals():
    raise NameError(
        "Please run cell 5A first so that aging_gene_old_vs_young_lfc_df exists."
    )

required_lfc_cols = {"actual_gene", "log2FC_old_vs_young_cell_mean"}
missing_lfc_cols = required_lfc_cols.difference(aging_gene_old_vs_young_lfc_df.columns)
if len(missing_lfc_cols) > 0:
    raise KeyError(
        "aging_gene_old_vs_young_lfc_df is missing required columns: "
        f"{sorted(missing_lfc_cols)}"
    )


# -----------------------------
# 1. Build aging gene set from MERFISH gene panel
# -----------------------------
genepanel_path = DATA_ROOT / "Supp_table" / "2023-12-22736D-TableS1_MERFISHGenePanel.xlsx"
if not input_path(genepanel_path).exists():
    raise FileNotFoundError(f"Cannot find MERFISH gene panel: {genepanel_path}")

genepanel_table = pd.read_excel(input_path(genepanel_path))
required_cols = {"Rationale for inclusion", "Vizgen Gene"}
missing_cols = required_cols.difference(genepanel_table.columns)
if len(missing_cols) > 0:
    raise KeyError(f"MERFISH gene-panel table is missing columns: {sorted(missing_cols)}")

rationale = genepanel_table["Rationale for inclusion"].fillna("").astype(str)
aging_mask = rationale.str.contains("aging|senescence", case=False, regex=True)
aging_genes = (
    genepanel_table.loc[aging_mask, "Vizgen Gene"]
    .dropna()
    .astype(str)
    .map(lambda x: x.strip())
)
aging_genes = [g for g in aging_genes if len(g) > 0]
aging_genes = list(dict.fromkeys(aging_genes))

if len(aging_genes) == 0:
    raise ValueError("No aging/senescence genes were found from the MERFISH gene-panel rationale column.")


# -----------------------------
# 2. Resolve genes in current expression matrix
# -----------------------------
ref_var_names = adata_list[0].var_names.astype(str)
resolved_genes, missing_aging_genes = _resolve_gene_names_from_varnames(ref_var_names, aging_genes)
aging_genes_used_requested_all = [g for g in aging_genes if g in resolved_genes]
aging_genes_actual_all = [resolved_genes[g] for g in aging_genes_used_requested_all]

if len(aging_genes_actual_all) == 0:
    raise ValueError("None of the aging/senescence genes are present in adata_list[0].var_names.")

print(f"Aging genes resolved before log2FC filtering: {len(aging_genes_actual_all)} / {len(aging_genes)}")
if len(missing_aging_genes) > 0:
    print(f"[Warning] Missing aging genes skipped before log2FC filtering: {missing_aging_genes}")


# -----------------------------
# 3. Keep only good aging genes:
#    cell-level log2FC(old vs young) > 0.2
# -----------------------------
lfc_df = aging_gene_old_vs_young_lfc_df.copy()
lfc_df["actual_gene"] = lfc_df["actual_gene"].astype(str)
lfc_df["log2FC_old_vs_young_cell_mean"] = pd.to_numeric(
    lfc_df["log2FC_old_vs_young_cell_mean"],
    errors="coerce",
)

good_lfc_df = lfc_df[
    np.isfinite(lfc_df["log2FC_old_vs_young_cell_mean"])
    & (lfc_df["log2FC_old_vs_young_cell_mean"] > GOOD_AGING_GENE_CELL_LFC_THRESHOLD)
].copy()

good_actual_gene_set = set(good_lfc_df["actual_gene"].astype(str))

aging_genes_used_requested = []
aging_genes_actual = []
for requested_gene, actual_gene in zip(aging_genes_used_requested_all, aging_genes_actual_all):
    if str(actual_gene) in good_actual_gene_set:
        aging_genes_used_requested.append(requested_gene)
        aging_genes_actual.append(actual_gene)

if len(aging_genes_actual) == 0:
    raise ValueError(
        "No aging genes passed the filter: "
        f"log2FC_old_vs_young_cell_mean > {GOOD_AGING_GENE_CELL_LFC_THRESHOLD}."
    )

good_lfc_keep_df = good_lfc_df[good_lfc_df["actual_gene"].isin(aging_genes_actual)].copy()

print(
    "Good aging genes used after cell-level log2FC filtering: "
    f"{len(aging_genes_actual)} / {len(aging_genes_actual_all)} resolved aging genes"
)
print(f"Filter: log2FC_old_vs_young_cell_mean > {GOOD_AGING_GENE_CELL_LFC_THRESHOLD}")

display(
    good_lfc_keep_df[
        [
            "requested_gene",
            "actual_gene",
            "mean_young_cell_expression",
            "mean_old_cell_expression",
            "log2FC_old_vs_young_cell_mean",
            "log2FC_old_vs_young_slice_mean",
        ]
    ].sort_values("log2FC_old_vs_young_cell_mean", ascending=False)
)


# -----------------------------
# 4. Concatenate selected good-aging-gene expression across slices
# -----------------------------
X_aging_parts = []
for slice_index, adata_sub in enumerate(adata_list):
    missing_cur = [g for g in aging_genes_actual if g not in adata_sub.var_names]
    if len(missing_cur) > 0:
        raise ValueError(f"Slice {slice_index} is missing selected good aging genes: {missing_cur}")
    X_aging_parts.append(_get_dense_gene_matrix(adata_sub, aging_genes_actual))

X_aging_all = np.vstack(X_aging_parts).astype(float, copy=False)


# -----------------------------
# 5. Global z-score per selected good aging gene, then mean per cell
# -----------------------------
gene_mean = np.nanmean(X_aging_all, axis=0)
gene_std = np.nanstd(X_aging_all, axis=0, ddof=1)
gene_std[(~np.isfinite(gene_std)) | (gene_std == 0)] = np.nan

with np.errstate(invalid="ignore", divide="ignore"):
    X_aging_z = (X_aging_all - gene_mean.reshape(1, -1)) / gene_std.reshape(1, -1)
X_aging_z = np.nan_to_num(X_aging_z, nan=0.0, posinf=0.0, neginf=0.0)
aging_score_all = np.mean(X_aging_z, axis=1).astype(float, copy=False)

if aging_score_all.shape[0] != len(cell_meta_df):
    raise ValueError(f"Aging score length {aging_score_all.shape[0]} != cell metadata length {len(cell_meta_df)}")

cell_meta_df["AgingGeneExp"] = aging_score_all
cell_meta_df["AgingGeneExp_score_method"] = MODULE_SCORE_METHOD
cell_meta_df["AgingGeneExp_gene_filter"] = f"log2FC_old_vs_young_cell_mean>{GOOD_AGING_GENE_CELL_LFC_THRESHOLD}"

aging_gene_used_df = pd.DataFrame({
    "requested_gene": aging_genes_used_requested,
    "actual_gene": aging_genes_actual,
    "gene_mean_global": gene_mean,
    "gene_std_global": gene_std,
})

aging_gene_used_df = aging_gene_used_df.merge(
    good_lfc_keep_df[
        [
            "actual_gene",
            "mean_young_cell_expression",
            "mean_old_cell_expression",
            "log2FC_old_vs_young_cell_mean",
            "mean_young_slice_mean_expression",
            "mean_old_slice_mean_expression",
            "log2FC_old_vs_young_slice_mean",
        ]
    ],
    on="actual_gene",
    how="left",
)


# -----------------------------
# 6. Save outputs for reproducibility and backward compatibility
# -----------------------------
aging_score_path = OUT_DIR / "Aging_module_score_good_aging_genes_cellLFC_gt0p2_global_zscore_mean.csv"
aging_gene_path = OUT_DIR / "Aging_module_score_good_aging_genes_cellLFC_gt0p2_global_zscore_mean_genes_used.csv"
cell_meta_df.to_csv(output_path(aging_score_path), index=False)
aging_gene_used_df.to_csv(output_path(aging_gene_path), index=False)

# Compatibility outputs with the original filenames, now containing the filtered-gene score.
compat_aging_score_path = OUT_DIR / "Aging_module_score_global_zscore_mean.csv"
compat_aging_gene_path = OUT_DIR / "Aging_module_score_global_zscore_mean_genes_used.csv"
cell_meta_df.to_csv(output_path(compat_aging_score_path), index=False)
aging_gene_used_df.to_csv(output_path(compat_aging_gene_path), index=False)

print("Saved good-aging-gene aging score:", aging_score_path)
print("Saved good-aging-gene list:", aging_gene_path)
print("Compatibility score path:", compat_aging_score_path)
print("Compatibility gene path:", compat_aging_gene_path)

display(cell_meta_df[["barcode", "slice_index", "age", "celltype", "AgingGeneExp"]].head())
display(cell_meta_df["AgingGeneExp"].describe())
display(aging_gene_used_df.sort_values("log2FC_old_vs_young_cell_mean", ascending=False))


In [ ]:
# ============================================================
# B7.7. Load edge-level features from each CCC method
# ============================================================

def get_commot_path_for_slice(adata_sub):
    age_cur = np.unique(adata_sub.obs[AGE_COL].astype(str))[0]
    return COMMOT_PATH_MAIN / f"aging_coronal_age{age_cur}_commot.h5ad"


def get_commot_pathway_keys(commot_adata):
    keys = list(commot_adata.obsp.keys())
    total_key = "commot-cellchat-total-total"
    pathway_keys = [
        k for k in keys
        if k.startswith("commot-cellchat-") and len(k.split("-")) == 3 and k != total_key
    ]
    return sorted(pathway_keys)


def load_commot_outputs_union():
    # Build union columns first so feature dimensions are aligned across slices.
    pathway_union_set = set()
    for adata_sub in adata_list:
        commot_path = get_commot_path_for_slice(adata_sub)
        if not input_path(commot_path).exists():
            raise FileNotFoundError(f"Missing COMMOT file: {commot_path}")
        commot_adata = sc.read_h5ad(input_path(commot_path))
        pathway_union_set.update(get_commot_pathway_keys(commot_adata))
    pathway_union = sorted(pathway_union_set)
    pathway_to_col = {k: i for i, k in enumerate(pathway_union)}

    outputs = []
    for slice_index, adata_sub in enumerate(adata_list):
        print(f"Loading COMMOT slice {slice_index + 1}/{len(adata_list)}")
        commot_path = get_commot_path_for_slice(adata_sub)
        commot_adata = sc.read_h5ad(input_path(commot_path))
        edge_index = edge_index_to_e2(get_data_item(SpiderNet_data_pyg_list[slice_index], "edge_index"))
        rows = edge_index[:, 0].astype(np.int64)
        cols = edge_index[:, 1].astype(np.int64)
        arr = np.zeros((edge_index.shape[0], len(pathway_union)), dtype=np.float32)
        for key in get_commot_pathway_keys(commot_adata):
            j = pathway_to_col[key]
            arr[:, j] = edge_matrix_from_square(commot_adata.obsp[key], rows, cols)
        outputs.append(arr)

    feature_names = [k.replace("commot-cellchat-", "") for k in pathway_union]
    return outputs, feature_names


def load_sccchain_outputs_union():
    """Memory-safe scCChain loader.

    The original CCC_Coupling_benchmark code first built a dense n_cells x n_cells
    matrix for each scCChain program and then sampled SpiderNet edges from it.
    That is unsafe for large AgingBrain slices, e.g. 73,309^2 float32 values
    require about 20 GB per program.

    This function instead directly joins scCChain sender/receiver pairs to the
    SpiderNet edge_index and returns an E x K edge-feature matrix, where E is the
    number of SpiderNet spatial edges and K is the number of scCChain programs.
    """
    txt_path = SC_CCHAIN_PATH_MAIN / "h5ad_files.txt"
    if not input_path(txt_path).exists():
        raise FileNotFoundError(f"Missing scCChain h5ad list: {txt_path}")

    scc_h5ad_files = pd.read_csv(input_path(txt_path), header=None)[0].astype(str).tolist()
    n_obs_ref = np.asarray([ad.n_obs for ad in adata_list])

    n_obs_scc = []
    for p in scc_h5ad_files:
        ad = sc.read_h5ad(input_path(p), backed="r")
        n_obs_scc.append(ad.n_obs)
        del ad
    n_obs_scc = np.asarray(n_obs_scc)

    index_map = []
    used = set()
    for slice_index, n in enumerate(n_obs_ref):
        matched = np.where(n_obs_scc == n)[0]
        if len(matched) == 0:
            raise ValueError(f"Cannot match scCChain result by n_obs={n} for slice {slice_index}")
        # Keep the old benchmark behavior: match by n_obs. If multiple files have the same n_obs,
        # choose the first unused file; if all are used, fall back to the first match.
        chosen = None
        for m in matched:
            if int(m) not in used:
                chosen = int(m)
                break
        if chosen is None:
            chosen = int(matched[0])
        index_map.append(chosen)
        used.add(chosen)

    # Build union of scCChain program columns across slices.
    feature_union = []
    for slice_index, matched_idx in enumerate(index_map):
        src_path = Path(scc_h5ad_files[matched_idx])
        stem = src_path.stem
        score_path = SC_CCHAIN_PATH_MAIN / f"{stem}_ScCChain_edge_program_scores.csv"
        if not input_path(score_path).exists():
            raise FileNotFoundError(f"Missing scCChain score file: {score_path}")
        cols = pd.read_csv(input_path(score_path), nrows=1).columns.tolist()[2:]
        for c in cols:
            if c not in feature_union:
                feature_union.append(c)

    if len(feature_union) == 0:
        raise ValueError("No scCChain program columns were found in the score CSV files.")

    feature_to_col = {c: i for i, c in enumerate(feature_union)}
    outputs = []

    for slice_index, matched_idx in enumerate(index_map):
        print(f"Loading scCChain slice {slice_index + 1}/{len(adata_list)} with memory-safe edge join")
        src_path = Path(scc_h5ad_files[matched_idx])
        stem = src_path.stem
        score_path = SC_CCHAIN_PATH_MAIN / f"{stem}_ScCChain_edge_program_scores.csv"

        score_df = pd.read_csv(input_path(score_path))
        if not {"sender_index", "receiver_index"}.issubset(score_df.columns):
            raise ValueError(
                f"scCChain score file must contain sender_index and receiver_index columns: {score_path}"
            )

        score_cols = [c for c in score_df.columns.tolist()[2:] if c in feature_to_col]
        if len(score_cols) == 0:
            raise ValueError(f"No usable scCChain program columns found in {score_path}")

        # Normalize indices to the same 1-based convention as scCChain output.
        # SpiderNet edge_index is 0-based, so we add 1 before merging.
        edge_index = edge_index_to_e2(get_data_item(SpiderNet_data_pyg_list[slice_index], "edge_index"))
        rows_1based = edge_index[:, 0].astype(np.int64, copy=False) + 1
        cols_1based = edge_index[:, 1].astype(np.int64, copy=False) + 1
        E = edge_index.shape[0]

        edge_df = pd.DataFrame({
            "sender_index": rows_1based,
            "receiver_index": cols_1based,
            "__edge_pos": np.arange(E, dtype=np.int64),
        })

        keep_cols = ["sender_index", "receiver_index"] + score_cols
        score_use = score_df.loc[:, keep_cols].copy()
        score_use["sender_index"] = pd.to_numeric(score_use["sender_index"], errors="coerce").astype("Int64")
        score_use["receiver_index"] = pd.to_numeric(score_use["receiver_index"], errors="coerce").astype("Int64")
        score_use = score_use.dropna(subset=["sender_index", "receiver_index"])
        score_use["sender_index"] = score_use["sender_index"].astype(np.int64)
        score_use["receiver_index"] = score_use["receiver_index"].astype(np.int64)

        for c in score_cols:
            score_use[c] = pd.to_numeric(score_use[c], errors="coerce")

        # If duplicated cell pairs exist, keep the last row to mimic the old dense-matrix assignment:
        # A[sender_index, receiver_index] = values, where later assignments overwrite earlier ones.
        score_use = score_use.drop_duplicates(["sender_index", "receiver_index"], keep="last")

        merged = edge_df.merge(
            score_use,
            on=["sender_index", "receiver_index"],
            how="left",
            sort=False,
            copy=False,
        )
        if not np.array_equal(merged["__edge_pos"].to_numpy(), np.arange(E, dtype=np.int64)):
            merged = merged.sort_values("__edge_pos")

        arr = np.zeros((E, len(feature_union)), dtype=np.float32)
        vals = merged[score_cols].to_numpy(dtype=np.float32, copy=True)
        vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
        col_idx = [feature_to_col[c] for c in score_cols]
        arr[:, col_idx] = vals

        matched_any = np.isfinite(merged[score_cols].to_numpy(dtype=float, copy=True)).any(axis=1).sum()
        print(f"  matched scCChain score rows for {matched_any:,}/{E:,} SpiderNet edges")
        outputs.append(arr)

        # Help the garbage collector release large temporary DataFrames before the next slice.
        del score_df, score_use, edge_df, merged, vals

    return outputs, feature_union


def load_spacia_outputs():
    spacia_dir = SPACIA_PATH_MAIN / "spacia_outputs"
    if not input_path(spacia_dir).exists():
        raise FileNotFoundError(f"Missing Spacia output directory: {spacia_dir}")
    spacia_files = sorted([f for f in os.listdir(spacia_dir) if f.endswith(".h5ad")])
    if len(spacia_files) == 0:
        raise FileNotFoundError(f"No Spacia .h5ad files found in {spacia_dir}")

    n_obs_ref = np.asarray([ad.n_obs for ad in adata_list])
    n_obs_spacia = []
    for fname in spacia_files:
        ad = sc.read_h5ad(input_path(spacia_dir / fname), backed="r")
        n_obs_spacia.append(ad.n_obs)
        del ad
    n_obs_spacia = np.asarray(n_obs_spacia)

    index_map = []
    for n in n_obs_ref:
        matched = np.where(n_obs_spacia == n)[0]
        if len(matched) == 0:
            raise ValueError(f"Cannot match Spacia result by n_obs={n}")
        index_map.append(int(matched[0]))

    outputs = []
    n_programs = None
    for slice_index, matched_idx in enumerate(index_map):
        print(f"Loading Spacia slice {slice_index + 1}/{len(adata_list)}")
        spacia_path = spacia_dir / spacia_files[matched_idx]
        spacia_adata = sc.read_h5ad(input_path(spacia_path))
        edge_index = edge_index_to_e2(get_data_item(SpiderNet_data_pyg_list[slice_index], "edge_index"))
        rows = edge_index[:, 0].astype(np.int64)
        cols = edge_index[:, 1].astype(np.int64)
        scores = spacia_adata.obsp["interaction_scores"]
        if n_programs is None:
            n_programs = scores.shape[2]
        arr = np.zeros((edge_index.shape[0], n_programs), dtype=np.float32)
        for j in range(n_programs):
            arr[:, j] = edge_matrix_from_square(scores[:, :, j], rows, cols)
        outputs.append(arr)
    feature_names = [f"Spacia-{j+1}" for j in range(n_programs)]
    return outputs, feature_names


def validate_edge_feature_list(method, edge_feature_list):
    if len(edge_feature_list) != len(SpiderNet_data_pyg_list):
        raise ValueError(f"{method}: number of slices {len(edge_feature_list)} != {len(SpiderNet_data_pyg_list)}")
    for slice_index, arr in enumerate(edge_feature_list):
        edge_index = edge_index_to_e2(get_data_item(SpiderNet_data_pyg_list[slice_index], "edge_index"))
        if np.asarray(arr).shape[0] != edge_index.shape[0]:
            raise ValueError(
                f"{method} slice {slice_index}: edge feature rows {np.asarray(arr).shape[0]} != graph edges {edge_index.shape[0]}"
            )


method_edge_features = {}
method_feature_names = {}
method_load_errors = {}

for method in METHODS_TO_RUN:
    try:
        if method == "SpiderNet":
            method_edge_features[method] = [np.asarray(x, dtype=np.float32) for x in Factor_envir_list]
            method_feature_names[method] = [f"MI-{j+1}" for j in range(method_edge_features[method][0].shape[1])]
            n_spidernet_dims = method_edge_features[method][0].shape[1]
            if SPIDERNET_FIXED_DIM_ONE_BASED < 1 or SPIDERNET_FIXED_DIM_ONE_BASED > n_spidernet_dims:
                raise ValueError(
                    f"Requested SpiderNet fixed dimension {SPIDERNET_FIXED_DIM_ONE_BASED}, "
                    f"but Factor_envir_list has {n_spidernet_dims} dimensions."
                )
            print(f"SpiderNet will be fixed to {SPIDERNET_FIXED_FEATURE_NAME} for selected summaries/barplots.")
        elif method == "NMF-LR":
            from workflow_paths import same_file_content
            reuse_age_factors = (
                "Factor_LR_list" in globals()
                and (NMF_RANK, NMF_RANDOM_STATE, NMF_MAX_ITER)
                    == (NMF_LR_N_COMPONENTS, NMF_LR_RANDOM_STATE, NMF_LR_MAX_ITER)
                and same_file_content(
                    input_path(run_dirs["run_dir"] / "SpiderNet_data_pyg_list.pkl"),
                    PROCESSED_DATA_DIR / "SpiderNet_data_pyg_list.pkl",
                )
            )
            if reuse_age_factors:
                print("Reusing this run's global NMF-LR factors: identical graph bundle and parameters.")
                method_edge_features[method] = Factor_LR_list
            else:
                method_edge_features[method] = build_global_nmflr_factors(
                    SpiderNet_data_pyg_list,
                    n_components=NMF_RANK,
                    random_state=NMF_RANDOM_STATE,
                    max_iter=NMF_MAX_ITER,
                )
            method_feature_names[method] = [f"NMF-LR-{j+1}" for j in range(NMF_RANK)]
        elif method == "COMMOT":
            method_edge_features[method], method_feature_names[method] = load_commot_outputs_union()
        elif method == "ScCChain":
            method_edge_features[method], method_feature_names[method] = load_sccchain_outputs_union()
        elif method == "Spacia":
            method_edge_features[method], method_feature_names[method] = load_spacia_outputs()
        else:
            raise ValueError(f"Unknown method: {method}")

        validate_edge_feature_list(method, method_edge_features[method])
        print(f"{method}: loaded {len(method_edge_features[method])} slices; first shape = {np.asarray(method_edge_features[method][0]).shape}")
    except Exception as e:
        method_load_errors[method] = repr(e)
        if method in REQUIRE_METHODS_TO_LOAD or not SKIP_MISSING_BASELINES:
            raise RuntimeError(
                f"{method} was requested but could not be loaded. This would remove it from the final SMD barplots. "
                f"Original error: {repr(e)}"
            ) from e
        print(f"[Warning] Skipping {method}: {e}")

method_order_use = [m for m in METHODS_TO_RUN if m in method_edge_features]
print("Loaded methods:", method_order_use)
if method_load_errors:
    print("Method load errors:")
    for _m, _err in method_load_errors.items():
        print(f"  {_m}: {_err}")

In [ ]:
# ============================================================
# B7.8. Compute SMD scans and select dimensions
# ============================================================
# Outcome definitions:
#   T-cell outcome:
#       Group T cells by outgoing aggregated T-cell -> receiver strength.
#
#   Receiver outcome:
#       Group receiver cells by their own incoming aggregated T-cell -> receiver
#       strength for the same communication dimension.
#
#   The receiver outcome no longer assigns receiver cells according to whether
#   they neighbor High-group or Low-group T cells.

def aggregate_outgoing_tcell_feature(edge_features, edge_index_e2, celltype_cur, num_cells, agg="sum"):
    """Aggregate edge features over outgoing T-cell sender edges to the sender T cells."""
    edge_features = np.asarray(edge_features, dtype=np.float32)
    if edge_features.ndim == 1:
        edge_features = edge_features.reshape(-1, 1)

    edge_features = np.nan_to_num(edge_features, nan=0.0, posinf=0.0, neginf=0.0)
    sender = edge_index_e2[:, 0].astype(np.int64)
    receiver = edge_index_e2[:, 1].astype(np.int64)

    valid = (sender >= 0) & (sender < num_cells) & (receiver >= 0) & (receiver < num_cells)

    sender_is_tcell = np.zeros(edge_index_e2.shape[0], dtype=bool)
    sender_is_tcell[valid] = np.asarray(celltype_cur, dtype=str)[sender[valid]] == T_CELL_LABEL

    K = edge_features.shape[1]
    out = np.zeros((num_cells, K), dtype=np.float64)
    count = np.zeros((num_cells, K), dtype=np.float64)

    if np.any(sender_is_tcell):
        s = sender[sender_is_tcell]
        ef = edge_features[sender_is_tcell, :]

        if agg in ["sum", "mean"]:
            np.add.at(out, s, ef)
            np.add.at(count, s, (ef > 0).astype(np.float64))
            if agg == "mean":
                out = np.divide(out, count, out=np.zeros_like(out), where=count > 0)
        elif agg == "max":
            np.maximum.at(out, s, ef)
        else:
            raise ValueError(f"Unknown T_CELL_OUTGOING_AGG={agg}")

    return out.astype(np.float32), sender_is_tcell


def aggregate_incoming_receiver_feature_from_tcell(edge_features, edge_index_e2, celltype_cur, num_cells, agg="sum"):
    """
    Aggregate edge features over incoming T-cell -> receiver edges to receiver cells.

    This is the receiver-side analog of aggregate_outgoing_tcell_feature:
        receiver incoming score for dimension k
        = aggregate over all edges (T cell sender -> this receiver) of feature k.
    """
    edge_features = np.asarray(edge_features, dtype=np.float32)
    if edge_features.ndim == 1:
        edge_features = edge_features.reshape(-1, 1)

    edge_features = np.nan_to_num(edge_features, nan=0.0, posinf=0.0, neginf=0.0)
    sender = edge_index_e2[:, 0].astype(np.int64)
    receiver = edge_index_e2[:, 1].astype(np.int64)

    valid = (sender >= 0) & (sender < num_cells) & (receiver >= 0) & (receiver < num_cells)

    sender_is_tcell = np.zeros(edge_index_e2.shape[0], dtype=bool)
    sender_is_tcell[valid] = np.asarray(celltype_cur, dtype=str)[sender[valid]] == T_CELL_LABEL

    K = edge_features.shape[1]
    incoming = np.zeros((num_cells, K), dtype=np.float64)
    count = np.zeros((num_cells, K), dtype=np.float64)

    if np.any(sender_is_tcell):
        r = receiver[sender_is_tcell]
        ef = edge_features[sender_is_tcell, :]

        if agg in ["sum", "mean"]:
            np.add.at(incoming, r, ef)
            np.add.at(count, r, (ef > 0).astype(np.float64))
            if agg == "mean":
                incoming = np.divide(incoming, count, out=np.zeros_like(incoming), where=count > 0)
        elif agg == "max":
            np.maximum.at(incoming, r, ef)
        else:
            raise ValueError(f"Unknown RECEIVER_INCOMING_AGG={agg}")

    return incoming.astype(np.float32), sender_is_tcell


def _receiver_eval_mask_from_filter(receiver_values_all, method):
    """Build receiver evaluation mask from incoming aggregated values."""
    receiver_values_all = np.asarray(receiver_values_all, dtype=float)

    if method == "SpiderNet":
        eval_filter = SPIDERNET_RECEIVER_EVAL_FILTER
    else:
        eval_filter = BASELINE_RECEIVER_EVAL_FILTER

    if eval_filter == "positive_incoming":
        eval_receiver_mask = np.isfinite(receiver_values_all) & (receiver_values_all > 0)
    elif eval_filter == "all_finite":
        eval_receiver_mask = np.isfinite(receiver_values_all)
    else:
        raise ValueError(f"Unknown receiver evaluation filter: {eval_filter}")

    return eval_receiver_mask, eval_filter


def _build_high_low_masks_from_values(values_all, eval_mask, split_rule=SPLIT_RULE):
    """
    Split evaluated cells into High/Low based on the supplied values.
    Returns global high/low masks and the median threshold used for reporting.
    """
    values_all = np.asarray(values_all, dtype=float)
    eval_mask = np.asarray(eval_mask, dtype=bool)

    high_global_mask = np.zeros(values_all.shape[0], dtype=bool)
    low_global_mask = np.zeros(values_all.shape[0], dtype=bool)

    values_eval = values_all[eval_mask]

    if values_eval.size == 0:
        return high_global_mask, low_global_mask, np.nan

    high_rel_mask = _high_mask_from_split_rule(values_eval, split_rule=split_rule)
    eval_indices = np.where(eval_mask)[0]

    high_global_mask[eval_indices[high_rel_mask]] = True
    low_global_mask[eval_indices[~high_rel_mask]] = True

    threshold = float(np.nanmedian(values_eval))
    return high_global_mask, low_global_mask, threshold


def scan_method_tcell_sender_smd(method, edge_feature_list):
    # First pass:
    #   outgoing_all: T-cell sender-side aggregate values for T-cell grouping.
    #   incoming_all: receiver-side aggregate values from incoming T-cell edges for receiver grouping.
    K = np.asarray(edge_feature_list[0]).shape[1]
    outgoing_parts = []
    incoming_parts = []
    tcell_mask_parts = []

    for slice_index, edge_features in enumerate(edge_feature_list):
        adata_sub = adata_list[slice_index]
        data = SpiderNet_data_pyg_list[slice_index]
        edge_index = edge_index_to_e2(get_data_item(data, "edge_index"))
        num_cells = get_num_cells_from_data(data, adata_sub)
        celltype_cur = adata_sub.obs[CELLTYPE_COL].astype(str).to_numpy()

        outgoing_cur, _ = aggregate_outgoing_tcell_feature(
            edge_features=edge_features,
            edge_index_e2=edge_index,
            celltype_cur=celltype_cur,
            num_cells=num_cells,
            agg=T_CELL_OUTGOING_AGG,
        )
        incoming_cur, _ = aggregate_incoming_receiver_feature_from_tcell(
            edge_features=edge_features,
            edge_index_e2=edge_index,
            celltype_cur=celltype_cur,
            num_cells=num_cells,
            agg=RECEIVER_INCOMING_AGG,
        )

        outgoing_parts.append(outgoing_cur)
        incoming_parts.append(incoming_cur)
        tcell_mask_parts.append(celltype_cur == T_CELL_LABEL)

    outgoing_all = np.vstack(outgoing_parts)
    incoming_all = np.vstack(incoming_parts)
    tcell_mask_all = np.concatenate(tcell_mask_parts)

    if outgoing_all.shape[0] != aging_score_all.shape[0]:
        raise ValueError(f"{method}: outgoing matrix rows {outgoing_all.shape[0]} != aging score length {aging_score_all.shape[0]}")
    if incoming_all.shape[0] != aging_score_all.shape[0]:
        raise ValueError(f"{method}: incoming matrix rows {incoming_all.shape[0]} != aging score length {aging_score_all.shape[0]}")

    rows = []

    for j in range(K):
        feature_name = (
            method_feature_names.get(method, [])[j]
            if j < len(method_feature_names.get(method, []))
            else f"Feature-{j+1}"
        )

        # ------------------------------------------------------
        # Outcome 1: T-cell aging SMD
        # Group T cells by outgoing aggregated T-cell -> receiver strength.
        # ------------------------------------------------------
        tcell_feature_values_all = outgoing_all[:, j].astype(float)
        if method == "SpiderNet":
            tcell_eval_filter = SPIDERNET_TCELL_EVAL_FILTER
        else:
            tcell_eval_filter = BASELINE_TCELL_EVAL_FILTER

        if tcell_eval_filter == "positive_outgoing":
            eval_tcell_mask = (
                tcell_mask_all
                & np.isfinite(tcell_feature_values_all)
                & (tcell_feature_values_all > 0)
            )
        elif tcell_eval_filter == "all_finite":
            eval_tcell_mask = tcell_mask_all & np.isfinite(tcell_feature_values_all)
        else:
            raise ValueError(f"Unknown T-cell evaluation filter: {tcell_eval_filter}")

        high_tcell_global_mask, low_tcell_global_mask, tcell_threshold = _build_high_low_masks_from_values(
            values_all=tcell_feature_values_all,
            eval_mask=eval_tcell_mask,
            split_rule=SPLIT_RULE,
        )

        tcell_stats = compute_smd_from_groups(
            aging_score_all[high_tcell_global_mask],
            aging_score_all[low_tcell_global_mask],
            min_cells_per_group=MIN_CELLS_PER_GROUP,
        )

        rows.append({
            "Method": method,
            "Outcome": "Tcell_AgingGeneExp",
            "Feature_Dim": j + 1,
            "Feature_Name": feature_name,
            "Threshold": tcell_threshold,
            "Threshold_Mode": "median",
            "Grouping_Rule": SPLIT_RULE,
            "Grouping_Source": "outgoing_Tcell_to_receiver_aggregate",
            "Sender_Celltype": T_CELL_LABEL,
            "Receiver_Celltype_Filter": "any",
            "T_Cell_Outgoing_Agg": T_CELL_OUTGOING_AGG,
            "T_Cell_Eval_Filter": tcell_eval_filter,
            "Receiver_Incoming_Agg": RECEIVER_INCOMING_AGG,
            "Receiver_Eval_Filter": np.nan,
            "Ambiguous_Receiver_Policy": "not_used",
            **tcell_stats,
        })

        # ------------------------------------------------------
        # Outcome 2: Receiver aging SMD
        # Group receiver cells by incoming aggregated T-cell -> receiver strength.
        # This is independent of the T-cell High/Low grouping above.
        # ------------------------------------------------------
        receiver_feature_values_all = incoming_all[:, j].astype(float)
        eval_receiver_mask, receiver_eval_filter = _receiver_eval_mask_from_filter(
            receiver_feature_values_all,
            method=method,
        )

        high_receiver_global_mask, low_receiver_global_mask, receiver_threshold = _build_high_low_masks_from_values(
            values_all=receiver_feature_values_all,
            eval_mask=eval_receiver_mask,
            split_rule=SPLIT_RULE,
        )

        receiver_stats = compute_smd_from_groups(
            aging_score_all[high_receiver_global_mask],
            aging_score_all[low_receiver_global_mask],
            min_cells_per_group=MIN_CELLS_PER_GROUP,
        )

        rows.append({
            "Method": method,
            "Outcome": "Receiver_AgingGeneExp",
            "Feature_Dim": j + 1,
            "Feature_Name": feature_name,
            "Threshold": receiver_threshold,
            "Threshold_Mode": "median",
            "Grouping_Rule": SPLIT_RULE,
            "Grouping_Source": "incoming_Tcell_to_receiver_aggregate",
            "Sender_Celltype": T_CELL_LABEL,
            "Receiver_Celltype_Filter": "any",
            "T_Cell_Outgoing_Agg": T_CELL_OUTGOING_AGG,
            "T_Cell_Eval_Filter": tcell_eval_filter,
            "Receiver_Incoming_Agg": RECEIVER_INCOMING_AGG,
            "Receiver_Eval_Filter": receiver_eval_filter,
            "Ambiguous_Receiver_Policy": "not_used",
            "n_receiver_evaluated": int(eval_receiver_mask.sum()),
            **receiver_stats,
        })

    return pd.DataFrame(rows), outgoing_all, incoming_all, tcell_mask_all


all_scan_parts = []
method_aux = {}

for method in method_order_use:
    print("\n==============================")
    print("Scanning method:", method)
    scan_df, outgoing_all, incoming_all, tcell_mask_all = scan_method_tcell_sender_smd(
        method,
        method_edge_features[method],
    )
    all_scan_parts.append(scan_df)
    method_aux[method] = {
        "outgoing_all": outgoing_all,
        "incoming_all": incoming_all,
        "tcell_mask_all": tcell_mask_all,
    }

all_scan_df = pd.concat(all_scan_parts, ignore_index=True)

# Rank dimensions within each outcome and method. Larger positive SMD is better.
# SpiderNet is still scanned for diagnostics, but final selected SpiderNet dimension is forced to MI-29.
all_scan_df["SMD_rank_within_outcome"] = np.nan
for method in all_scan_df["Method"].unique():
    for outcome in all_scan_df["Outcome"].unique():
        mask = all_scan_df["Method"].eq(method) & all_scan_df["Outcome"].eq(outcome)
        all_scan_df.loc[mask, "SMD_rank_within_outcome"] = all_scan_df.loc[mask, "SMD"].rank(
            method="average",
            ascending=False,
            na_option="keep",
        )

feature_summary_df = (
    all_scan_df
    .groupby(["Method", "Feature_Dim", "Feature_Name"], as_index=False)
    .agg(
        mean_SMD_across_outcomes=("SMD", "mean"),
        median_SMD_across_outcomes=("SMD", "median"),
        mean_SMD_rank_across_outcomes=("SMD_rank_within_outcome", "mean"),
        max_SMD_rank_across_outcomes=("SMD_rank_within_outcome", "max"),
        valid_outcome_count=("SMD", lambda x: int(np.isfinite(x).sum())),
        valid_rank_count=("SMD_rank_within_outcome", lambda x: int(np.isfinite(x).sum())),
    )
)
feature_summary_df["n_outcomes"] = 2
feature_summary_df["Feature_Selection_Mode"] = np.where(
    feature_summary_df["Method"].eq("SpiderNet"),
    f"fixed_{SPIDERNET_FIXED_FEATURE_NAME}",
    BASELINE_FEATURE_SELECTION_MODE,
)

selected_rows = []
for method in method_order_use:
    cand_all = feature_summary_df.loc[feature_summary_df["Method"].eq(method)].copy()

    if method == "SpiderNet":
        fixed = cand_all.loc[cand_all["Feature_Dim"].astype(int).eq(SPIDERNET_FIXED_DIM_ONE_BASED)].copy()
        if fixed.empty:
            raise ValueError(
                f"SpiderNet fixed dimension {SPIDERNET_FIXED_DIM_ONE_BASED} was not found in feature_summary_df. "
                f"Available dimensions: {sorted(cand_all['Feature_Dim'].astype(int).unique().tolist())}"
            )
        if REQUIRE_BOTH_OUTCOMES_FOR_SELECTION and (
            int(fixed.iloc[0]["valid_outcome_count"]) < 2 or int(fixed.iloc[0]["valid_rank_count"]) < 2
        ):
            print(
                f"[Warning] SpiderNet {SPIDERNET_FIXED_FEATURE_NAME} does not have both outcomes valid; "
                "it is still kept because SpiderNet is fixed by design."
            )
        selected_rows.append(fixed.iloc[0])
        continue

    cand = cand_all.copy()
    if REQUIRE_BOTH_OUTCOMES_FOR_SELECTION:
        cand = cand.loc[(cand["valid_outcome_count"] == 2) & (cand["valid_rank_count"] == 2)].copy()
    if cand.empty:
        print(f"[Warning] {method}: no dimension with both outcomes valid; using best available dimension.")
        cand = cand_all.copy()
    cand_ranked = cand.loc[np.isfinite(cand["mean_SMD_rank_across_outcomes"].to_numpy(dtype=float))].copy()
    if cand_ranked.empty:
        # Keep the method visible in the selected summary/plot instead of silently dropping it.
        # This usually means all dimensions produced NaN SMDs, often because the method is very sparse
        # for T-cell sender edges. The final barplot will show this selected feature as NA.
        print(f"[Warning] {method}: no dimension has finite SMD rank; keeping best-available/first feature as NA.")
        if cand_all.empty:
            continue
        best = cand_all.sort_values(
            ["valid_outcome_count", "mean_SMD_across_outcomes", "Feature_Dim"],
            ascending=[False, False, True],
            na_position="last",
        ).iloc[0]
    else:
        best = cand_ranked.sort_values(
            ["mean_SMD_rank_across_outcomes", "max_SMD_rank_across_outcomes", "mean_SMD_across_outcomes"],
            ascending=[True, True, False],
            na_position="last",
        ).iloc[0]
    selected_rows.append(best)

selected_feature_df = pd.DataFrame(selected_rows)
selected_feature_df["Feature_Dim"] = selected_feature_df["Feature_Dim"].astype(int)

selected_summary_df = all_scan_df.merge(
    selected_feature_df[["Method", "Feature_Dim"]],
    on=["Method", "Feature_Dim"],
    how="inner",
)
selected_summary_df = selected_summary_df.merge(
    feature_summary_df,
    on=["Method", "Feature_Dim", "Feature_Name"],
    how="left",
    suffixes=("", "_feature"),
)

all_scan_path = OUT_DIR / "AgingBrain_Tcell_sender_any_receiver_CCC_all_dimension_SMD_scan.csv"
feature_summary_path = OUT_DIR / "AgingBrain_Tcell_sender_any_receiver_CCC_feature_selection_summary.csv"
selected_summary_path = OUT_DIR / "AgingBrain_Tcell_sender_any_receiver_CCC_selected_dimension_SMD_summary.csv"

all_scan_df.to_csv(output_path(all_scan_path), index=False)
feature_summary_df.to_csv(output_path(feature_summary_path), index=False)
selected_summary_df.to_csv(output_path(selected_summary_path), index=False)

print("Saved all-dimension SMD scan:", all_scan_path)
print("Saved feature-selection summary:", feature_summary_path)
print("Saved selected-dimension SMD summary:", selected_summary_path)

display(selected_feature_df.sort_values("Method"))
display(selected_summary_df.sort_values(["Method", "Outcome"]))


In [ ]:
# ============================================================
# B7.9. Build selected-dimension long tables for diagnostics
# ============================================================

def build_selected_long_for_method(method, selected_dim_one_based):
    j = int(selected_dim_one_based) - 1

    outgoing_all = method_aux[method]["outgoing_all"]
    incoming_all = method_aux[method]["incoming_all"]
    tcell_mask_all = method_aux[method]["tcell_mask_all"]

    feature_name = (
        method_feature_names.get(method, [])[j]
        if j < len(method_feature_names.get(method, []))
        else f"Feature-{selected_dim_one_based}"
    )

    # --------------------------------------------------------
    # T-cell long table:
    # group T cells by outgoing aggregated T-cell -> receiver strength.
    # --------------------------------------------------------
    tcell_feature_values_all = outgoing_all[:, j].astype(float)

    if method == "SpiderNet":
        tcell_eval_filter = SPIDERNET_TCELL_EVAL_FILTER
    else:
        tcell_eval_filter = BASELINE_TCELL_EVAL_FILTER

    if tcell_eval_filter == "positive_outgoing":
        eval_tcell_mask = (
            tcell_mask_all
            & np.isfinite(tcell_feature_values_all)
            & (tcell_feature_values_all > 0)
        )
    elif tcell_eval_filter == "all_finite":
        eval_tcell_mask = tcell_mask_all & np.isfinite(tcell_feature_values_all)
    else:
        raise ValueError(f"Unknown T-cell evaluation filter: {tcell_eval_filter}")

    high_tcell_global_mask, low_tcell_global_mask, _ = _build_high_low_masks_from_values(
        values_all=tcell_feature_values_all,
        eval_mask=eval_tcell_mask,
        split_rule=SPLIT_RULE,
    )

    tcell_mask = high_tcell_global_mask | low_tcell_global_mask
    tcell_long = pd.DataFrame({
        "Method": method,
        "Outcome": "Tcell_AgingGeneExp",
        "Feature_Dim": selected_dim_one_based,
        "Feature_Name": feature_name,
        "global_cell_index": np.where(tcell_mask)[0],
    })
    tcell_long["Group"] = np.where(
        high_tcell_global_mask[tcell_long["global_cell_index"].to_numpy()],
        "High",
        "Low",
    )
    tcell_long["Score"] = aging_score_all[tcell_long["global_cell_index"].to_numpy()]
    tcell_long["Feature_Value"] = tcell_feature_values_all[tcell_long["global_cell_index"].to_numpy()]
    tcell_long["Grouping_Source"] = "outgoing_Tcell_to_receiver_aggregate"
    tcell_long["T_Cell_Eval_Filter"] = tcell_eval_filter
    tcell_long["Receiver_Eval_Filter"] = np.nan

    # --------------------------------------------------------
    # Receiver long table:
    # group receiver cells by incoming aggregated T-cell -> receiver strength.
    # This is independent of the T-cell High/Low groups.
    # --------------------------------------------------------
    receiver_feature_values_all = incoming_all[:, j].astype(float)
    eval_receiver_mask, receiver_eval_filter = _receiver_eval_mask_from_filter(
        receiver_feature_values_all,
        method=method,
    )

    high_receiver_global_mask, low_receiver_global_mask, _ = _build_high_low_masks_from_values(
        values_all=receiver_feature_values_all,
        eval_mask=eval_receiver_mask,
        split_rule=SPLIT_RULE,
    )

    receiver_mask = high_receiver_global_mask | low_receiver_global_mask
    receiver_long = pd.DataFrame({
        "Method": method,
        "Outcome": "Receiver_AgingGeneExp",
        "Feature_Dim": selected_dim_one_based,
        "Feature_Name": feature_name,
        "global_cell_index": np.where(receiver_mask)[0],
    })
    receiver_long["Group"] = np.where(
        high_receiver_global_mask[receiver_long["global_cell_index"].to_numpy()],
        "High",
        "Low",
    )
    receiver_long["Score"] = aging_score_all[receiver_long["global_cell_index"].to_numpy()]
    receiver_long["Feature_Value"] = receiver_feature_values_all[receiver_long["global_cell_index"].to_numpy()]
    receiver_long["Grouping_Source"] = "incoming_Tcell_to_receiver_aggregate"
    receiver_long["T_Cell_Eval_Filter"] = tcell_eval_filter
    receiver_long["Receiver_Eval_Filter"] = receiver_eval_filter

    out = pd.concat([tcell_long, receiver_long], ignore_index=True)
    out = out.merge(
        cell_meta_df[["global_cell_index", "barcode", "slice_index", "age", "celltype"]],
        on="global_cell_index",
        how="left",
    )
    out["Sender_Celltype"] = T_CELL_LABEL
    out["Receiver_Celltype_Filter"] = "any"
    out["T_Cell_Outgoing_Agg"] = T_CELL_OUTGOING_AGG
    out["Receiver_Incoming_Agg"] = RECEIVER_INCOMING_AGG
    out["Ambiguous_Receiver_Policy"] = "not_used"
    return out


selected_long_parts = []
for _, row in selected_feature_df.iterrows():
    selected_long_parts.append(build_selected_long_for_method(row["Method"], int(row["Feature_Dim"])))

selected_long_df = pd.concat(selected_long_parts, ignore_index=True) if selected_long_parts else pd.DataFrame()
selected_long_path = OUT_DIR / "AgingBrain_Tcell_sender_any_receiver_CCC_selected_dimension_score_long.csv"
selected_long_df.to_csv(output_path(selected_long_path), index=False)
print("Saved selected-dimension long table:", selected_long_path)
display(selected_long_df.head())

display(
    selected_long_df
    .groupby(["Method", "Outcome", "Group"], as_index=False)
    .size()
    .pivot_table(index=["Method", "Outcome"], columns="Group", values="size", fill_value=0)
)


In [ ]:
# ============================================================
# B7.10. Plot the selected-dimension SMDs
# ============================================================
set_plot_style()

OUTCOME_LABELS = {
    "Tcell_AgingGeneExp": "T-cell aging module SMD",
    "Receiver_AgingGeneExp": "Receiver-cell aging module SMD",
}

OUTCOME_FILE_STEMS = {
    "Tcell_AgingGeneExp": "AgingBrain_Tcell_sender_own_AgingGeneExp_SMD_barplot",
    "Receiver_AgingGeneExp": "AgingBrain_Tcell_sender_receiver_AgingGeneExp_SMD_barplot",
}

EXPECTED_METHOD_ORDER = ["SpiderNet", "NMF-LR", "COMMOT", "ScCChain", "Spacia"]
method_order_plot = [
    m for m in EXPECTED_METHOD_ORDER
    if (m in METHODS_TO_RUN) and (m in method_order_use or m in selected_summary_df["Method"].unique())
]
print("Methods loaded:", method_order_use)
print("Methods in selected_summary_df:", selected_summary_df["Method"].drop_duplicates().tolist())
print("Methods plotted:", method_order_plot)

if "ScCChain" in METHODS_TO_RUN and "ScCChain" not in method_order_plot:
    raise RuntimeError(
        "ScCChain is requested but is not available for plotting. Check the output of Cell 6/7: "
        "it either failed to load or produced no selected feature row."
    )


def _complete_plot_df_for_outcome(outcome):
    plot_df = selected_summary_df.loc[selected_summary_df["Outcome"].eq(outcome)].copy()
    completed_rows = []
    for method in method_order_plot:
        sub = plot_df.loc[plot_df["Method"].eq(method)].copy()
        if sub.empty:
            completed_rows.append({
                "Method": method,
                "Outcome": outcome,
                "SMD": np.nan,
                "Feature_Dim": np.nan,
                "Feature_Name": "NA",
                "plot_status": "no selected row",
            })
        else:
            # There should be one row per method/outcome after feature selection.
            row = sub.iloc[0].to_dict()
            row["plot_status"] = "ok" if np.isfinite(row.get("SMD", np.nan)) else "SMD is NA"
            completed_rows.append(row)
    out = pd.DataFrame(completed_rows)
    out["Method"] = pd.Categorical(out["Method"], categories=method_order_plot, ordered=True)
    return out.sort_values("Method")


def _feature_label_for_row(row):
    if pd.isna(row.get("Feature_Dim", np.nan)):
        return "NA"
    if row["Method"] == "SpiderNet":
        return str(row.get("Feature_Name", SPIDERNET_FIXED_FEATURE_NAME))
    return f"dim {int(row['Feature_Dim'])}"


def plot_smd_bar_for_outcome(outcome):
    plot_df = _complete_plot_df_for_outcome(outcome)

    fig_width = max(3.2, 0.72 * len(plot_df) + 1.3)
    fig, ax = plt.subplots(figsize=(fig_width, 3.0), facecolor="white")
    ax.set_facecolor("white")

    x = np.arange(plot_df.shape[0])
    finite_smd = plot_df["SMD"].to_numpy(dtype=float)
    finite_smd = finite_smd[np.isfinite(finite_smd)]
    if finite_smd.size:
        y_min = min(0.0, float(np.min(finite_smd)))
        y_max = max(0.0, float(np.max(finite_smd)))
        pad = max(0.08, 0.18 * (y_max - y_min if y_max > y_min else 1.0))
        ax.set_ylim(y_min - pad, y_max + pad)
    else:
        ax.set_ylim(-0.5, 0.5)

    for i, (_, row) in enumerate(plot_df.iterrows()):
        method = str(row["Method"])
        colors = METHOD_COLORS.get(method, {"fill": "#DDDDDD", "edge": "#333333"})
        y = row.get("SMD", np.nan)
        if np.isfinite(y):
            ax.bar(
                i,
                y,
                width=0.65,
                color=colors["fill"],
                edgecolor=colors["edge"],
                linewidth=1.0,
            )
            va = "bottom" if y >= 0 else "top"
            dy = 0.025 if y >= 0 else -0.025
            ax.text(i, y + dy, _feature_label_for_row(row), ha="center", va=va, fontsize=7, color="black")
        else:
            # Keep the method visible even if SMD is unavailable.
            ax.bar(
                i,
                0,
                width=0.65,
                color="white",
                edgecolor=colors["edge"],
                linewidth=1.0,
            )
            ax.text(i, 0, "NA", ha="center", va="bottom", fontsize=7, color="black")

    # Matplotlib does not recognize R-style color names like "grey55".
    # Use a grayscale string or a hex code instead.
    ax.axhline(0, color="0.55", linewidth=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df["Method"].astype(str).tolist(), rotation=35, ha="right")
    ax.set_ylabel("SMD")
    ax.set_title(OUTCOME_LABELS[outcome])

    sns.despine(ax=ax, top=True, right=True)
    ax.spines["left"].set_visible(True)
    ax.spines["bottom"].set_visible(True)
    ax.spines["left"].set_color("black")
    ax.spines["bottom"].set_color("black")

    note_lines = [f"{row['Method']}: {_feature_label_for_row(row)}" for _, row in plot_df.iterrows()]
    ax.text(
        1.02,
        0.98,
        "Selected dim\n" + "\n".join(note_lines),
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=7,
    )

    plt.tight_layout()
    pdf_path = OUT_DIR / f"{OUTCOME_FILE_STEMS[outcome]}.pdf"
    png_path = OUT_DIR / f"{OUTCOME_FILE_STEMS[outcome]}.png"
    fig.savefig(output_path(pdf_path), bbox_inches="tight", dpi=300)
    fig.savefig(output_path(png_path), bbox_inches="tight", dpi=300)
    print("Saved:", pdf_path)
    print("Saved:", png_path)
    display(plot_df[["Method", "Outcome", "Feature_Dim", "Feature_Name", "SMD", "plot_status"]])
    plt.show()
    plt.close(fig)


for outcome in ["Tcell_AgingGeneExp", "Receiver_AgingGeneExp"]:
    plot_smd_bar_for_outcome(outcome)

In [ ]:
# ============================================================
# B7.11. Sanity checks and output summary
# ============================================================
print("Methods included:", method_order_use)
print("Feature selection mode:", FEATURE_SELECTION_MODE)
print("SpiderNet fixed feature:", SPIDERNET_FIXED_FEATURE_NAME)
print("Receiver grouping source: incoming_Tcell_to_receiver_aggregate")
print("Receiver incoming aggregation:", RECEIVER_INCOMING_AGG)
print("SpiderNet receiver eval filter:", SPIDERNET_RECEIVER_EVAL_FILTER)
print("Baseline receiver eval filter:", BASELINE_RECEIVER_EVAL_FILTER)

print("\nSelected features:")
display(
    selected_feature_df
    .sort_values("Method")
    [["Method", "Feature_Dim", "Feature_Name", "Feature_Selection_Mode", "mean_SMD_across_outcomes", "mean_SMD_rank_across_outcomes", "valid_outcome_count"]]
)

print("\nSelected outcome SMDs:")
display(
    selected_summary_df
    .sort_values(["Outcome", "Method"])
    [["Method", "Outcome", "Feature_Dim", "Feature_Name", "n_high", "n_low", "mean_high", "mean_low", "SMD", "p_wilcox", "significance"]]
)

print("\nAll outputs are saved under:", OUT_DIR)

## Optional in silico perturbation analyses

These sections test how local predicted age changes after perturbing either neighborhood cell composition or MI-associated genes. The workflow is kept explicit so that each perturbation step can be inspected and modified directly.


## Optional C1. In silico cell replacement

In [ ]:
## Step 2. Predict age for the original data
predicted_age_pd_merge = pd.DataFrame({'cell_index': [],
                  "age_predicted": []})
for cellclass_cur in cellclass_unique:
    # print(cellclass_cur)
    ##
    cellindex_cur = np.where((celltype_all == cellclass_cur))[0]
    age_cur = np.array(cellage_all[cellindex_cur])
    # from sklearn.preprocessing import StandardScaler
    # scaler = StandardScaler()
    # X_train_scaled = scaler.fit_transform(MI_agg_merge[cellindex_cur,:])
    X_train_scaled = MI_SR_agg_all[cellindex_cur,:]
    ##
    reg_scale_cur = reg_scale_dict[cellclass_cur]
    ##
    predicted_age_pd = pd.DataFrame({'cell_index': cellindex_cur,
                  "age_predicted": reg_scale_cur.predict(X_train_scaled)})
    predicted_age_pd_merge = pd.concat([predicted_age_pd_merge,predicted_age_pd])
predicted_age_pd_merge['cell_index'] = predicted_age_pd_merge['cell_index'].astype(int)
## Sort rows by cell index
predicted_age_pd_merge = predicted_age_pd_merge.sort_values(by = ['cell_index'])
predicted_age_pd_merge['true_age'] = cellage_all[predicted_age_pd_merge['cell_index'].values]
print(predicted_age_pd_merge)

In [ ]:
genepanel_table = pd.read_excel(input_path(str(DATA_ROOT) + "/Supp_table/2023-12-22736D-TableS1_MERFISHGenePanel.xlsx"))
ration = np.array(genepanel_table['Rationale for inclusion'])
ration_sep = []
ration_sep_extend = []
for i in range(len(ration)):
    ration_cur = ration[i]
    ## Separate the ration_cur by the "; "
    ration_cur_split = ration_cur.split("; ")
    ration_sep.append(ration_cur_split)
    ration_sep_extend.extend(ration_cur_split)
ration_sep_unique = np.unique(ration_sep_extend)
ration_sep_unique_aging = [ration_sep_unique[i] for i in range(len(ration_sep_unique)) if "aging" in ration_sep_unique[i] or "senescence" in ration_sep_unique[i]]
marker_genes_dict = {}
for i in range(len(ration_sep_unique)):
    marker_genes_dict[ration_sep_unique[i]] = []
    for j in range(len(ration_sep)):
        if ration_sep_unique[i] in ration_sep[j]:
            marker_genes_dict[ration_sep_unique[i]].append(genepanel_table['Vizgen Gene'][j])
marker_genes_dict["aging_all"] = np.unique(marker_genes_dict[ration_sep_unique_aging[0]] + marker_genes_dict[ration_sep_unique_aging[1]] + marker_genes_dict[ration_sep_unique_aging[2]] + marker_genes_dict[ration_sep_unique_aging[3]]).tolist()
genenames = processed.spidernet_data[0]['genenames']
aging_genes_index = np.where(np.isin(genenames, marker_genes_dict["aging_all"]))[0]
aging_genes = genenames[aging_genes_index]


In [ ]:
T_cell_index = np.where(celltype_all == 'T cell')[0]
non_T_cell_index = np.where(celltype_all != 'T cell')[0]
age_T_cell_index = np.array(cellage_all)[T_cell_index]
Astrocyte_cell_index = np.where(celltype_all == 'Astrocyte')[0]
non_Astrocyte_cell_index = np.where(celltype_all != 'Astrocyte')[0]
age_Astrocyte_cell_index = np.array(cellage_all)[Astrocyte_cell_index]
Endo_cell_index = np.where(celltype_all == 'Endothelial')[0]
non_Endo_cell_index = np.where(celltype_all != 'Endothelial')[0]
age_Endo_cell_index = np.array(cellage_all)[Endo_cell_index]
NSC_index = np.where(celltype_all == 'NSC')[0]
non_NSC_index = np.where(celltype_all != 'NSC')[0]
age_NSC_index = np.array(cellage_all)[NSC_index]
cellindex_dict = {'T cell': T_cell_index,
                      'Astrocyte': Astrocyte_cell_index,
                      'Endothelial': Endo_cell_index,
                  'NSC': NSC_index}

In [ ]:
num_replacecell_perslice = 500

In [ ]:
permute_celltype_list = ['T cell', 'Astrocyte', 'Endothelial']

In [ ]:
celltype_onehot = [processed.spidernet_data[i]['cell_class_onehot'].to("cpu").numpy() for i in range(len(processed.spidernet_data))]
celltype_onehot_all = np.vstack(celltype_onehot)
print(celltype_onehot_all.shape)

cellsize_list = [processed.spidernet_data[i].x.shape[0] for i in range(len(processed.spidernet_data))]
startcellindex_list = np.cumsum([0] + cellsize_list[:-1])
print(startcellindex_list)

In [ ]:
MI_OI = "MI29"
# MI_OI = "MI22"
SR_type = "Receiver"
MI_OI_index = int(MI_OI.split("MI")[1]) - 1
if SR_type == "Receiver":
    MI_OI_index = MI_OI_index + DIM_ENVIR
print(MI_OI_index)

In [ ]:
neighoringcell_age_change_dict = {}
# for permute_celltype_cur in permute_celltype_list:
diff_mean_all_dict = {}
aging_geneexp_all_dict = {}
MI_OI_change_mean_all_dict = {}
for permute_celltype_cur in permute_celltype_list:
    SpiderNet_data_pyg_list_perturbation = pd.read_pickle(input_path(str(run_dirs['run_dir']) + '/SpiderNet_data_pyg_list.pkl'))
    SpiderNet_data_pyg_list_perturbation = [SpiderNet_data_pyg_list_perturbation[i].to(device) for i in range(len(SpiderNet_data_pyg_list_perturbation))]
    ##
    neighboring_cells_index_all = []
    MI_SR_agg_all_perturbation = []
    cellindex_curpermute_celltype_all = []
    exp_recon_perturb_all = []
    for slice_index in range(len(SpiderNet_data_pyg_list_perturbation)):
        cellindex_notcurpermute_celltype = np.where(np.array(SpiderNet_data_pyg_list_perturbation[slice_index]['cell_class']) != permute_celltype_cur)[0]
        if len(cellindex_notcurpermute_celltype) > num_replacecell_perslice:
            cellindex_curpermute_celltype = np.random.choice(cellindex_notcurpermute_celltype, (num_replacecell_perslice),replace=False)
        else:
            cellindex_curpermute_celltype = cellindex_notcurpermute_celltype
        cellindex_curpermute_celltype_all.append(cellindex_curpermute_celltype)
        ##Get the neighboring cell index of cellindex_curpermute_celltype
        edge_index_cur = SpiderNet_data_pyg_list_perturbation[slice_index]['edge_index']
        edge_index_cur_np = edge_index_cur.to("cpu").numpy()
        neighboring_cells = np.hstack([edge_index_cur_np[:,0][np.isin(edge_index_cur_np[:,1], cellindex_curpermute_celltype)],
                                       edge_index_cur_np[:,1][np.isin(edge_index_cur_np[:,0], cellindex_curpermute_celltype)]]).flatten()
        neighboring_cells = np.unique(neighboring_cells)
        neighboring_cells_index = np.zeros(len(SpiderNet_data_pyg_list_perturbation[slice_index]['cell_class']))
        neighboring_cells_index[neighboring_cells] = 1
        neighboring_cells_index_all.append(neighboring_cells_index)
        ##Get the nei
        ##
        num_cellindex_curpermute_celltype = len(cellindex_curpermute_celltype)
        cellindex_useforplace_global = cellindex_dict[permute_celltype_cur]
        cellindex_useforplace_global = cellindex_useforplace_global[np.random.choice(len(cellindex_useforplace_global), (num_cellindex_curpermute_celltype),replace=False)]
        ##
        exp_cellindex_useforplace_global = geneexp_all[cellindex_useforplace_global, :]
        cellclass_one__useforplace_global =  celltype_onehot_all[cellindex_useforplace_global, :]
        ##Replace the elements in SpiderNet_data_pyg_list_perturbation[slice_index]
        SpiderNet_data_pyg_list_perturbation[slice_index]['x'][cellindex_curpermute_celltype, :] = torch.tensor(exp_cellindex_useforplace_global, dtype=torch.float32).to(device)
        SpiderNet_data_pyg_list_perturbation[slice_index]['cell_class_onehot'][cellindex_curpermute_celltype, :] = torch.tensor(cellclass_one__useforplace_global, dtype=torch.float32).to(device)
        ##Infer the MI for the perturbed data
        model.eval()
        exp_recon_perturb, _, _, _, Factor_envir_curbatch_perturbation, _, _, _ = model(SpiderNet_data_pyg_list_perturbation[slice_index].to(device))
        exp_recon_perturb_all.append(exp_recon_perturb.to("cpu").detach().numpy())
        ##
        Factor_envir_curbatch_perturbation = Factor_envir_curbatch_perturbation.to(device)
        edge_index_cur = SpiderNet_data_pyg_list_perturbation[slice_index]['edge_index']
        num_cell_cur = SpiderNet_data_pyg_list_perturbation[slice_index].x.shape[0]
        ##
        MI_receiver_agg_cur = scatter_mean(torch.tensor(Factor_envir_curbatch_perturbation).to(device),
                                       edge_index_cur[:, 1].to(torch.int64), dim=0,
                                       dim_size=num_cell_cur).to("cpu").numpy()
        MI_sender_agg_cur = scatter_mean(torch.tensor(Factor_envir_curbatch_perturbation).to(device),
                                     edge_index_cur[:, 0].to(torch.int64), dim=0,
                                     dim_size=num_cell_cur).to("cpu").numpy()
        MI_SR_agg_cur_perturbation = np.hstack([MI_sender_agg_cur, MI_receiver_agg_cur])
        MI_SR_agg_all_perturbation.append(MI_SR_agg_cur_perturbation)
    MI_SR_agg_all_perturbation = np.vstack(MI_SR_agg_all_perturbation)
    neighboring_cells_index_all = np.hstack(neighboring_cells_index_all)
    exp_recon_perturb_all = np.vstack(exp_recon_perturb_all)
    MI_SR_agg_all_perturbation_OI = MI_SR_agg_all_perturbation[:, MI_OI_index]
    ##
    ##3.2 Extract the predicted age for the perturbation data
    predicted_age_pd_merge_perturbation = pd.DataFrame({'cell_index': [],
                      "age_predicted": []})
    for cellclass_cur in cellclass_unique:
        # print(cellclass_cur)
        ##
        cellindex_cur = np.where((celltype_all == cellclass_cur))[0]
        age_cur = np.array(cellage_all[cellindex_cur])
        X_train_scaled = MI_SR_agg_all_perturbation[cellindex_cur,:]
        ##
        reg_scale_cur = reg_scale_dict[cellclass_cur]
        ##
        predicted_age_pd = pd.DataFrame({'cell_index': cellindex_cur,
                      "age_predicted": reg_scale_cur.predict(X_train_scaled)})
        predicted_age_pd_merge_perturbation = pd.concat([predicted_age_pd_merge_perturbation,predicted_age_pd])
    predicted_age_pd_merge_perturbation['cell_index'] = predicted_age_pd_merge_perturbation['cell_index'].astype(int)
    ##Sort the row of predicted_age_pd_merge_perturbation by the cell_index
    predicted_age_pd_merge_perturbation = predicted_age_pd_merge_perturbation.sort_values(by = ['cell_index'])
    predicted_age_pd_merge_perturbation['true_age'] = cellage_all[predicted_age_pd_merge_perturbation['cell_index'].values]
    predicted_age_pd_merge_perturbation['is_neighboring_perturbcell'] = neighboring_cells_index_all[predicted_age_pd_merge_perturbation['cell_index'].values]
    # print(predicted_age_pd_merge_perturbation)
    # print(np.sum(predicted_age_pd_merge_perturbation['is_neighboring_perturbcell']))
    predicted_age_pd_merge_perturbation_is_neighboring_perturbcell = predicted_age_pd_merge_perturbation[predicted_age_pd_merge_perturbation['is_neighboring_perturbcell'] == 1]
    predicted_age_pd_merge_ori_is_neighboring_perturbcell = predicted_age_pd_merge[predicted_age_pd_merge_perturbation['is_neighboring_perturbcell'] == 1]
    # print(predicted_age_pd_merge_perturbation_is_neighboring_perturbcell)
    # print(predicted_age_pd_merge_ori_is_neighboring_perturbcell)
    neighoringcell_age_change = predicted_age_pd_merge_perturbation_is_neighboring_perturbcell['age_predicted'].values - predicted_age_pd_merge_ori_is_neighboring_perturbcell['age_predicted'].values
    # print(np.quantile(neighoringcell_age_change, [0.05, 0.25, 0.5, 0.75, 0.95]))
    neighoringcell_age_change_dict[permute_celltype_cur] = neighoringcell_age_change
    ##
    diff_mean_all = []
    diff_aginggeneexp_mean_all = []
    MI_OI_change_mean_all = []
    for slice_index in range(len(SpiderNet_data_pyg_list_perturbation)):
        start_index_cur = startcellindex_list[slice_index]
        end_index_cur = startcellindex_list[slice_index] + cellsize_list[slice_index]
        predicted_age_pd_merge_perturbation_slice = predicted_age_pd_merge_perturbation.iloc[start_index_cur:end_index_cur, :]
        predicted_age_pd_merge_ori_slice = predicted_age_pd_merge.iloc[start_index_cur:end_index_cur, :]
        cellindex_curpermute_celltype_slice = cellindex_curpermute_celltype_all[slice_index]
        MI_SR_agg_all_perturbation_OI_cur = MI_SR_agg_all_perturbation_OI[start_index_cur:end_index_cur]
        MI_SR_agg_all_ori_OI_cur = MI_SR_agg_all[ start_index_cur:end_index_cur, MI_OI_index]
        ##
        exp_recon_perturb_all_slice = exp_recon_perturb_all[start_index_cur:end_index_cur, :]
        exprecon_use_slice = geneexp_all[start_index_cur:end_index_cur, :]
        exp_recon_perturb_all_slice_aginggene = exp_recon_perturb_all_slice[:, aging_genes_index]
        exprecon_use_slice_aginggene = exprecon_use_slice[:, aging_genes_index]
        ##
        diff_mean_list = []
        diff_aginggeneexp_mean_list = []
        MI_OI_change_mean_list = []
        for relcellindex in range(len(cellindex_curpermute_celltype_slice)):
            neighboring_cells_index_curcell0 = np.where(SpiderNet_data_pyg_list_perturbation[slice_index]['edge_index'].to("cpu").numpy()[ :,0] == cellindex_curpermute_celltype_slice[relcellindex])[0]
            neighboring_cells_index_curcell1 = np.where(SpiderNet_data_pyg_list_perturbation[slice_index]['edge_index'].to("cpu").numpy()[ :,1] == cellindex_curpermute_celltype_slice[relcellindex])[0]
            neighboring_cells_index_curcell = np.hstack([SpiderNet_data_pyg_list_perturbation[slice_index]['edge_index'].to("cpu").numpy()[ neighboring_cells_index_curcell0,1],
                                                        SpiderNet_data_pyg_list_perturbation[slice_index]['edge_index'].to("cpu").numpy()[ neighboring_cells_index_curcell1,0]])
            # if relcellindex == 0:
            #     print(neighboring_cells_index_curcell.shape)
            ##
            predicted_age_pd_merge_perturbation_slice_choose = predicted_age_pd_merge_perturbation_slice.iloc[neighboring_cells_index_curcell, :][['age_predicted']]
            predicted_age_pd_merge_ori_slice_choose = predicted_age_pd_merge_ori_slice.iloc[neighboring_cells_index_curcell, :][['age_predicted']]
            diff = predicted_age_pd_merge_perturbation_slice_choose.values - predicted_age_pd_merge_ori_slice_choose.values
            diff_mean = np.mean(diff)
            diff_mean_list.append(diff_mean)
            ##
            exprecon_use_slice_aginggene_choose = exprecon_use_slice_aginggene[neighboring_cells_index_curcell, :]
            exp_recon_perturb_all_slice_aginggene_choose = exp_recon_perturb_all_slice_aginggene[neighboring_cells_index_curcell, :]
            # diff_aginggeneexp_mean_list.append(np.mean((exp_recon_perturb_all_slice_aginggene_choose - exprecon_use_slice_aginggene_choose)/(exprecon_use_slice_aginggene_choose + 1e-3)))
            # diff_aginggeneexp_mean_list.append(np.median((exp_recon_perturb_all_slice_aginggene_choose - exprecon_use_slice_aginggene_choose)/(exprecon_use_slice_aginggene_choose + 1e-3)))
            diff_aginggeneexp_mean_list.append(np.mean((exp_recon_perturb_all_slice_aginggene_choose - exprecon_use_slice_aginggene_choose)))
            # diff_aginggeneexp_mean_list.append((np.mean(exp_recon_perturb_all_slice_aginggene_choose) - np.mean(exprecon_use_slice_aginggene_choose))/np.mean(exprecon_use_slice_aginggene_choose))
            MI_OI_change_mean_list.append(np.mean(MI_SR_agg_all_perturbation_OI_cur[neighboring_cells_index_curcell] - MI_SR_agg_all_ori_OI_cur[neighboring_cells_index_curcell]))
        diff_mean_all = diff_mean_all + diff_mean_list
        diff_aginggeneexp_mean_all = diff_aginggeneexp_mean_all + diff_aginggeneexp_mean_list
        MI_OI_change_mean_all = MI_OI_change_mean_all + MI_OI_change_mean_list
    ##
    diff_mean_all_dict[permute_celltype_cur] = diff_mean_all  
    aging_geneexp_all_dict[permute_celltype_cur] = diff_aginggeneexp_mean_all
    MI_OI_change_mean_all_dict[permute_celltype_cur] = MI_OI_change_mean_all

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import mannwhitneyu, wilcoxon

# Prepare data for plotting
plot_data = []
for celltype, changes in diff_mean_all_dict.items():
    for change in changes:
        plot_data.append({'Permuted_CellType': celltype, 'Mean_Age_Change': change})
plot_df = pd.DataFrame(plot_data)

# Save CSV for R
csv_path = str(run_dirs['run_dir']) + "/NeighborAgeChange_plotdata.csv"
plot_df.to_csv(output_path(csv_path), index=False)
print("Saved CSV:", csv_path)

# Define x-axis order
order = ['T cell', 'Endothelial', 'Astrocyte']
# order = ['T cell', 'Endothelial','NSC']

# ===== Two-sided, unpaired Mann–Whitney U tests (T cell vs others) =====
pvalues = {}
rank_sum_rows = []
try:
    stat1, p1 = mannwhitneyu(
        diff_mean_all_dict['T cell'],
        diff_mean_all_dict['Endothelial'],
        alternative="two-sided"
    )
    pvalues['Tcell_vs_Endo'] = p1
    rank_sum_rows.append({"group1": "T cell", "group2": "Endothelial",
        "n1": len(diff_mean_all_dict["T cell"]), "n2": len(diff_mean_all_dict["Endothelial"]),
        "statistic_U": stat1, "pvalue": p1})

    stat2, p2 = mannwhitneyu(
        diff_mean_all_dict['T cell'],
        diff_mean_all_dict['Astrocyte'],
        alternative="two-sided"
    )
    pvalues['Tcell_vs_Astrocyte'] = p2
    rank_sum_rows.append({"group1": "T cell", "group2": "Astrocyte",
        "n1": len(diff_mean_all_dict["T cell"]), "n2": len(diff_mean_all_dict["Astrocyte"]),
        "statistic_U": stat2, "pvalue": p2})
    # stat2, p2 = mannwhitneyu(
    #     diff_mean_all_dict['Endothelial'],
    #     diff_mean_all_dict['NSC'],
    #     alternative='greater'
    # )
    # pvalues['Endothelial_vs_NSC'] = p2

except Exception as e:
    print("Error:", e)

# Record the between-condition tests used by the plot annotations.
rank_sum_tests = pd.DataFrame(rank_sum_rows).assign(
    test="Mann-Whitney U (Wilcoxon rank-sum)", alternative="two-sided", paired=False,
    method="auto", use_continuity=True,
)
rank_sum_tests.to_csv(output_path(run_dirs["run_dir"] / "NeighborAgeChange_ranksum_tests.csv"), index=False)
display(rank_sum_tests)

# ===== Extra: Wilcoxon test vs zero =====
print("\n=== Wilcoxon one-sample test (Mean_Age_Change > 0) ===")
pvalues_vs_zero = {}

for ct in order:
    try:
        arr = plot_df.loc[plot_df["Permuted_CellType"] == ct, "Mean_Age_Change"].values
        # must ensure non-zero variance & enough samples
        if len(arr) > 0 and (arr != 0).any():
            stat_z, p_z = wilcoxon(arr, alternative='greater', zero_method='wilcox')
            pvalues_vs_zero[ct] = p_z
            print(f"{ct}: p = {p_z:.3e}")
        else:
            pvalues_vs_zero[ct] = None
            print(f"{ct}: cannot test (all zeros or too few samples)")
    except Exception as e:
        print(f"{ct}: error {e}")
        pvalues_vs_zero[ct] = None

print("\n=== Wilcoxon one-sample test (Mean_Age_Change < 0) ===")
pvalues_vs_zerosmall = {}

for ct in order:
    try:
        arr = plot_df.loc[plot_df["Permuted_CellType"] == ct, "Mean_Age_Change"].values
        # must ensure non-zero variance & enough samples
        if len(arr) > 0 and (arr != 0).any():
            stat_z, p_z = wilcoxon(arr, alternative='less', zero_method='wilcox')
            pvalues_vs_zerosmall[ct] = p_z
            print(f"{ct}: p = {p_z:.3e}")
        else:
            pvalues_vs_zerosmall[ct] = None
            print(f"{ct}: cannot test (all zeros or too few samples)")
    except Exception as e:
        print(f"{ct}: error {e}")
        pvalues_vs_zerosmall[ct] = None
        
# ===== Violin plot =====
plt.figure(figsize=(8, 6))

custom_colors = ["#7F27FF", "#FDBF60", "#FF8911"]

sns.violinplot(
    x='Permuted_CellType',
    y='Mean_Age_Change',
    data=plot_df,
    order=order,
    palette=custom_colors,
    cut=0
)

plt.title('Effect on Predicted Neighbor Age (Mean)')
plt.ylabel('Mean Change in Predicted Age')
plt.xlabel('Permuted Cell Type')
plt.axhline(0, color='gray', linestyle='--')

# ---- annotate p-values for pairwise tests ----
ax = plt.gca()
ymax = plot_df['Mean_Age_Change'].max()

def add_p_label(x1, x2, y, pval, label_shift=0.03):
    line_height = y + ymax * label_shift
    ax.plot([x1, x1, x2, x2], [y, line_height, line_height, y], lw=1.5, c='black')
    text = f"p = {pval:.3e}"
    ax.text((x1 + x2) / 2, line_height + ymax * 0.02, text,
            ha='center', va='bottom', fontsize=10)

if 'Tcell_vs_Endo' in pvalues:
    add_p_label(0, 1, ymax * 1.02, pvalues['Tcell_vs_Endo'])

if 'Tcell_vs_Astrocyte' in pvalues:
    add_p_label(0, 2, ymax * 1.18, pvalues['Tcell_vs_Astrocyte'])
    
# if 'Endothelial_vs_NSC' in pvalues:
#     add_p_label(1, 2, ymax * 1.34, pvalues['Endothelial_vs_NSC'])

plt.tight_layout()
plt.savefig(output_path(str(run_dirs['run_dir']) + '/NeighboringCell_AgeChange_Mean_ViolinPlot.png'), dpi=300)
plt.show()
plt.close()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.stats import mannwhitneyu, wilcoxon

# Prepare data for plotting
plot_data = []
for celltype, changes in MI_OI_change_mean_all_dict.items():
    for change in changes:
        plot_data.append({'Permuted_CellType': celltype, 'Mean_MIOI_Change': change})
plot_df = pd.DataFrame(plot_data)

# Save CSV for R
csv_path = str(run_dirs['run_dir']) + "/NeighborMIOIChange_plotdata.csv"
plot_df.to_csv(output_path(csv_path), index=False)
print("Saved CSV:", csv_path)

# Define x-axis order
order = ['T cell', 'Endothelial', 'Astrocyte']
# order = ['T cell', 'Endothelial', 'NSC']

# ===== Two-sided, unpaired Mann–Whitney U tests (T cell vs others) =====
pvalues = {}
rank_sum_rows = []
try:
    stat1, p1 = mannwhitneyu(
        MI_OI_change_mean_all_dict['T cell'],
        MI_OI_change_mean_all_dict['Endothelial'],
        alternative="two-sided"
    )
    pvalues['Tcell_vs_Endo'] = p1
    rank_sum_rows.append({"group1": "T cell", "group2": "Endothelial",
        "n1": len(MI_OI_change_mean_all_dict["T cell"]), "n2": len(MI_OI_change_mean_all_dict["Endothelial"]),
        "statistic_U": stat1, "pvalue": p1})

    stat2, p2 = mannwhitneyu(
        MI_OI_change_mean_all_dict['T cell'],
        MI_OI_change_mean_all_dict['Astrocyte'],
        alternative="two-sided"
    )
    pvalues['Tcell_vs_Astrocyte'] = p2
    rank_sum_rows.append({"group1": "T cell", "group2": "Astrocyte",
        "n1": len(MI_OI_change_mean_all_dict["T cell"]), "n2": len(MI_OI_change_mean_all_dict["Astrocyte"]),
        "statistic_U": stat2, "pvalue": p2})

except Exception as e:
    print("Error:", e)

# Record the between-condition tests used by the plot annotations.
rank_sum_tests = pd.DataFrame(rank_sum_rows).assign(
    test="Mann-Whitney U (Wilcoxon rank-sum)", alternative="two-sided", paired=False,
    method="auto", use_continuity=True,
)
rank_sum_tests.to_csv(output_path(run_dirs["run_dir"] / "NeighborMIOIChange_ranksum_tests.csv"), index=False)
display(rank_sum_tests)

# ===== Extra: Wilcoxon test vs zero =====
print("\n=== Wilcoxon one-sample test (Mean_MIOI_Change > 0) ===")
pvalues_vs_zero = {}

for ct in order:
    try:
        arr = plot_df.loc[plot_df["Permuted_CellType"] == ct, "Mean_MIOI_Change"].values
        # must ensure non-zero variance & enough samples
        if len(arr) > 0 and (arr != 0).any():
            stat_z, p_z = wilcoxon(arr, alternative='greater', zero_method='wilcox')
            pvalues_vs_zero[ct] = p_z
            print(f"{ct}: p = {p_z:.3e}")
        else:
            pvalues_vs_zero[ct] = None
            print(f"{ct}: cannot test (all zeros or too few samples)")
    except Exception as e:
        print(f"{ct}: error {e}")
        pvalues_vs_zero[ct] = None

# ===== Violin plot =====
plt.figure(figsize=(8, 6))

custom_colors = ["#7F27FF", "#FDBF60", "#FF8911"]

sns.violinplot(
    x='Permuted_CellType',
    y='Mean_MIOI_Change',
    data=plot_df,
    order=order,
    palette=custom_colors,
    cut=0
)

# plt.title('Neighbor receiving MI-29 strength change')
plt.ylabel('Change in MI-29 strength received by neighbors')
plt.xlabel('Permuted Cell Type')
plt.axhline(0, color='gray', linestyle='--')

# ---- annotate p-values for pairwise tests ----
ax = plt.gca()
ymax = plot_df['Mean_MIOI_Change'].max()

def add_p_label(x1, x2, y, pval, label_shift=0.03):
    line_height = y + ymax * label_shift
    ax.plot([x1, x1, x2, x2], [y, line_height, line_height, y], lw=1.5, c='black')
    text = f"p = {pval:.3e}"
    ax.text((x1 + x2) / 2, line_height + ymax * 0.02, text,
            ha='center', va='bottom', fontsize=10)

if 'Tcell_vs_Endo' in pvalues:
    add_p_label(0, 1, ymax * 1.02, pvalues['Tcell_vs_Endo'])

if 'Tcell_vs_Astrocyte' in pvalues:
    add_p_label(0, 2, ymax * 1.18, pvalues['Tcell_vs_Astrocyte'])

plt.tight_layout()
plt.savefig(output_path(str(run_dirs['run_dir']) + '/NeighboringCell_MIOIChange_Mean_ViolinPlot.png'), dpi=300)
plt.show()
plt.close()


## Optional C2. In silico knockout of MI-associated genes

In [ ]:
MI_OI = 'MI29'
MI_OI_numeric = int(MI_OI.replace('MI', '')) - 1
loading_threshold = 0.5

In [ ]:
loading_LR_use = np.load(input_path(run_dirs['run_dir'] / "loading_LR_use.npy"))
LR_list = pd.read_pickle(input_path(run_dirs['run_dir'] / "LR_list.pkl"))
loading_LR_use_df = pd.DataFrame(
    loading_LR_use,
    columns=[('+').join(lr[0]) + " -> " + ('+').join(lr[1]) for lr in LR_list],
    index = [f"MI{i+1}" for i in range(loading_LR_use.shape[0])]
)
# print(loading_LR_use_df)
# print(loading_LR_use_df.loc["MI29",:])
##Sort the loading_LR_use_df.loc["MI29",:]
# print(loading_LR_use_df.loc["MI29", :].sort_values(ascending=False))

In [ ]:
# print(MI_OI)
np.random.seed(123)
# Get the top ligand-receptor pairs from LR——loading of the MI_OI
# -----------------------------------------
# Step 1. Collect all ligands and receptors
# -----------------------------------------
gene_index_list = processed.adata_all.var.index.tolist()
# print(gene_index_list)

# Extract unique ligand and receptor genes from LR_list
ligand_all = list({g for lr in LR_list for g in lr[0]})
receptor_all = list({g for lr in LR_list for g in lr[1]})

# Map ligands/receptors to their indices in the expression matrix
ligand_all_geneindex = [gene_index_list.index(g) for g in ligand_all if g in gene_index_list]
receptor_all_geneindex = [gene_index_list.index(g) for g in receptor_all if g in gene_index_list]
# print("The number of ligands:", len(ligand_all_geneindex))
# print("The number of receptors:", len(receptor_all_geneindex))
# print(ligand_all_geneindex)
# print(receptor_all_geneindex)
# print(np.intersect1d(receptor_all_geneindex, ligand_all_geneindex).shape)

# -----------------------------------------
# Step 2. Normalize LR loading matrix
# -----------------------------------------
# print(loading_LR_use_df)
# Each LR column is normalized by its total loading
loading_LR_use_df_norm = loading_LR_use_df.div(loading_LR_use_df.sum(axis=0), axis=1)
# loading_LR_use_df_norm = loading_LR_use_df
# print(np.sum(loading_LR_use_df_norm,axis = 0))
loading_LR_use_df_norm_MIOI = loading_LR_use_df_norm.loc[MI_OI, :]

# Select LR pairs with strong association (>0.5)
loading_LR_use_df_norm_MIOI_sorted = loading_LR_use_df_norm_MIOI.sort_values(ascending=False)
# print(loading_LR_use_df_norm_MIOI_sorted)
loading_LR_use_df_norm_MIOI_sorted_choose = loading_LR_use_df_norm_MIOI_sorted[
    # loading_LR_use_df_norm_MIOI_sorted > 0.8
    loading_LR_use_df_norm_MIOI_sorted > loading_threshold
    # loading_LR_use_df_norm_MIOI_sorted > 0.3
    # loading_LR_use_df_norm_MIOI_sorted > 0.1
]
# print(loading_LR_use_df_norm_MIOI_sorted_choose)
print(loading_LR_use_df_norm_MIOI_sorted_choose.index)

# -----------------------------------------
# Step 3. Parse ligand information
# -----------------------------------------
# print(loading_LR_use_df_norm_MIOI_sorted_choose.index)
# Extract ligand names from "ligand -> receptor" notation
ligand_top_list = [x.split(' -> ')[0] for x in loading_LR_use_df_norm_MIOI_sorted_choose.index]

# Handle multi-subunit ligands (e.g., "IL12A+IL12B")
ligand_top_list_split = [
    sub for l in ligand_top_list for sub in l.replace('(', '').replace(')', '').split('+')
]

# Identify unique ligands and their indices
ligand_top_list_split_unique = list(set(ligand_top_list_split))
# print(ligand_top_list_split_unique)
ligand_top_list_split_unique_geneindex = [
    gene_index_list.index(g) for g in ligand_top_list_split_unique if g in gene_index_list
]

# Baseline ligand indices (random selection from all ligands excluding top ones)
num_topL = len(ligand_top_list_split_unique_geneindex)
ligand_top_list_split_unique_geneindex_baseline = np.random.choice(
    np.setdiff1d(ligand_all_geneindex, ligand_top_list_split_unique_geneindex),
    size=num_topL,
    replace=False
)

# -----------------------------------------
# Step 4. Parse receptor information
# -----------------------------------------
# Extract receptor names from "ligand -> receptor" notation
receptor_top_list = [x.split(' -> ')[1] for x in loading_LR_use_df_norm_MIOI_sorted_choose.index]

# Handle multi-subunit receptors (e.g., "IL2RA+IL2RB")
receptor_top_list_split = [
    sub for r in receptor_top_list for sub in r.replace('(', '').replace(')', '').split('+')
]

# Identify unique receptors and their indices
receptor_top_list_split_unique = list(set(receptor_top_list_split))
receptor_top_list_split_unique_geneindex = [
    gene_index_list.index(g) for g in receptor_top_list_split_unique if g in gene_index_list
]

# Baseline receptor indices (random selection from all receptors excluding top ones)
num_topR = len(receptor_top_list_split_unique_geneindex)
receptor_top_list_split_unique_geneindex_baseline = np.random.choice(
    np.setdiff1d(receptor_all_geneindex, receptor_top_list_split_unique_geneindex),
    size=num_topR,
    replace=False
)

In [ ]:
knockout_geneindex_dict = {"Top":{"Ligand": ligand_top_list_split_unique_geneindex,
                                  "Receptor": receptor_top_list_split_unique_geneindex},
                            "Baseline":{"Ligand": ligand_top_list_split_unique_geneindex_baseline.tolist(),
                                        "Receptor": receptor_top_list_split_unique_geneindex_baseline.tolist()}}
print(knockout_geneindex_dict)

In [ ]:
## Identify candidate regulator and target genes for knockout
loading_receiver_use = np.load(input_path(run_dirs['run_dir'] / "loading_receiver_use.npy"))
loading_sender_use = np.load(input_path(run_dirs['run_dir'] / "loading_sender_use.npy"))
loading_receiver_use_df = pd.DataFrame(loading_receiver_use, index=["MI" + str(i+1) for i in range(loading_receiver_use.shape[0])],
                                        columns=processed.adata_list[0].var_names)
loading_sender_use_df = pd.DataFrame(loading_sender_use, index=["MI" + str(i+1) for i in range(loading_sender_use.shape[0])],
                                        columns=processed.adata_list[0].var_names)
loading_receiver_use_df_colsum = loading_receiver_use_df.abs().sum(axis=0)
# print(loading_receiver_use_df_colsum)
loading_sender_use_df_colsum = loading_sender_use_df.abs().sum(axis=0)
loading_receiver_use_df_norm = loading_receiver_use_df.div(loading_receiver_use_df_colsum, axis=1)
# print(np.sum(loading_receiver_use_df_norm,axis = 0))
loading_sender_use_df_norm = loading_sender_use_df.div(loading_sender_use_df_colsum, axis=1)

loading_receiver_use_df_norm_MIOI = loading_receiver_use_df_norm.loc[MI_OI, :]
loading_sender_use_df_norm_MIOI = loading_sender_use_df_norm.loc[MI_OI, :]

In [ ]:
# print(np.where(loading_sender_use_df_norm_MIOI > 0.5)[0])
top_regulators_geneindex = np.where(loading_sender_use_df_norm_MIOI > loading_threshold)[0].tolist()
top_regulators_gene = processed.adata_all.var_names[top_regulators_geneindex].tolist()
print("top_regulators_gene:", top_regulators_gene)
knockout_geneindex_dict["Top"]["Ligand"] =knockout_geneindex_dict["Top"]["Ligand"] +  top_regulators_geneindex
top_targets_geneindex = np.where(loading_receiver_use_df_norm_MIOI > loading_threshold)[0].tolist()
top_targets_gene = processed.adata_all.var_names[top_targets_geneindex].tolist()
print("top_targets_gene:", top_targets_gene)
knockout_geneindex_dict["Top"]["Receptor"] =knockout_geneindex_dict["Top"]["Receptor"] +  top_targets_geneindex

In [ ]:
# Also extend the baseline set
np.random.seed(123)
knockout_geneindex_dict["Baseline"]["Ligand"] = knockout_geneindex_dict["Baseline"]["Ligand"] + np.random.choice(
    np.setdiff1d(np.arange(processed.adata_all.shape[1]), np.union1d(knockout_geneindex_dict["Top"]["Ligand"],knockout_geneindex_dict["Baseline"]["Ligand"])),
    size=len(top_regulators_geneindex),
    replace=False
).tolist()
knockout_geneindex_dict["Baseline"]["Receptor"] = knockout_geneindex_dict["Baseline"]["Receptor"] + np.random.choice(
    np.setdiff1d(np.arange(processed.adata_all.shape[1]), np.union1d(knockout_geneindex_dict["Top"]["Receptor"],knockout_geneindex_dict["Baseline"]["Receptor"])),
    size=len(top_targets_geneindex),
    replace=False
).tolist()

In [ ]:
## Build a control knockout gene set with weak association to the selected MI
np.random.seed(123)
# print(MI_OI)
# Get the top ligand-receptor pairs from LR——loading of the MI_OI
# -----------------------------------------
# Step 1. Collect all ligands and receptors
# -----------------------------------------
gene_index_list = processed.adata_all.var.index.tolist()
# print(gene_index_list)

# Extract unique ligand and receptor genes from LR_list
ligand_all = list({g for lr in LR_list for g in lr[0]})
receptor_all = list({g for lr in LR_list for g in lr[1]})

# Map ligands/receptors to their indices in the expression matrix
ligand_all_geneindex = [gene_index_list.index(g) for g in ligand_all if g in gene_index_list]
receptor_all_geneindex = [gene_index_list.index(g) for g in receptor_all if g in gene_index_list]
# print("The number of ligands:", len(ligand_all_geneindex))
# print("The number of receptors:", len(receptor_all_geneindex))
# print(ligand_all_geneindex)
# print(receptor_all_geneindex)
# print(np.intersect1d(receptor_all_geneindex, ligand_all_geneindex).shape)

# -----------------------------------------
# Step 2. Normalize LR loading matrix
# -----------------------------------------
# print(loading_LR_use_df)
# Each LR column is normalized by its total loading
loading_LR_use_df_norm = loading_LR_use_df.div(loading_LR_use_df.sum(axis=0), axis=1)
# loading_LR_use_df_norm = loading_LR_use_df
# print(np.sum(loading_LR_use_df_norm,axis = 0))
loading_LR_use_df_norm_MIOI = loading_LR_use_df_norm.loc[MI_OI, :]

# Select LR pairs with strong association (>0.5)
loading_LR_use_df_norm_MIOI_sorted = loading_LR_use_df_norm_MIOI.sort_values(ascending=False)
# print(loading_LR_use_df_norm_MIOI_sorted)
loading_LR_use_df_norm_MIOI_sorted_choose = loading_LR_use_df_norm_MIOI_sorted[
    # loading_LR_use_df_norm_MIOI_sorted > 0.8
    (loading_LR_use_df_norm_MIOI_sorted < loading_threshold) & (loading_LR_use_df_norm_MIOI_sorted > loading_threshold/2)
    # loading_LR_use_df_norm_MIOI_sorted > 0.3
    # loading_LR_use_df_norm_MIOI_sorted > 0.1
]
# print(loading_LR_use_df_norm_MIOI_sorted_choose)
# print(loading_LR_use_df_norm_MIOI_sorted_choose.index)

# -----------------------------------------
# Step 3. Parse ligand information
# -----------------------------------------
# print(loading_LR_use_df_norm_MIOI_sorted_choose.index)
# Extract ligand names from "ligand -> receptor" notation
ligand_top_list = [x.split(' -> ')[0] for x in loading_LR_use_df_norm_MIOI_sorted_choose.index]

# Handle multi-subunit ligands (e.g., "IL12A+IL12B")
ligand_top_list_split = [
    sub for l in ligand_top_list for sub in l.replace('(', '').replace(')', '').split('+')
]

# Identify unique ligands and their indices
ligand_top_list_split_unique = list(set(ligand_top_list_split))
# print(ligand_top_list_split_unique)
ligand_top_list_split_unique_geneindex = [
    gene_index_list.index(g) for g in ligand_top_list_split_unique if g in gene_index_list
]

# Baseline ligand indices (random selection from all ligands excluding top ones)
num_topL = len(ligand_top_list_split_unique_geneindex)
ligand_top_list_split_unique_geneindex_baseline = np.random.choice(
    np.setdiff1d(ligand_all_geneindex, ligand_top_list_split_unique_geneindex),
    size=num_topL,
    replace=False
)

# -----------------------------------------
# Step 4. Parse receptor information
# -----------------------------------------
# Extract receptor names from "ligand -> receptor" notation
receptor_top_list = [x.split(' -> ')[1] for x in loading_LR_use_df_norm_MIOI_sorted_choose.index]

# Handle multi-subunit receptors (e.g., "IL2RA+IL2RB")
receptor_top_list_split = [
    sub for r in receptor_top_list for sub in r.replace('(', '').replace(')', '').split('+')
]

# Identify unique receptors and their indices
receptor_top_list_split_unique = list(set(receptor_top_list_split))
receptor_top_list_split_unique_geneindex = [
    gene_index_list.index(g) for g in receptor_top_list_split_unique if g in gene_index_list
]

# Baseline receptor indices (random selection from all receptors excluding top ones)
num_topR = len(receptor_top_list_split_unique_geneindex)
receptor_top_list_split_unique_geneindex_baseline = np.random.choice(
    np.setdiff1d(receptor_all_geneindex, receptor_top_list_split_unique_geneindex),
    size=num_topR,
    replace=False
)
knockout_geneindex_dict["Mild"] = {"Ligand": ligand_top_list_split_unique_geneindex,
                                  "Receptor": receptor_top_list_split_unique_geneindex}
## Plan to dropout some regulators and targets
loading_receiver_use_df = pd.DataFrame(loading_receiver_use, index=["MI" + str(i+1) for i in range(loading_receiver_use.shape[0])],
                                        columns=processed.adata_list[0].var_names)
loading_sender_use_df = pd.DataFrame(loading_sender_use, index=["MI" + str(i+1) for i in range(loading_sender_use.shape[0])],
                                        columns=processed.adata_list[0].var_names)
loading_receiver_use_df_colsum = loading_receiver_use_df.abs().sum(axis=0)
# print(loading_receiver_use_df_colsum)
loading_sender_use_df_colsum = loading_sender_use_df.abs().sum(axis=0)
loading_receiver_use_df_norm = loading_receiver_use_df.div(loading_receiver_use_df_colsum, axis=1)
# print(np.sum(loading_receiver_use_df_norm,axis = 0))
loading_sender_use_df_norm = loading_sender_use_df.div(loading_sender_use_df_colsum, axis=1)
loading_receiver_use_df_norm_MIOI = loading_receiver_use_df_norm.loc[MI_OI, :]
loading_sender_use_df_norm_MIOI = loading_sender_use_df_norm.loc[MI_OI, :]
# print(np.where(loading_sender_use_df_norm_MIOI > 0.5)[0])
top_regulators_geneindex = np.where((loading_sender_use_df_norm_MIOI < loading_threshold) * (loading_sender_use_df_norm_MIOI > loading_threshold/2))[0].tolist()
top_regulators_gene = processed.adata_all.var_names[top_regulators_geneindex].tolist()
# print("top_regulators_gene:", top_regulators_gene)
## Add regulator and target control sets
knockout_geneindex_dict["Mild"]["Ligand"] = knockout_geneindex_dict["Mild"]["Ligand"] +  top_regulators_geneindex
top_targets_geneindex = np.where((loading_receiver_use_df_norm_MIOI < loading_threshold) * (loading_receiver_use_df_norm_MIOI > loading_threshold/2))[0].tolist()
top_targets_gene = processed.adata_all.var_names[top_targets_geneindex].tolist()
# print("top_targets_gene:", top_targets_gene)
knockout_geneindex_dict["Mild"]["Receptor"] =knockout_geneindex_dict["Mild"]["Receptor"] +  top_targets_geneindex

In [ ]:
sender_celltype = "T cell"
# sender_celltype = "NSC"

In [ ]:
permute_type_list = ['Top', 'Top_half', "Mild", 'Baseline']

In [ ]:
cellsize_list = [processed.spidernet_data[i].x.shape[0] for i in range(len(processed.spidernet_data))]
startcellindex_list = np.cumsum([0] + cellsize_list[:-1])
print(startcellindex_list)

In [ ]:
neighoringcell_age_change_dict = {}
diff_mean_all_dict = {}
aging_geneexp_all_dict = {}
for permute_type_cur in permute_type_list:
    print(permute_type_cur)
    SpiderNet_data_pyg_list_perturbation = pd.read_pickle(input_path(str(run_dirs['run_dir']) + '/SpiderNet_data_pyg_list.pkl'))
    SpiderNet_data_pyg_list_perturbation = [SpiderNet_data_pyg_list_perturbation[i].to(device) for i in range(len(SpiderNet_data_pyg_list_perturbation))]
    ##
    neighboring_cells_index_all = []
    MI_SR_agg_all_perturbation = []
    MI_SR_agg_all_noperturbation = []
    cellindex_curpermute_type_all = []
    exp_recon_perturb_all = []
    for slice_index in range(len(SpiderNet_data_pyg_list_perturbation)):
        # cellindex_notcurpermute_type = np.where(np.array(SpiderNet_data_pyg_list_perturbation[slice_index]['cell_class']) != permute_type_cur)[0]
        # if len(cellindex_notcurpermute_type) > num_replacecell_perslice:
        #     cellindex_curpermute_type = np.random.choice(cellindex_notcurpermute_type, (num_replacecell_perslice),replace=False)
        # else:
        #     cellindex_curpermute_type = cellindex_notcurpermute_type
        # cellindex_curpermute_type_all.append(cellindex_curpermute_type)
        cellindex_curpermute_type = np.where(np.array(SpiderNet_data_pyg_list_perturbation[slice_index]['cell_class']) == "T cell")[0]
        ##Get the neighboring cell index of cellindex_curpermute_type
        edge_index_cur = SpiderNet_data_pyg_list_perturbation[slice_index]['edge_index']
        edge_index_cur_np = edge_index_cur.to("cpu").numpy()
        neighboring_cells = edge_index_cur_np[:,1][np.isin(edge_index_cur_np[:,0], cellindex_curpermute_type)]
        neighboring_cells = np.unique(neighboring_cells)
        ##
        if permute_type_cur == 'reverse':
            cellindex_curpermute_type_copy = cellindex_curpermute_type.copy()
            cellindex_curpermute_type = neighboring_cells
            neighboring_cells = cellindex_curpermute_type_copy
        # ##Remove the T cell in neighboring_cells
        # neighboring_cells = neighboring_cells[~np.isin(neighboring_cells, cellindex_curpermute_type)]
        neighboring_cells_index = np.zeros(len(SpiderNet_data_pyg_list_perturbation[slice_index]['cell_class']))
        neighboring_cells_index[neighboring_cells] = 1
        ##
        cellindex_curpermute_type_all.append(cellindex_curpermute_type)
        neighboring_cells_index_all.append(neighboring_cells_index)
        ##Get the nei
        exp_recon_noperturb, _, _, _, Factor_envir_curbatch_noperturbation, _, _, _ = model(
        SpiderNet_data_pyg_list_perturbation[slice_index].to(device))
        ##
        Factor_envir_curbatch_noperturbation = Factor_envir_curbatch_noperturbation.to(device)
        edge_index_cur = SpiderNet_data_pyg_list_perturbation[slice_index]['edge_index']
        num_cell_cur = SpiderNet_data_pyg_list_perturbation[slice_index].x.shape[0]
        ##
        ##
        MI_receiver_agg_cur = scatter_mean(torch.tensor(Factor_envir_curbatch_noperturbation).to(device),
                                           edge_index_cur[:, 1].to(torch.int64), dim=0,
                                           dim_size=num_cell_cur).to("cpu").numpy()
        MI_sender_agg_cur = scatter_mean(torch.tensor(Factor_envir_curbatch_noperturbation).to(device),
                                         edge_index_cur[:, 0].to(torch.int64), dim=0,
                                         dim_size=num_cell_cur).to("cpu").numpy()
        MI_SR_agg_cur_noperturbation = np.hstack([MI_sender_agg_cur, MI_receiver_agg_cur])
        MI_SR_agg_all_noperturbation.append(MI_SR_agg_cur_noperturbation)
        ##
        # num_cellindex_curpermute_type = len(cellindex_curpermute_type)
        # cellindex_useforplace_global = cellindex_dict[permute_type_cur]
        # cellindex_useforplace_global = cellindex_useforplace_global[np.random.choice(len(cellindex_useforplace_global), (num_cellindex_curpermute_type),replace=False)]
        # ##
        # exp_cellindex_useforplace_global = geneexp_all[cellindex_useforplace_global, :]
        # cellclass_one__useforplace_global =  celltype_onehot_all[cellindex_useforplace_global, :]
        # ##Replace the elements in SpiderNet_data_pyg_list_perturbation[slice_index]
        # SpiderNet_data_pyg_list_perturbation[slice_index]['x'][cellindex_curpermute_type, :] = torch.tensor(exp_cellindex_useforplace_global, dtype=torch.float32).to(device)
        # SpiderNet_data_pyg_list_perturbation[slice_index]['cell_class_onehot'][cellindex_curpermute_type, :] = torch.tensor(cellclass_one__useforplace_global, dtype=torch.float32).to(device)
        # SpiderNet_data_pyg_list_perturbation[slice_index]['x']
        if permute_type_cur == 'Top' or permute_type_cur == 'reverse':
            knockout_geneindex_cur = knockout_geneindex_dict['Top']
            changerate = 0
        elif permute_type_cur == 'Top_half':
            knockout_geneindex_cur = knockout_geneindex_dict['Top']
            changerate = 0.5
        elif permute_type_cur == 'Baseline':
            knockout_geneindex_cur = knockout_geneindex_dict['Baseline']
            changerate = 0
        elif permute_type_cur == 'Mild':
            knockout_geneindex_cur = knockout_geneindex_dict['Mild']
            changerate = 0
        ##Ligand knockout in T cells
        SpiderNet_data_pyg_list_perturbation[slice_index]['x'][cellindex_curpermute_type[:, None], knockout_geneindex_cur['Ligand']] = \
            SpiderNet_data_pyg_list_perturbation[slice_index]['x'][cellindex_curpermute_type[:, None], knockout_geneindex_cur['Ligand']] * changerate
        ##Receptor knockout in neighboring cells of T cells
        SpiderNet_data_pyg_list_perturbation[slice_index]['x'][neighboring_cells[:, None], knockout_geneindex_cur['Receptor']] = \
            SpiderNet_data_pyg_list_perturbation[slice_index]['x'][neighboring_cells[:, None], knockout_geneindex_cur['Receptor']] * changerate
        ##Infer the MI for the perturbed data
        model.eval()
        exp_recon_perturb, _, _, _, Factor_envir_curbatch_perturbation, _, _, _ = model(SpiderNet_data_pyg_list_perturbation[slice_index].to(device))
        exp_recon_perturb_all.append(exp_recon_perturb.to("cpu").detach().numpy())
        ##
        Factor_envir_curbatch_perturbation = Factor_envir_curbatch_perturbation.to(device)
        edge_index_cur = SpiderNet_data_pyg_list_perturbation[slice_index]['edge_index']
        num_cell_cur = SpiderNet_data_pyg_list_perturbation[slice_index].x.shape[0]
        ##
        MI_receiver_agg_cur = scatter_mean(torch.tensor(Factor_envir_curbatch_perturbation).to(device),
                                       edge_index_cur[:, 1].to(torch.int64), dim=0,
                                       dim_size=num_cell_cur).to("cpu").numpy()
        MI_sender_agg_cur = scatter_mean(torch.tensor(Factor_envir_curbatch_perturbation).to(device),
                                     edge_index_cur[:, 0].to(torch.int64), dim=0,
                                     dim_size=num_cell_cur).to("cpu").numpy()
        MI_SR_agg_cur_perturbation = np.hstack([MI_sender_agg_cur, MI_receiver_agg_cur])
        MI_SR_agg_all_perturbation.append(MI_SR_agg_cur_perturbation)
    MI_SR_agg_all_perturbation = np.vstack(MI_SR_agg_all_perturbation)
    MI_SR_agg_all_noperturbation = np.vstack(MI_SR_agg_all_noperturbation)
    test_vec = MI_SR_agg_all_perturbation[:,58] - MI_SR_agg_all_noperturbation[:,58]
    # test_vec = MI_SR_agg_all_perturbation[:,28] - MI_SR_agg_all[:,28]
    # print(np.mean(MI_SR_agg_all_perturbation[:,58] - MI_SR_agg_all[:,58]))
    # test_vec = np.mean(MI_SR_agg_all_perturbation - MI_SR_agg_all, axis=0)
    test_vec_nonzero = test_vec[test_vec !=0]
    print(np.quantile(test_vec,(0,0.2,0.5,0.8,1.0)))
    print(np.quantile(test_vec_nonzero,(0,0.2,0.5,0.8,1.0)))
    neighboring_cells_index_all = np.hstack(neighboring_cells_index_all)
    exp_recon_perturb_all = np.vstack(exp_recon_perturb_all)
    ##
    ##3.2 Extract the predicted age for the perturbation data
    predicted_age_pd_merge_perturbation = pd.DataFrame({'cell_index': [],
                      "age_predicted": []})
    ##test
    predicted_age_pd_merge_noperturbation = pd.DataFrame({'cell_index': [],
                      "age_predicted": []})
    for cellclass_cur in cellclass_unique:
        # print(cellclass_cur)
        ##
        cellindex_cur = np.where((celltype_all == cellclass_cur))[0]
        age_cur = np.array(cellage_all[cellindex_cur])
        ##
        X_train_scaled = MI_SR_agg_all_perturbation[cellindex_cur,:]
        ##
        reg_scale_cur = reg_scale_dict[cellclass_cur]
        ##
        predicted_age_pd = pd.DataFrame({'cell_index': cellindex_cur,
                      "age_predicted": reg_scale_cur.predict(X_train_scaled)})
        predicted_age_pd_merge_perturbation = pd.concat([predicted_age_pd_merge_perturbation,predicted_age_pd])
        ##
        # X_train_scaled = MI_SR_agg_all[cellindex_cur,:]
        X_train_scaled = MI_SR_agg_all_noperturbation[cellindex_cur,:]
        ##
        reg_scale_cur = reg_scale_dict[cellclass_cur]
        ##
        predicted_age_pd = pd.DataFrame({'cell_index': cellindex_cur,
                      "age_predicted": reg_scale_cur.predict(X_train_scaled)})
        predicted_age_pd_merge_noperturbation = pd.concat([predicted_age_pd_merge_noperturbation,predicted_age_pd])
    predicted_age_pd_merge_perturbation['cell_index'] = predicted_age_pd_merge_perturbation['cell_index'].astype(int)
    predicted_age_pd_merge_noperturbation['cell_index'] = predicted_age_pd_merge_noperturbation['cell_index'].astype(int)
    ##Sort the row of predicted_age_pd_merge_perturbation by the cell_index
    predicted_age_pd_merge_perturbation = predicted_age_pd_merge_perturbation.sort_values(by = ['cell_index'])
    predicted_age_pd_merge_noperturbation = predicted_age_pd_merge_noperturbation.sort_values(by = ['cell_index'])
    predicted_age_pd_merge_perturbation['true_age'] = cellage_all[predicted_age_pd_merge_perturbation['cell_index'].values]
    predicted_age_pd_merge_noperturbation['true_age'] = cellage_all[predicted_age_pd_merge_noperturbation['cell_index'].values]
    predicted_age_pd_merge_perturbation['is_neighboring_perturbcell'] = neighboring_cells_index_all[predicted_age_pd_merge_perturbation['cell_index'].values]
    predicted_age_pd_merge_noperturbation['is_neighboring_perturbcell'] = neighboring_cells_index_all[predicted_age_pd_merge_noperturbation['cell_index'].values]
    # print(predicted_age_pd_merge_perturbation)
    # print(np.sum(predicted_age_pd_merge_perturbation['is_neighboring_perturbcell']))
    predicted_age_pd_merge_perturbation_is_neighboring_perturbcell = predicted_age_pd_merge_perturbation[predicted_age_pd_merge_perturbation['is_neighboring_perturbcell'] == 1]
    predicted_age_pd_merge_noperturbation_is_neighboring_perturbcell = predicted_age_pd_merge_noperturbation[predicted_age_pd_merge_noperturbation['is_neighboring_perturbcell'] == 1]
    # print(predicted_age_pd_merge_noperturbation_is_neighboring_perturbcell.shape)
    predicted_age_pd_merge_ori_is_neighboring_perturbcell = predicted_age_pd_merge_noperturbation[predicted_age_pd_merge_perturbation['is_neighboring_perturbcell'] == 1]
    # print(predicted_age_pd_merge_perturbation_is_neighboring_perturbcell)
    # print(predicted_age_pd_merge_ori_is_neighboring_perturbcell)
    neighoringcell_age_change = predicted_age_pd_merge_perturbation_is_neighboring_perturbcell['age_predicted'].values - predicted_age_pd_merge_ori_is_neighboring_perturbcell['age_predicted'].values
    neighoringcell_age_change_test = predicted_age_pd_merge_perturbation_is_neighboring_perturbcell['age_predicted'].values - predicted_age_pd_merge_noperturbation_is_neighboring_perturbcell['age_predicted'].values
    print("Test")
    print(np.quantile(neighoringcell_age_change_test, [0.0, 0.25, 0.5, 0.75, 1.0]))
    # print(np.quantile(neighoringcell_age_change, [0.05, 0.25, 0.5, 0.75, 0.95]))
    neighoringcell_age_change_dict[permute_type_cur] = neighoringcell_age_change
    ##
    diff_mean_all = []
    diff_aginggeneexp_mean_all = []
    for slice_index in range(len(SpiderNet_data_pyg_list_perturbation)):
        start_index_cur = startcellindex_list[slice_index]
        end_index_cur = startcellindex_list[slice_index] + cellsize_list[slice_index]
        predicted_age_pd_merge_perturbation_slice = predicted_age_pd_merge_perturbation.iloc[start_index_cur:end_index_cur, :]
        predicted_age_pd_merge_ori_slice = predicted_age_pd_merge_noperturbation.iloc[start_index_cur:end_index_cur, :]
        cellindex_curpermute_type_slice = cellindex_curpermute_type_all[slice_index]
        ##
        exp_recon_perturb_all_slice = exp_recon_perturb_all[start_index_cur:end_index_cur, :]
        exprecon_use_slice = geneexp_all[start_index_cur:end_index_cur, :]
        exp_recon_perturb_all_slice_aginggene = exp_recon_perturb_all_slice[:, aging_genes_index]
        exprecon_use_slice_aginggene = exprecon_use_slice[:, aging_genes_index]
        ##
        diff_mean_list = []
        diff_aginggeneexp_mean_list = []
        for relcellindex in range(len(cellindex_curpermute_type_slice)):
            neighboring_cells_index_curcell0 = np.where(SpiderNet_data_pyg_list_perturbation[slice_index]['edge_index'].to("cpu").numpy()[ :,0] == cellindex_curpermute_type_slice[relcellindex])[0]
            # neighboring_cells_index_curcell1 = np.where(SpiderNet_data_pyg_list_perturbation[slice_index]['edge_index'].to("cpu").numpy()[ :,1] == cellindex_curpermute_type_slice[relcellindex])[0]
            neighboring_cells_index_curcell = SpiderNet_data_pyg_list_perturbation[slice_index]['edge_index'].to("cpu").numpy()[ neighboring_cells_index_curcell0,1]
            if permute_type_cur == 'reverse':
                neighboring_cells_index_curcell_cellclass = celltype_all[start_index_cur:end_index_cur][neighboring_cells_index_curcell]
                neighboring_cells_index_curcell_isTcell = neighboring_cells_index_curcell_cellclass == "T cell"
                neighboring_cells_index_curcell = neighboring_cells_index_curcell[neighboring_cells_index_curcell_isTcell]
            # if permute_type_cur == 'reverse':
            #     neighboring_cells_index_curcell = cellindex_curpermute_type_slice[relcellindex]
            # if relcellindex == 0:
            #     print(neighboring_cells_index_curcell.shape)
            ##
            predicted_age_pd_merge_perturbation_slice_choose = predicted_age_pd_merge_perturbation_slice.iloc[neighboring_cells_index_curcell, :][['age_predicted']]
            predicted_age_pd_merge_ori_slice_choose = predicted_age_pd_merge_ori_slice.iloc[neighboring_cells_index_curcell, :][['age_predicted']]
            diff = predicted_age_pd_merge_perturbation_slice_choose.values - predicted_age_pd_merge_ori_slice_choose.values
            diff_mean = np.mean(diff)
            diff_mean_list.append(diff_mean)
            ##
            exprecon_use_slice_aginggene_choose = exprecon_use_slice_aginggene[neighboring_cells_index_curcell, :]
            exp_recon_perturb_all_slice_aginggene_choose = exp_recon_perturb_all_slice_aginggene[neighboring_cells_index_curcell, :]
            # diff_aginggeneexp_mean_list.append(np.mean((exp_recon_perturb_all_slice_aginggene_choose - exprecon_use_slice_aginggene_choose)/(exprecon_use_slice_aginggene_choose + 1e-3)))
            # diff_aginggeneexp_mean_list.append(np.median((exp_recon_perturb_all_slice_aginggene_choose - exprecon_use_slice_aginggene_choose)/(exprecon_use_slice_aginggene_choose + 1e-3)))
            diff_aginggeneexp_mean_list.append(np.mean((exp_recon_perturb_all_slice_aginggene_choose - exprecon_use_slice_aginggene_choose)))
            # diff_aginggeneexp_mean_list.append((np.mean(exp_recon_perturb_all_slice_aginggene_choose) - np.mean(exprecon_use_slice_aginggene_choose))/np.mean(exprecon_use_slice_aginggene_choose))
        diff_mean_all = diff_mean_all + diff_mean_list
        diff_aginggeneexp_mean_all = diff_aginggeneexp_mean_all + diff_aginggeneexp_mean_list
    ##
    ##Fill the nan in diff_mean_all to 0
    diff_mean_all = np.array(diff_mean_all)
    diff_mean_all[np.isnan(diff_mean_all)] = 0
    diff_mean_all = diff_mean_all.tolist()
    diff_mean_all_dict[permute_type_cur] = diff_mean_all  
    aging_geneexp_all_dict[permute_type_cur] = diff_aginggeneexp_mean_all

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, wilcoxon

# Prepare data for plotting
plot_data = []
for celltype, changes in diff_mean_all_dict.items():
    for change in changes:
        plot_data.append({"Permuted_Type": celltype, "Mean_Age_Change": change})
plot_df = pd.DataFrame(plot_data)

# Save CSV for R
csv_path = str(run_dirs['run_dir']) + "/NeighborAgeChange_plotdata_LRknockout.csv"
plot_df.to_csv(output_path(csv_path), index=False)
print("Saved CSV:", csv_path)

# ======================================================
# NEW order
order = ["Top", "Top_half", "Mild", "Baseline"]

# ======================================================
# Mann–Whitney U tests: Top vs all other conditions
#    unpaired, two-sided comparison
# ======================================================
pvalues = {}
rank_sum_rows = []

def run_test_two_sided(group1, group2):
    return mannwhitneyu(
        diff_mean_all_dict[group1],
        diff_mean_all_dict[group2],
        alternative="two-sided"
    )

compare_groups = ["Top_half", "Mild", "Baseline"]

for g in compare_groups:
    try:
        stat, p = run_test_two_sided("Top", g)
        pvalues[f"Top_vs_{g}"] = p
        rank_sum_rows.append({"group1": "Top", "group2": g,
            "n1": len(diff_mean_all_dict["Top"]), "n2": len(diff_mean_all_dict[g]),
            "statistic_U": stat, "pvalue": p})
    except Exception as e:
        print(f"Error testing Top vs {g}:", e)

print("\n=== Mann–Whitney U unpaired, two-sided ===")
for k in [f"Top_vs_{g}" for g in compare_groups]:
    print(f"{k}: p = {pvalues.get(k, np.nan):.3e}")

# Record the between-condition tests used by the plot annotations.
rank_sum_tests = pd.DataFrame(rank_sum_rows).assign(
    test="Mann-Whitney U (Wilcoxon rank-sum)", alternative="two-sided", paired=False,
    method="auto", use_continuity=True,
)
rank_sum_tests.to_csv(output_path(run_dirs["run_dir"] / "NeighborAgeChange_LRknockout_ranksum_tests.csv"), index=False)
display(rank_sum_tests)

# ======================================================
# Wilcoxon vs zero (H1: median < 0)
# ======================================================
print("\n=== Wilcoxon one-sample test (Median < 0) ===")
pvalues_vs_zero = {}

for ct in order:
    try:
        arr = plot_df.loc[plot_df["Permuted_Type"] == ct, "Mean_Age_Change"].values.astype(float)

        if len(arr) > 0 and (arr != 0).any():
            stat_z, p_z = wilcoxon(arr, alternative="less", zero_method="wilcox")
            pvalues_vs_zero[ct] = p_z
            print(f"{ct}: p = {p_z:.3e}")
        else:
            pvalues_vs_zero[ct] = None
            print(f"{ct}: cannot test (all zeros or too few samples)")
    except Exception as e:
        print(f"{ct}: error {e}")
        pvalues_vs_zero[ct] = None

# ======================================================
# Violin plot
# ======================================================
plt.figure(figsize=(8, 6))

# keep colors aligned to the NEW order
custom_colors = ["#7F27FF", "#2A9D8F", "#FDBF60", "#4CC9F0", "#FF8911"]

sns.violinplot(
    x="Permuted_Type",
    y="Mean_Age_Change",
    data=plot_df,
    order=order,
    palette=custom_colors,
    cut=0
)

plt.title("Effect on Predicted Neighbor Age (Mean)")
plt.ylabel("Mean Change in Predicted Age")
plt.xlabel("Permuted Cell Type")
plt.axhline(0, color="gray", linestyle="--")

ax = plt.gca()
ymax = plot_df["Mean_Age_Change"].max()

# ======================================================
# Add p-value annotations for Top vs all others
# ======================================================
def add_p_label(x1, x2, y, pval, label_shift=0.03):
    line_height = y + ymax * label_shift
    ax.plot([x1, x1, x2, x2],
            [y, line_height, line_height, y],
            lw=1.5, c="black")
    text = f"p = {pval:.3e}"
    ax.text((x1 + x2) / 2, line_height + ymax * 0.02,
            text, ha="center", va="bottom", fontsize=10)

# x positions follow NEW order = ['Top'(0), 'reverse'(1), 'Top_half'(2), 'Mild'(3), 'Baseline'(4)]
pair_positions = {
    "Top_vs_Top_half": (0, 1),
    "Top_vs_Mild": (0, 2),
    "Top_vs_Baseline": (0, 3)
}

offset = 1.02
step = 0.14

# annotate in this explicit order (to control stacking)
annot_order = ["Top_vs_Top_half", "Top_vs_Mild", "Top_vs_Baseline"]
for i, key in enumerate(annot_order):
    if key in pvalues and pvalues[key] is not None and np.isfinite(pvalues[key]):
        x1, x2 = pair_positions[key]
        add_p_label(x1, x2, ymax * (offset + step * i), pvalues[key])

plt.tight_layout()
plt.savefig(output_path(str(run_dirs['run_dir']) + "/NeighboringCell_AgeChange_Mean_ViolinPlot_LRknockout.png"), dpi=300)
plt.show()
plt.close()


## Optional GO enrichment analyses of the top regulators and targets

This section performs GO enrichment on the top regulators and target genes identified from the loading analysis. 

In [ ]:
import time
import gseapy as gp
from gseapy.enrichr import EnrichrAPIError


def enrichr_with_retry(**kwargs):
    """Retry a rate-limited Enrichr request without changing analysis arguments."""
    for attempt in range(3):
        try:
            return gp.enrichr(**kwargs)
        except EnrichrAPIError as exc:
            if "status code: 429" not in str(exc) or attempt == 2:
                raise
            delay = 30 * (attempt + 1)
            print(f"Enrichr rate limit (429); retrying in {delay} seconds ({attempt + 2}/3).", flush=True)
            time.sleep(delay)


In [ ]:
MI_OI = "MI29"

In [ ]:
loading_receiver_use_df_norm_choose = loading_receiver_use_df_norm.loc[MI_OI]
loading_sender_use_df_norm_choose = loading_sender_use_df_norm.loc[MI_OI]
loading_receiver_use_df_norm_choose = loading_receiver_use_df_norm_choose.sort_values(ascending=False)
loading_sender_use_df_norm_choose = loading_sender_use_df_norm_choose.sort_values(ascending=False)

In [ ]:
##Get the top regulators and targets based on the loading
top_targetgene_MIOI = loading_receiver_use_df_norm_choose.index[loading_receiver_use_df_norm_choose > 0.2].tolist()
print(top_targetgene_MIOI)
top_regulatorgene_MIOI = loading_sender_use_df_norm_choose.index[loading_sender_use_df_norm_choose > 0.2].tolist()
print(top_regulatorgene_MIOI)

### Top regulators

In [ ]:
go_res = enrichr_with_retry(
    gene_list=top_regulatorgene_MIOI,
    gene_sets=['GO_Biological_Process_2021'],
    organism="mouse",
    outdir=str(output_path(run_dirs["run_dir"] / "GO_MIOI_target")),
    cutoff=0.05 
)

df_filter = go_res.results[
    (go_res.results['Adjusted P-value'] < 0.05)]

print(df_filter)

In [ ]:
GO_term_choose = ["positive regulation of cellular senescence (GO:2000774)",
                  "cellular senescence (GO:0090398)",
                  "regulation of interferon-gamma production (GO:0032649)",
                  "interferon-gamma-mediated signaling pathway (GO:0060333)",
                  "T cell activation (GO:0042110)",
                  "positive regulation of T cell cytokine production (GO:0002726)",
                  "cytokine-mediated signaling pathway (GO:0019221)"]

df_filter = df_filter[df_filter['Term'].isin(GO_term_choose)]
print(df_filter)
df_filter.to_csv(output_path(str(run_dirs['run_dir']) + '/GO_enrichment_topregulator_MI29.csv'))

### Top targets

In [ ]:
go_res = enrichr_with_retry(
    gene_list=top_targetgene_MIOI,
    gene_sets=['GO_Biological_Process_2021'],
    organism="mouse",
    outdir=str(output_path(run_dirs["run_dir"] / "GO_MIOI_target")), 
    cutoff=0.05
)

df_filter = go_res.results[
    (go_res.results['Adjusted P-value'] < 0.05)]

print(df_filter)

In [ ]:
GO_term_choose = ["inflammatory response (GO:0006954)",
                  "positive regulation of leukocyte cell-cell adhesion (GO:1903039)",
                  "antigen receptor-mediated signaling pathway (GO:0050851)",
                  "global genome nucleotide-excision repair (GO:0070911)",
                  "nucleotide-excision repair, DNA incision (GO:0033683)"]

df_filter = df_filter[df_filter['Term'].isin(GO_term_choose)]
print(df_filter)
df_filter.to_csv(output_path(str(run_dirs['run_dir']) + '/GO_enrichment_toptarget_MI29.csv'))